## Cell 1 - Environment, deterministic seed and output directories.

Không đọc dataset, caption hoặc Test Set.

In [1]:
import importlib.util
import json
import os
import platform
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torchvision


PHASE2_SEED = 42
PHASE2_ROOT = Path(
    "/kaggle/working//phase2_student_baselines"
)

PHASE2_CONFIG_DIR = PHASE2_ROOT / "configs"
PHASE2_AUDIT_DIR = PHASE2_ROOT / "audits"
PHASE2_CHECKPOINT_DIR = PHASE2_ROOT / "checkpoints"
PHASE2_LOG_DIR = PHASE2_ROOT / "logs"
PHASE2_PREDICTION_DIR = PHASE2_ROOT / "predictions"
PHASE2_METRIC_DIR = PHASE2_ROOT / "metrics"

for directory in [
    PHASE2_ROOT,
    PHASE2_CONFIG_DIR,
    PHASE2_AUDIT_DIR,
    PHASE2_CHECKPOINT_DIR,
    PHASE2_LOG_DIR,
    PHASE2_PREDICTION_DIR,
    PHASE2_METRIC_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# Cần đặt trước các kernel operation yêu cầu deterministic workspace.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = str(PHASE2_SEED)

random.seed(PHASE2_SEED)
np.random.seed(PHASE2_SEED)
torch.manual_seed(PHASE2_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(PHASE2_SEED)
    torch.cuda.manual_seed_all(PHASE2_SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(
    True,
    warn_only=True,
)


cuda_available = torch.cuda.is_available()
device = torch.device(
    "cuda"
    if cuda_available
    else "cpu"
)

gpu_name = (
    torch.cuda.get_device_name(0)
    if cuda_available
    else None
)

gpu_capability = (
    list(torch.cuda.get_device_capability(0))
    if cuda_available
    else None
)

gpu_memory_bytes = (
    torch.cuda.get_device_properties(0).total_memory
    if cuda_available
    else None
)

optional_dependencies = {
    package_name: (
        importlib.util.find_spec(package_name)
        is not None
    )
    for package_name in [
        "timm",
        "transformers",
        "torchmetrics",
        "onnx",
        "onnxruntime",
    ]
}

environment_audit = {
    "status": "passed" if cuda_available else "failed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": cuda_available,
    "torch_cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "device": str(device),
    "gpu_name": gpu_name,
    "gpu_capability": gpu_capability,
    "gpu_memory_bytes": gpu_memory_bytes,
    "seed": PHASE2_SEED,
    "determinism": {
        "cublas_workspace_config":
            os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_deterministic":
            torch.backends.cudnn.deterministic,
        "deterministic_algorithms_warn_only": True,
    },
    "optional_dependencies": optional_dependencies,
    "policy": {
        "dataset_read": False,
        "caption_read": False,
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}

PHASE2_ENVIRONMENT_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_environment_audit.json"
)

with PHASE2_ENVIRONMENT_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 72)
print("PHASE 2 ENVIRONMENT")
print("=" * 72)
print("Python                 :", sys.version.split()[0])
print("Platform               :", platform.platform())
print("PyTorch                :", torch.__version__)
print("TorchVision            :", torchvision.__version__)
print("CUDA available         :", cuda_available)
print("PyTorch CUDA           :", torch.version.cuda)
print("cuDNN                  :", torch.backends.cudnn.version())
print("Device                 :", device)
print("GPU                    :", gpu_name)
print("GPU capability         :", gpu_capability)
print(
    "GPU memory            :",
    (
        f"{gpu_memory_bytes / (1024 ** 3):.2f} GiB"
        if gpu_memory_bytes is not None
        else "N/A"
    ),
)
print("Seed                   :", PHASE2_SEED)
print("CUDNN benchmark        :", torch.backends.cudnn.benchmark)
print(
    "CUDNN deterministic    :",
    torch.backends.cudnn.deterministic,
)

print("\nOptional dependencies")
for package_name, available in optional_dependencies.items():
    print(f"  {package_name:12s}: {available}")

print("\nOutput root            :", PHASE2_ROOT)
print("Environment audit      :", PHASE2_ENVIRONMENT_AUDIT_PATH)


assert sys.version_info >= (3, 10)
assert cuda_available, (
    "Phase 2 training yêu cầu GPU. "
    "Kaggle Settings → Accelerator → GPU T4, "
    "sau đó restart session và chạy lại Cell 1."
)
assert PHASE2_ENVIRONMENT_AUDIT_PATH.is_file()
assert PHASE2_ENVIRONMENT_AUDIT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("PHASE 2 CELL 1 STATUS")
print("=" * 72)
print("Status                 : PASSED")
print("Accelerator            :", gpu_name)
print("Deterministic seed     : 42")
print("Dataset read           : False")
print("Caption read           : False")
print("Test Set used          : False")
print("Flickr data used       : False")
print("Training started       : False")
print(
    "Next                   : "
    "discover Phase 1 output and MS COCO image input"
)

PHASE 2 ENVIRONMENT
Python                 : 3.12.13
Platform               : Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch                : 2.10.0+cu128
TorchVision            : 0.25.0+cu128
CUDA available         : True
PyTorch CUDA           : 12.8
cuDNN                  : 91002
Device                 : cuda
GPU                    : Tesla T4
GPU capability         : [7, 5]
GPU memory            : 14.56 GiB
Seed                   : 42
CUDNN benchmark        : False
CUDNN deterministic    : True

Optional dependencies
  timm        : True
  transformers: True
  torchmetrics: True
  onnx        : True
  onnxruntime : False

Output root            : /kaggle/working//phase2_student_baselines
Environment audit      : /kaggle/working//phase2_student_baselines/audits/phase2_environment_audit.json

PHASE 2 CELL 1 STATUS
Status                 : PASSED
Accelerator            : Tesla T4
Deterministic seed     : 42
Dataset read           : False
Caption read           : False
Test Set used      

## Cell 2: Discover và xác minh input

### Mục tiêu

- Tìm duy nhất formal READY marker của Phase 1.
- Rebase đường dẫn artifact từ Kaggle Input hiện tại.
- Không sử dụng đường dẫn `/kaggle/working` cũ trong READY marker.
- Xác minh toàn bộ 32 SHA-256 của Phase 1.
- Tìm dataset MS COCO 2014.
- Xử lý cấu trúc thư mục lồng:
  - train2014/train2014
  - val2014/val2014
- Xác nhận 82.783 ảnh train2014.
- Xác nhận 40.504 ảnh val2014.
- Không đọc caption.
- Không mở Test Parquet.
- Không bắt đầu huấn luyện.

In [2]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path


CELL2_STARTED_AT = time.perf_counter()

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
FORMAL_READY_FILENAME = (
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

EXPECTED_PHASE1_CHECKSUM_ROWS = 32
EXPECTED_TRAIN_IMAGE_FILES = 82_783
EXPECTED_VAL_IMAGE_FILES = 40_504


required_runtime_names = [
    "PHASE2_ROOT",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1: "
    f"{missing_runtime_names}. Hãy chạy lại Cell 1."
)

assert KAGGLE_INPUT_ROOT.is_dir(), (
    f"Không tìm thấy {KAGGLE_INPUT_ROOT}."
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def discover_formal_ready_markers(input_root):
    """
    Dò READY marker nhưng không đi vào các thư mục chứa hàng chục nghìn ảnh.
    """

    matches = []
    excluded_directory_names = {
        "train2014",
        "val2014",
        "annotations_trainval2014",
    }

    for current_root, directory_names, filenames in os.walk(
        input_root
    ):
        directory_names[:] = [
            directory_name
            for directory_name in directory_names
            if directory_name
            not in excluded_directory_names
        ]

        if FORMAL_READY_FILENAME in filenames:
            matches.append(
                Path(current_root)
                / FORMAL_READY_FILENAME
            )

    return sorted(set(matches))


def parse_sha256sums(path):
    pattern = re.compile(
        r"^(?P<sha256>[0-9a-fA-F]{64})  "
        r"(?P<relative_path>.+)$"
    )
    rows = []
    invalid_lines = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, raw_line in enumerate(
            file,
            start=1,
        ):
            line = raw_line.rstrip("\n\r")

            if not line:
                continue

            match = pattern.fullmatch(line)

            if match is None:
                invalid_lines.append(
                    {
                        "line_number": line_number,
                        "text": line,
                    }
                )
                continue

            rows.append(
                {
                    "sha256":
                        match.group("sha256").lower(),
                    "relative_path":
                        match.group("relative_path"),
                }
            )

    assert not invalid_lines, (
        f"SHA256SUMS có dòng sai: {invalid_lines[:10]}"
    )

    return rows


def verify_phase1_sha256sums(
    phase1_root,
    sha256sums_path,
):
    rows = parse_sha256sums(sha256sums_path)
    failures = []

    for row in rows:
        artifact_path = (
            phase1_root
            / Path(row["relative_path"])
        )

        if not artifact_path.is_file():
            failures.append(
                {
                    "relative_path": row["relative_path"],
                    "reason": "missing",
                }
            )
            continue

        actual_hash = sha256_file(artifact_path)

        if actual_hash != row["sha256"]:
            failures.append(
                {
                    "relative_path": row["relative_path"],
                    "reason": "sha256_mismatch",
                    "expected": row["sha256"],
                    "actual": actual_hash,
                }
            )

    assert not failures, (
        "Phase 1 checksum verification failed: "
        f"{failures[:10]}"
    )

    return rows


def find_mscoco_dataset_roots(input_root):
    candidates = []

    for current_root, directory_names, _ in os.walk(
        input_root
    ):
        current_path = Path(current_root)

        if current_path.name == "mscoco2014-1":
            candidates.append(current_path)
            directory_names[:] = []
            continue

        # Không đi sâu vào image payload khi chỉ tìm dataset root.
        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                "train2014",
                "val2014",
            }
        ]

    return sorted(set(candidates))


def resolve_coco_image_leaf(
    root_directory,
    split_name,
    maximum_depth=3,
):
    root_directory = Path(root_directory)
    filename_glob = f"COCO_{split_name}2014_*.jpg"
    current_level = [root_directory]
    matches = []

    for _ in range(maximum_depth + 1):
        next_level = []

        for candidate in current_level:
            first_image = next(
                candidate.glob(filename_glob),
                None,
            )

            if first_image is not None:
                matches.append(candidate)
                continue

            next_level.extend(
                child
                for child in candidate.iterdir()
                if child.is_dir()
            )

        if matches:
            break

        current_level = next_level

    assert len(matches) == 1, (
        f"Không resolve duy nhất {split_name}2014 leaf. "
        f"Root={root_directory}; matches={matches}"
    )

    return matches[0]


def count_valid_coco_images(
    image_directory,
    split_name,
):
    filename_pattern = re.compile(
        rf"^COCO_{split_name}2014_\d{{12}}\.jpg$",
        flags=re.IGNORECASE,
    )

    valid_count = 0
    invalid_examples = []

    for path in Path(image_directory).iterdir():
        if not path.is_file():
            continue

        if filename_pattern.fullmatch(path.name):
            valid_count += 1
        elif path.suffix.lower() == ".jpg":
            if len(invalid_examples) < 10:
                invalid_examples.append(path.name)

    assert not invalid_examples, (
        f"Tên file ảnh sai: {invalid_examples}"
    )

    return valid_count


print("=" * 72)
print("DISCOVER PHASE 1 FORMAL OUTPUT")
print("=" * 72)

formal_ready_matches = discover_formal_ready_markers(
    KAGGLE_INPUT_ROOT
)

print("READY marker matches:", len(formal_ready_matches))
for path in formal_ready_matches:
    print(" -", path)

assert len(formal_ready_matches) == 1, (
    "Cần gắn đúng một Phase 1 formal output. "
    f"Tìm thấy {len(formal_ready_matches)}."
)

PHASE1_FORMAL_READY_PATH = formal_ready_matches[0]
PHASE1_INPUT_ROOT = PHASE1_FORMAL_READY_PATH.parent

with PHASE1_FORMAL_READY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    phase1_formal_ready = json.load(file)

assert phase1_formal_ready["status"] == "ready"
assert (
    phase1_formal_ready["phase"]
    == "phase1_mscoco_data_contract"
)
assert phase1_formal_ready["test_set_status"] == "sealed"
assert phase1_formal_ready["test_caption_exported"] is False
assert phase1_formal_ready["test_used"] is False
assert phase1_formal_ready["flickr_used"] is False


# Không dùng absolute /kaggle/working path lưu trong marker cũ.
# Rebase mọi artifact theo PHASE1_INPUT_ROOT vừa discover.
PHASE1_SHA256SUMS_PATH = (
    PHASE1_INPUT_ROOT
    / "SHA256SUMS.txt"
)
PHASE1_EXPERIMENT_CONTRACT_PATH = (
    PHASE1_INPUT_ROOT
    / "configs"
    / "experiment_contract_v2.json"
)
PHASE1_RESOLVED_INPUT_CONTRACT_PATH = (
    PHASE1_INPUT_ROOT
    / "configs"
    / "resolved_input_contract.json"
)
PHASE1_IMAGE_MANIFEST_PARQUET_PATH = (
    PHASE1_INPUT_ROOT
    / "parquet"
    / "mscoco_image_manifest.parquet"
)
PHASE1_TRAIN_PARQUET_PATH = (
    PHASE1_INPUT_ROOT
    / "parquet"
    / "mscoco_train.parquet"
)
PHASE1_VALIDATION_PARQUET_PATH = (
    PHASE1_INPUT_ROOT
    / "parquet"
    / "mscoco_validation.parquet"
)
PHASE1_TEST_PARQUET_PATH = (
    PHASE1_INPUT_ROOT
    / "parquet"
    / "mscoco_test.parquet"
)
PHASE1_VOCABULARY_PATH = (
    PHASE1_INPUT_ROOT
    / "vocabulary"
    / "word_vocabulary_minfreq5.json"
)
PHASE1_TOKENIZER_CONTRACT_PATH = (
    PHASE1_INPUT_ROOT
    / "vocabulary"
    / "tokenizer_contract.json"
)
PHASE1_SEQUENCE_CONTRACT_PATH = (
    PHASE1_INPUT_ROOT
    / "vocabulary"
    / "sequence_encoding_contract.json"
)
PHASE1_DATASET_STATISTICS_PATH = (
    PHASE1_INPUT_ROOT
    / "audits"
    / "dataset_statistics.json"
)
PHASE1_DATA_INTEGRITY_AUDIT_PATH = (
    PHASE1_INPUT_ROOT
    / "audits"
    / "data_integrity_audit.json"
)

required_phase1_paths = [
    PHASE1_SHA256SUMS_PATH,
    PHASE1_EXPERIMENT_CONTRACT_PATH,
    PHASE1_RESOLVED_INPUT_CONTRACT_PATH,
    PHASE1_IMAGE_MANIFEST_PARQUET_PATH,
    PHASE1_TRAIN_PARQUET_PATH,
    PHASE1_VALIDATION_PARQUET_PATH,
    PHASE1_TEST_PARQUET_PATH,
    PHASE1_VOCABULARY_PATH,
    PHASE1_TOKENIZER_CONTRACT_PATH,
    PHASE1_SEQUENCE_CONTRACT_PATH,
    PHASE1_DATASET_STATISTICS_PATH,
    PHASE1_DATA_INTEGRITY_AUDIT_PATH,
]

for path in required_phase1_paths:
    assert path.is_file(), f"Thiếu Phase 1 artifact: {path}"
    assert path.stat().st_size > 0


print("\n" + "=" * 72)
print("VERIFY PHASE 1 SHA256SUMS")
print("=" * 72)

phase1_checksum_rows = verify_phase1_sha256sums(
    phase1_root=PHASE1_INPUT_ROOT,
    sha256sums_path=PHASE1_SHA256SUMS_PATH,
)

assert (
    len(phase1_checksum_rows)
    == EXPECTED_PHASE1_CHECKSUM_ROWS
), (
    f"SHA256SUMS có {len(phase1_checksum_rows)} dòng, "
    f"expected={EXPECTED_PHASE1_CHECKSUM_ROWS}."
)

phase1_sha256sums_hash = sha256_file(
    PHASE1_SHA256SUMS_PATH
)

assert (
    phase1_sha256sums_hash
    == phase1_formal_ready["sha256sums_sha256"]
), (
    "SHA256SUMS checksum không khớp formal READY marker."
)

print("Rows verified      :", len(phase1_checksum_rows))
print("SHA256SUMS SHA-256 :", phase1_sha256sums_hash)


print("\n" + "=" * 72)
print("DISCOVER MS COCO IMAGE DATASET")
print("=" * 72)

mscoco_dataset_roots = find_mscoco_dataset_roots(
    KAGGLE_INPUT_ROOT
)

print("MS COCO dataset matches:", len(mscoco_dataset_roots))
for path in mscoco_dataset_roots:
    print(" -", path)

assert len(mscoco_dataset_roots) == 1, (
    "Cần gắn đúng một dataset mscoco2014-1. "
    f"Tìm thấy {len(mscoco_dataset_roots)}."
)

MSCOCO_INPUT_ROOT = mscoco_dataset_roots[0]
COCO_TRAIN_IMAGES_ROOT_DIR = (
    MSCOCO_INPUT_ROOT / "train2014"
)
COCO_VAL_IMAGES_ROOT_DIR = (
    MSCOCO_INPUT_ROOT / "val2014"
)

COCO_TRAIN_IMAGES_LEAF_DIR = resolve_coco_image_leaf(
    COCO_TRAIN_IMAGES_ROOT_DIR,
    "train",
)
COCO_VAL_IMAGES_LEAF_DIR = resolve_coco_image_leaf(
    COCO_VAL_IMAGES_ROOT_DIR,
    "val",
)

train_image_count = count_valid_coco_images(
    COCO_TRAIN_IMAGES_LEAF_DIR,
    "train",
)
val_image_count = count_valid_coco_images(
    COCO_VAL_IMAGES_LEAF_DIR,
    "val",
)

assert train_image_count == EXPECTED_TRAIN_IMAGE_FILES
assert val_image_count == EXPECTED_VAL_IMAGE_FILES

print("Train root         :", COCO_TRAIN_IMAGES_ROOT_DIR)
print("Train leaf         :", COCO_TRAIN_IMAGES_LEAF_DIR)
print("Train image count  :", train_image_count)
print("Val root           :", COCO_VAL_IMAGES_ROOT_DIR)
print("Val leaf           :", COCO_VAL_IMAGES_LEAF_DIR)
print("Val image count    :", val_image_count)


phase2_input_contract = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "phase1": {
        "input_root": str(PHASE1_INPUT_ROOT),
        "formal_ready_marker": str(
            PHASE1_FORMAL_READY_PATH
        ),
        "sha256sums_path": str(
            PHASE1_SHA256SUMS_PATH
        ),
        "sha256sums_sha256":
            phase1_sha256sums_hash,
        "checksum_rows_verified":
            len(phase1_checksum_rows),
    },
    "mscoco_images": {
        "dataset_root": str(MSCOCO_INPUT_ROOT),
        "train_root": str(
            COCO_TRAIN_IMAGES_ROOT_DIR
        ),
        "train_leaf": str(
            COCO_TRAIN_IMAGES_LEAF_DIR
        ),
        "train_image_count": train_image_count,
        "val_root": str(COCO_VAL_IMAGES_ROOT_DIR),
        "val_leaf": str(COCO_VAL_IMAGES_LEAF_DIR),
        "val_image_count": val_image_count,
    },
    "resolved_phase1_artifacts": {
        "experiment_contract": str(
            PHASE1_EXPERIMENT_CONTRACT_PATH
        ),
        "image_manifest_parquet": str(
            PHASE1_IMAGE_MANIFEST_PARQUET_PATH
        ),
        "train_parquet": str(
            PHASE1_TRAIN_PARQUET_PATH
        ),
        "validation_parquet": str(
            PHASE1_VALIDATION_PARQUET_PATH
        ),
        "test_parquet": str(
            PHASE1_TEST_PARQUET_PATH
        ),
        "vocabulary": str(
            PHASE1_VOCABULARY_PATH
        ),
        "tokenizer_contract": str(
            PHASE1_TOKENIZER_CONTRACT_PATH
        ),
        "sequence_contract": str(
            PHASE1_SEQUENCE_CONTRACT_PATH
        ),
    },
    "policy": {
        "caption_read": False,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "training_started": False,
    },
    "elapsed_seconds": (
        time.perf_counter()
        - CELL2_STARTED_AT
    ),
}

PHASE2_INPUT_CONTRACT_PATH = (
    Path(PHASE2_CONFIG_DIR)
    / "phase2_input_contract.json"
)

with PHASE2_INPUT_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        phase2_input_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert PHASE2_INPUT_CONTRACT_PATH.is_file()
assert PHASE2_INPUT_CONTRACT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("PHASE 2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 1 formal READY      : VERIFIED")
print("Phase 1 checksum rows     :", len(phase1_checksum_rows))
print("Phase 1 checksums         : ALL PASSED")
print("MS COCO train images      :", train_image_count)
print("MS COCO val images        :", val_image_count)
print("Caption read              : False")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Input contract            :", PHASE2_INPUT_CONTRACT_PATH)
print(
    "Elapsed                   :",
    round(
        time.perf_counter()
        - CELL2_STARTED_AT,
        2,
    ),
    "giây",
)
print(
    "Next                      : "
    "load and validate Phase 1 contracts and Parquet schemas"
)


DISCOVER PHASE 1 FORMAL OUTPUT
READY marker matches: 1
 - /kaggle/input/notebooks/anhermes/01-mscoco-data-contract//phase1_mscoco/PHASE1_FORMAL_DELIVERABLES_READY.json

VERIFY PHASE 1 SHA256SUMS
Rows verified      : 32
SHA256SUMS SHA-256 : 0d502666d429fe539a29db4d647b30e5b0b6bf795361310f90fd22ff2d87a93b

DISCOVER MS COCO IMAGE DATASET
MS COCO dataset matches: 1
 - /kaggle/input/datasets/nguynthnhdan/mscoco2014-1
Train root         : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014
Train leaf         : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/train2014/train2014
Train image count  : 82783
Val root           : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014
Val leaf           : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014/val2014
Val image count    : 40504

PHASE 2 CELL 2 STATUS
Status                    : PASSED
Phase 1 formal READY      : VERIFIED
Phase 1 checksum rows     : 32
Phase 1 checksums         : ALL PASSED
MS COCO train images      : 827

## Cell 3: Khóa contract và kiểm tra Parquet

### Mục tiêu

- Đọc Experiment Contract.
- Đọc Vocabulary và Tokenizer Contract.
- Đọc Sequence Encoding Contract.
- Xác nhận vocabulary size = 10.080.
- Xác nhận PAD/BOS/EOS/UNK = 0/1/2/3.
- Xác nhận maximum decoder steps = 31.
- Kiểm tra schema và số dòng của bốn Parquet.
- Đọc image-manifest metadata để tạo ánh xạ image_id → đường dẫn ảnh.
- Tách runtime map cho Training và Validation.
- Chỉ giữ tập ID của Test để kiểm tra leakage.
- Đọc Test Parquet dưới dạng metadata-only.
- Xác nhận Test không có caption hoặc token IDs.
- Chưa đọc caption payload Training/Validation.
- Chưa bắt đầu huấn luyện.

In [3]:
import json
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq


CELL3_STARTED_AT = time.perf_counter()

EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_MIN_FREQUENCY = 5
EXPECTED_MAX_LEXICAL_TOKENS = 30
EXPECTED_MAX_FULL_SEQUENCE = 32
EXPECTED_MAX_DECODER_STEPS = 31

EXPECTED_TRAIN_PARQUET_ROWS = 566_747
EXPECTED_VALIDATION_PARQUET_ROWS = 25_010
EXPECTED_TEST_PARQUET_ROWS = 5_000
EXPECTED_IMAGE_MANIFEST_ROWS = 123_287

EXPECTED_EXPERIMENT_SPLIT_COUNTS = {
    "train": 113_287,
    "validation": 5_000,
    "held_out_test": 5_000,
}

EXPECTED_KARPATHY_SPLIT_COUNTS = {
    "train": 82_783,
    "restval": 30_504,
    "val": 5_000,
    "test": 5_000,
}

EXPECTED_DEVELOPMENT_COLUMNS = [
    "annotation_id",
    "image_id",
    "karpathy_split",
    "source_partition",
    "caption",
    "token_ids",
    "original_lexical_length",
    "retained_lexical_length",
    "was_truncated",
]

EXPECTED_TEST_COLUMNS = [
    "image_id",
    "karpathy_split",
    "experiment_split",
    "source_partition",
    "image_path",
    "caption_included",
    "test_set_status",
]

EXPECTED_IMAGE_MANIFEST_COLUMNS = [
    "image_id",
    "karpathy_split",
    "experiment_split",
    "source_partition",
    "image_path",
]


required_runtime_names = [
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE1_EXPERIMENT_CONTRACT_PATH",
    "PHASE1_IMAGE_MANIFEST_PARQUET_PATH",
    "PHASE1_TRAIN_PARQUET_PATH",
    "PHASE1_VALIDATION_PARQUET_PATH",
    "PHASE1_TEST_PARQUET_PATH",
    "PHASE1_VOCABULARY_PATH",
    "PHASE1_TOKENIZER_CONTRACT_PATH",
    "PHASE1_SEQUENCE_CONTRACT_PATH",
    "PHASE1_DATASET_STATISTICS_PATH",
    "PHASE1_DATA_INTEGRITY_AUDIT_PATH",
    "COCO_TRAIN_IMAGES_LEAF_DIR",
    "COCO_VAL_IMAGES_LEAF_DIR",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–2: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)


def read_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def assert_exact_schema_names(path, expected_names):
    schema = pq.read_schema(path)

    assert schema.names == expected_names, (
        f"Schema của {path} không đúng.\n"
        f"Actual={schema.names}\n"
        f"Expected={expected_names}"
    )

    return schema


def parquet_summary(path):
    metadata = pq.read_metadata(path)

    return {
        "path": str(path),
        "rows": metadata.num_rows,
        "row_groups": metadata.num_row_groups,
        "columns": metadata.num_columns,
        "size_bytes": Path(path).stat().st_size,
    }


print("=" * 72)
print("LOAD LOCKED PHASE 1 CONTRACTS")
print("=" * 72)

PHASE2_EXPERIMENT_CONTRACT = read_json(
    PHASE1_EXPERIMENT_CONTRACT_PATH
)
PHASE2_VOCABULARY_PAYLOAD = read_json(
    PHASE1_VOCABULARY_PATH
)
PHASE2_TOKENIZER_CONTRACT = read_json(
    PHASE1_TOKENIZER_CONTRACT_PATH
)
PHASE2_SEQUENCE_CONTRACT = read_json(
    PHASE1_SEQUENCE_CONTRACT_PATH
)
PHASE2_DATASET_STATISTICS = read_json(
    PHASE1_DATASET_STATISTICS_PATH
)
PHASE2_DATA_INTEGRITY_AUDIT = read_json(
    PHASE1_DATA_INTEGRITY_AUDIT_PATH
)


assert PHASE2_EXPERIMENT_CONTRACT["dataset"]["name"] == "MS COCO"
assert PHASE2_EXPERIMENT_CONTRACT["dataset"]["release"] == "2014"
assert (
    PHASE2_EXPERIMENT_CONTRACT["dataset"]
    ["split_protocol"]
    == "Karpathy"
)
assert (
    PHASE2_EXPERIMENT_CONTRACT["dataset"]
    ["experiment_splits"]["test"]["status"]
    == "sealed"
)
assert (
    PHASE2_EXPERIMENT_CONTRACT["project"]
    ["flickr_allowed"]
    is False
)
assert (
    PHASE2_DATA_INTEGRITY_AUDIT["status"]
    == "passed"
)
assert (
    PHASE2_DATA_INTEGRITY_AUDIT["policy"]
    ["test_set_status"]
    == "sealed"
)
assert (
    PHASE2_DATA_INTEGRITY_AUDIT["policy"]
    ["test_used"]
    is False
)


PHASE2_ID_TO_TOKEN = (
    PHASE2_VOCABULARY_PAYLOAD["id_to_token"]
)
PHASE2_TOKEN_TO_ID = {
    token: token_id
    for token_id, token in enumerate(
        PHASE2_ID_TO_TOKEN
    )
}
PHASE2_VOCABULARY_SIZE = len(
    PHASE2_ID_TO_TOKEN
)

assert (
    PHASE2_VOCABULARY_SIZE
    == EXPECTED_VOCABULARY_SIZE
)
assert (
    PHASE2_VOCABULARY_PAYLOAD["vocabulary_size"]
    == EXPECTED_VOCABULARY_SIZE
)
assert (
    PHASE2_VOCABULARY_PAYLOAD["minimum_frequency"]
    == EXPECTED_MIN_FREQUENCY
)
assert (
    PHASE2_TOKENIZER_CONTRACT["vocabulary_size"]
    == EXPECTED_VOCABULARY_SIZE
)
assert PHASE2_TOKEN_TO_ID["<pad>"] == 0
assert PHASE2_TOKEN_TO_ID["<bos>"] == 1
assert PHASE2_TOKEN_TO_ID["<eos>"] == 2
assert PHASE2_TOKEN_TO_ID["<unk>"] == 3
assert len(PHASE2_TOKEN_TO_ID) == PHASE2_VOCABULARY_SIZE


length_policy = PHASE2_SEQUENCE_CONTRACT[
    "length_policy"
]

assert (
    length_policy["maximum_lexical_tokens"]
    == EXPECTED_MAX_LEXICAL_TOKENS
)
assert (
    length_policy["maximum_full_sequence_tokens"]
    == EXPECTED_MAX_FULL_SEQUENCE
)
assert (
    length_policy["maximum_decoder_time_steps"]
    == EXPECTED_MAX_DECODER_STEPS
)
assert (
    PHASE2_SEQUENCE_CONTRACT["special_token_ids"]
    == {
        "pad": 0,
        "bos": 1,
        "eos": 2,
        "unk": 3,
    }
)
assert (
    PHASE2_SEQUENCE_CONTRACT["data_policy"]
    ["test_read"]
    is False
)


print("Dataset                : MS COCO 2014")
print("Split protocol         : Karpathy")
print("Vocabulary size        :", PHASE2_VOCABULARY_SIZE)
print("Minimum frequency      :", EXPECTED_MIN_FREQUENCY)
print("PAD/BOS/EOS/UNK        : 0 / 1 / 2 / 3")
print("Maximum lexical tokens :", EXPECTED_MAX_LEXICAL_TOKENS)
print("Maximum full sequence  :", EXPECTED_MAX_FULL_SEQUENCE)
print("Maximum decoder steps  :", EXPECTED_MAX_DECODER_STEPS)
print("Test Set status        : SEALED")


print("\n" + "=" * 72)
print("PARQUET SCHEMA AND ROW-COUNT AUDIT")
print("=" * 72)

train_schema = assert_exact_schema_names(
    PHASE1_TRAIN_PARQUET_PATH,
    EXPECTED_DEVELOPMENT_COLUMNS,
)
validation_schema = assert_exact_schema_names(
    PHASE1_VALIDATION_PARQUET_PATH,
    EXPECTED_DEVELOPMENT_COLUMNS,
)
test_schema = assert_exact_schema_names(
    PHASE1_TEST_PARQUET_PATH,
    EXPECTED_TEST_COLUMNS,
)
image_manifest_schema = assert_exact_schema_names(
    PHASE1_IMAGE_MANIFEST_PARQUET_PATH,
    EXPECTED_IMAGE_MANIFEST_COLUMNS,
)

assert pa.types.is_list(
    train_schema.field("token_ids").type
)
assert pa.types.is_int32(
    train_schema.field("token_ids").type.value_type
)
assert pa.types.is_string(
    train_schema.field("caption").type
)
assert "caption" not in test_schema.names
assert "token_ids" not in test_schema.names
assert "annotation_id" not in test_schema.names

parquet_summaries = {
    "train": parquet_summary(
        PHASE1_TRAIN_PARQUET_PATH
    ),
    "validation": parquet_summary(
        PHASE1_VALIDATION_PARQUET_PATH
    ),
    "test": parquet_summary(
        PHASE1_TEST_PARQUET_PATH
    ),
    "image_manifest": parquet_summary(
        PHASE1_IMAGE_MANIFEST_PARQUET_PATH
    ),
}

assert (
    parquet_summaries["train"]["rows"]
    == EXPECTED_TRAIN_PARQUET_ROWS
)
assert (
    parquet_summaries["validation"]["rows"]
    == EXPECTED_VALIDATION_PARQUET_ROWS
)
assert (
    parquet_summaries["test"]["rows"]
    == EXPECTED_TEST_PARQUET_ROWS
)
assert (
    parquet_summaries["image_manifest"]["rows"]
    == EXPECTED_IMAGE_MANIFEST_ROWS
)

for name, summary in parquet_summaries.items():
    print(
        f"{name:14s}: "
        f"{summary['rows']:,} rows | "
        f"{summary['row_groups']} row groups | "
        f"{summary['columns']} columns"
    )


print("\n" + "=" * 72)
print("IMAGE-MANIFEST MEMBERSHIP AUDIT")
print("=" * 72)

image_manifest_table = pq.read_table(
    PHASE1_IMAGE_MANIFEST_PARQUET_PATH,
    columns=[
        "image_id",
        "karpathy_split",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
)

manifest_columns = image_manifest_table.to_pydict()

image_ids_seen = set()
karpathy_split_counts = Counter()
experiment_split_counts = Counter()
source_partition_counts = Counter()
duplicate_image_id_examples = []
invalid_filename_examples = []

PHASE2_TRAIN_IMAGE_PATH_BY_ID = {}
PHASE2_VALIDATION_IMAGE_PATH_BY_ID = {}
PHASE2_HELD_OUT_TEST_IMAGE_IDS = set()

train_filename_pattern = re.compile(
    r"^COCO_train2014_\d{12}\.jpg$",
    flags=re.IGNORECASE,
)
val_filename_pattern = re.compile(
    r"^COCO_val2014_\d{12}\.jpg$",
    flags=re.IGNORECASE,
)

for (
    image_id,
    karpathy_split,
    experiment_split,
    source_partition,
    stored_image_path,
) in zip(
    manifest_columns["image_id"],
    manifest_columns["karpathy_split"],
    manifest_columns["experiment_split"],
    manifest_columns["source_partition"],
    manifest_columns["image_path"],
):
    image_id = int(image_id)

    if image_id in image_ids_seen:
        if len(duplicate_image_id_examples) < 10:
            duplicate_image_id_examples.append(image_id)
        continue

    image_ids_seen.add(image_id)
    karpathy_split_counts[karpathy_split] += 1
    experiment_split_counts[experiment_split] += 1
    source_partition_counts[source_partition] += 1

    filename = Path(stored_image_path).name

    if source_partition == "train2014":
        current_image_path = (
            Path(COCO_TRAIN_IMAGES_LEAF_DIR)
            / filename
        )
        valid_filename = (
            train_filename_pattern.fullmatch(filename)
            is not None
        )
    elif source_partition == "val2014":
        current_image_path = (
            Path(COCO_VAL_IMAGES_LEAF_DIR)
            / filename
        )
        valid_filename = (
            val_filename_pattern.fullmatch(filename)
            is not None
        )
    else:
        current_image_path = None
        valid_filename = False

    if not valid_filename:
        if len(invalid_filename_examples) < 10:
            invalid_filename_examples.append(
                {
                    "image_id": image_id,
                    "source_partition": source_partition,
                    "filename": filename,
                }
            )
        continue

    if experiment_split == "train":
        PHASE2_TRAIN_IMAGE_PATH_BY_ID[
            image_id
        ] = current_image_path
    elif experiment_split == "validation":
        PHASE2_VALIDATION_IMAGE_PATH_BY_ID[
            image_id
        ] = current_image_path
    elif experiment_split == "held_out_test":
        PHASE2_HELD_OUT_TEST_IMAGE_IDS.add(
            image_id
        )
    else:
        raise AssertionError(
            f"Unknown experiment split: {experiment_split}"
        )


assert not duplicate_image_id_examples
assert not invalid_filename_examples
assert len(image_ids_seen) == EXPECTED_IMAGE_MANIFEST_ROWS
assert (
    dict(experiment_split_counts)
    == EXPECTED_EXPERIMENT_SPLIT_COUNTS
)
assert (
    dict(karpathy_split_counts)
    == EXPECTED_KARPATHY_SPLIT_COUNTS
)
assert (
    len(PHASE2_TRAIN_IMAGE_PATH_BY_ID)
    == EXPECTED_EXPERIMENT_SPLIT_COUNTS["train"]
)
assert (
    len(PHASE2_VALIDATION_IMAGE_PATH_BY_ID)
    == EXPECTED_EXPERIMENT_SPLIT_COUNTS[
        "validation"
    ]
)
assert (
    len(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
    == EXPECTED_EXPERIMENT_SPLIT_COUNTS[
        "held_out_test"
    ]
)

assert not (
    set(PHASE2_TRAIN_IMAGE_PATH_BY_ID)
    & set(PHASE2_VALIDATION_IMAGE_PATH_BY_ID)
)
assert not (
    set(PHASE2_TRAIN_IMAGE_PATH_BY_ID)
    & PHASE2_HELD_OUT_TEST_IMAGE_IDS
)
assert not (
    set(PHASE2_VALIDATION_IMAGE_PATH_BY_ID)
    & PHASE2_HELD_OUT_TEST_IMAGE_IDS
)

# Xác minh tồn tại trên một mẫu đầu/cuối; Cell 2 đã đếm toàn bộ file.
sample_paths = (
    list(PHASE2_TRAIN_IMAGE_PATH_BY_ID.values())[:32]
    + list(PHASE2_TRAIN_IMAGE_PATH_BY_ID.values())[-32:]
    + list(PHASE2_VALIDATION_IMAGE_PATH_BY_ID.values())[:32]
    + list(PHASE2_VALIDATION_IMAGE_PATH_BY_ID.values())[-32:]
)

missing_sample_paths = [
    str(path)
    for path in sample_paths
    if not path.is_file()
]

assert not missing_sample_paths, (
    f"Thiếu image path trong mẫu: {missing_sample_paths[:10]}"
)

print("Karpathy split counts :", dict(karpathy_split_counts))
print("Experiment splits     :", dict(experiment_split_counts))
print("Source partitions     :", dict(source_partition_counts))
print(
    "Train path map        :",
    len(PHASE2_TRAIN_IMAGE_PATH_BY_ID),
)
print(
    "Validation path map   :",
    len(PHASE2_VALIDATION_IMAGE_PATH_BY_ID),
)
print(
    "Held-out Test IDs     :",
    len(PHASE2_HELD_OUT_TEST_IMAGE_IDS),
)
print("Sample paths missing  :", len(missing_sample_paths))


print("\n" + "=" * 72)
print("TEST PARQUET METADATA-ONLY AUDIT")
print("=" * 72)

test_metadata_table = pq.read_table(
    PHASE1_TEST_PARQUET_PATH,
    columns=[
        "image_id",
        "caption_included",
        "test_set_status",
    ],
)
test_metadata_columns = test_metadata_table.to_pydict()

assert set(
    int(image_id)
    for image_id in test_metadata_columns["image_id"]
) == PHASE2_HELD_OUT_TEST_IMAGE_IDS
assert not any(
    test_metadata_columns["caption_included"]
)
assert set(
    test_metadata_columns["test_set_status"]
) == {"sealed"}

print("Rows                     :", test_metadata_table.num_rows)
print("Caption included true    : 0")
print("Test statuses            : {'sealed'}")
print("Caption column exists    : False")
print("Token IDs column exists  : False")


del image_manifest_table
del manifest_columns
del test_metadata_table
del test_metadata_columns


cell3_elapsed_seconds = (
    time.perf_counter()
    - CELL3_STARTED_AT
)

phase2_contract_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "locked_contract": {
        "dataset": "MS COCO 2014",
        "split": "Karpathy",
        "vocabulary_size": PHASE2_VOCABULARY_SIZE,
        "minimum_frequency": EXPECTED_MIN_FREQUENCY,
        "special_token_ids": {
            "pad": 0,
            "bos": 1,
            "eos": 2,
            "unk": 3,
        },
        "maximum_lexical_tokens":
            EXPECTED_MAX_LEXICAL_TOKENS,
        "maximum_full_sequence":
            EXPECTED_MAX_FULL_SEQUENCE,
        "maximum_decoder_steps":
            EXPECTED_MAX_DECODER_STEPS,
    },
    "parquet": parquet_summaries,
    "membership": {
        "karpathy_split_counts":
            dict(karpathy_split_counts),
        "experiment_split_counts":
            dict(experiment_split_counts),
        "source_partition_counts":
            dict(source_partition_counts),
        "duplicate_image_ids": 0,
        "invalid_filenames": 0,
        "sample_missing_paths": 0,
    },
    "runtime_maps": {
        "training_image_paths":
            len(PHASE2_TRAIN_IMAGE_PATH_BY_ID),
        "validation_image_paths":
            len(PHASE2_VALIDATION_IMAGE_PATH_BY_ID),
        "held_out_test_ids":
            len(PHASE2_HELD_OUT_TEST_IMAGE_IDS),
    },
    "policy": {
        "train_caption_payload_read": False,
        "validation_caption_payload_read": False,
        "test_metadata_read": True,
        "test_caption_available": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "training_started": False,
    },
    "elapsed_seconds": cell3_elapsed_seconds,
}

PHASE2_CONTRACT_AUDIT_PATH = (
    Path(PHASE2_AUDIT_DIR)
    / "phase2_contract_and_parquet_audit.json"
)

with PHASE2_CONTRACT_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        phase2_contract_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert PHASE2_CONTRACT_AUDIT_PATH.is_file()
assert PHASE2_CONTRACT_AUDIT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("PHASE 2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Experiment contract       : LOCKED")
print("Vocabulary size           :", PHASE2_VOCABULARY_SIZE)
print("Minimum frequency         : 5")
print("PAD/BOS/EOS/UNK IDs       : 0 / 1 / 2 / 3")
print("Maximum decoder steps     : 31")
print("Train Parquet rows        : 566747")
print("Validation Parquet rows   : 25010")
print("Image manifest rows       : 123287")
print("Test metadata rows        : 5000")
print("Test caption available    : False")
print("Membership overlap        : 0")
print("Caption payload read      : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Audit                     :", PHASE2_CONTRACT_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell3_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "build deterministic image preprocessing contracts"
)


LOAD LOCKED PHASE 1 CONTRACTS
Dataset                : MS COCO 2014
Split protocol         : Karpathy
Vocabulary size        : 10080
Minimum frequency      : 5
PAD/BOS/EOS/UNK        : 0 / 1 / 2 / 3
Maximum lexical tokens : 30
Maximum full sequence  : 32
Maximum decoder steps  : 31
Test Set status        : SEALED

PARQUET SCHEMA AND ROW-COUNT AUDIT
train         : 566,747 rows | 57 row groups | 9 columns
validation    : 25,010 rows | 3 row groups | 9 columns
test          : 5,000 rows | 1 row groups | 7 columns
image_manifest: 123,287 rows | 1 row groups | 5 columns

IMAGE-MANIFEST MEMBERSHIP AUDIT
Karpathy split counts : {'train': 82783, 'restval': 30504, 'val': 5000, 'test': 5000}
Experiment splits     : {'train': 113287, 'validation': 5000, 'held_out_test': 5000}
Source partitions     : {'train2014': 82783, 'val2014': 40504}
Train path map        : 113287
Validation path map   : 5000
Held-out Test IDs     : 5000
Sample paths missing  : 0

TEST PARQUET METADATA-ONLY AUDIT
Rows       

## Cell 4: Khóa preprocessing cho ba Student

### Mục tiêu

- Đăng ký MobileNetV3-Large, MobileViT-S và TinyViT-5M.
- Dùng chung kích thước ảnh RGB 224×224.
- Dùng chung augmentation hình học giữa ba Student.
- Dùng normalization tương ứng pretrained config của từng encoder.
- Khóa validation transform hoàn toàn deterministic.
- Kiểm tra seeded training transform có thể tái lập.
- Kiểm tra shape, dtype và NaN/Inf.
- Smoke test trên synthetic image và ba ảnh Validation thật.
- Không đọc caption.
- Không đọc ảnh Test.
- Chưa tải trọng số model và chưa huấn luyện.

### Preprocessing chung

Training:
- RandomResizedCrop 224×224.
- Scale từ 0,80 đến 1,00.
- Aspect ratio từ 3/4 đến 4/3.
- Horizontal flip với xác suất 0,5.
- Bicubic interpolation.
- Float32 và normalization.

Validation:
- Resize cạnh ngắn về 256.
- Center crop 224×224.
- Bicubic interpolation.
- Không có phép biến đổi ngẫu nhiên.

In [4]:
"""
PHASE 2 — CELL 4
Deterministic and comparable preprocessing contracts for S1/S2/S3.

Student encoders:
- S1: TorchVision MobileNetV3-Large.
- S2: timm MobileViT-S.
- S3: timm TinyViT-5M.

Không đọc caption, không đọc ảnh Test và không huấn luyện.
"""

import hashlib
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torchvision import transforms
from torchvision.models import (
    MobileNet_V3_Large_Weights,
)
from torchvision.transforms import InterpolationMode


CELL4_STARTED_AT = time.perf_counter()

COMMON_INPUT_SIZE = 224
COMMON_EVAL_RESIZE_SHORTER = 256
COMMON_INTERPOLATION = "bicubic"

TRAIN_RANDOM_RESIZED_CROP_SCALE = [0.80, 1.00]
TRAIN_RANDOM_RESIZED_CROP_RATIO = [
    3.0 / 4.0,
    4.0 / 3.0,
]
TRAIN_HORIZONTAL_FLIP_PROBABILITY = 0.50

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]


required_runtime_names = [
    "PHASE2_SEED",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_VALIDATION_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–3: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def select_exact_timm_model(
    candidates,
    diagnostic_pattern,
):
    available_pretrained_models = set(
        timm.list_models(pretrained=True)
    )

    for candidate in candidates:
        if candidate in available_pretrained_models:
            return candidate

    diagnostic_matches = sorted(
        model_name
        for model_name in available_pretrained_models
        if diagnostic_pattern in model_name
    )

    raise AssertionError(
        "Không tìm thấy timm pretrained model đã khóa. "
        f"Candidates={candidates}; "
        f"available_matches={diagnostic_matches[:30]}"
    )


MOBILEVIT_MODEL_NAME = select_exact_timm_model(
    candidates=[
        "mobilevit_s.cvnets_in1k",
        "mobilevit_s",
    ],
    diagnostic_pattern="mobilevit",
)

TINYVIT_MODEL_NAME = select_exact_timm_model(
    candidates=[
        "tiny_vit_5m_224.dist_in22k_ft_in1k",
        "tiny_vit_5m_224.in1k",
        "tiny_vit_5m_224",
    ],
    diagnostic_pattern="tiny_vit_5m",
)


mobilenet_weights = (
    MobileNet_V3_Large_Weights.IMAGENET1K_V2
)
mobilenet_weight_transform = mobilenet_weights.transforms()

mobilevit_pretrained_cfg = timm.get_pretrained_cfg(
    MOBILEVIT_MODEL_NAME
).to_dict()
tinyvit_pretrained_cfg = timm.get_pretrained_cfg(
    TINYVIT_MODEL_NAME
).to_dict()


def tuple_of_floats(values):
    return tuple(float(value) for value in values)


PHASE2_STUDENT_SPECS = {
    "S1": {
        "student_name": "MobileNetV3-Large",
        "library": "torchvision",
        "model_name": "mobilenet_v3_large",
        "pretrained_identifier": (
            "MobileNet_V3_Large_Weights.IMAGENET1K_V2"
        ),
        "input_size": COMMON_INPUT_SIZE,
        "pretrained_mean": tuple_of_floats(
            mobilenet_weight_transform.mean
        ),
        "pretrained_std": tuple_of_floats(
            mobilenet_weight_transform.std
        ),
        "selection_rationale": (
            "strong CNN mobile baseline with approximately "
            "5M encoder parameters"
        ),
    },
    "S2": {
        "student_name": "MobileViT-S",
        "library": "timm",
        "model_name": MOBILEVIT_MODEL_NAME,
        "pretrained_identifier": MOBILEVIT_MODEL_NAME,
        "input_size": COMMON_INPUT_SIZE,
        "pretrained_mean": tuple_of_floats(
            mobilevit_pretrained_cfg["mean"]
        ),
        "pretrained_std": tuple_of_floats(
            mobilevit_pretrained_cfg["std"]
        ),
        "selection_rationale": (
            "hybrid convolution-transformer mobile baseline"
        ),
    },
    "S3": {
        "student_name": "TinyViT-5M",
        "library": "timm",
        "model_name": TINYVIT_MODEL_NAME,
        "pretrained_identifier": TINYVIT_MODEL_NAME,
        "input_size": COMMON_INPUT_SIZE,
        "pretrained_mean": tuple_of_floats(
            tinyvit_pretrained_cfg["mean"]
        ),
        "pretrained_std": tuple_of_floats(
            tinyvit_pretrained_cfg["std"]
        ),
        "selection_rationale": (
            "compact transformer near the encoder-parameter "
            "budget of MobileNetV3-Large and MobileViT-S"
        ),
    },
}

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS

for student_id, student_spec in PHASE2_STUDENT_SPECS.items():
    assert student_spec["input_size"] == COMMON_INPUT_SIZE
    assert len(student_spec["pretrained_mean"]) == 3
    assert len(student_spec["pretrained_std"]) == 3
    assert all(
        math.isfinite(value)
        for value in student_spec["pretrained_mean"]
    )
    assert all(
        value > 0.0 and math.isfinite(value)
        for value in student_spec["pretrained_std"]
    )


def create_train_transform(mean, std):
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(
                size=COMMON_INPUT_SIZE,
                scale=tuple(
                    TRAIN_RANDOM_RESIZED_CROP_SCALE
                ),
                ratio=tuple(
                    TRAIN_RANDOM_RESIZED_CROP_RATIO
                ),
                interpolation=InterpolationMode.BICUBIC,
                antialias=True,
            ),
            transforms.RandomHorizontalFlip(
                p=TRAIN_HORIZONTAL_FLIP_PROBABILITY
            ),
            transforms.PILToTensor(),
            transforms.ConvertImageDtype(
                torch.float32
            ),
            transforms.Normalize(
                mean=mean,
                std=std,
            ),
        ]
    )


def create_validation_transform(mean, std):
    return transforms.Compose(
        [
            transforms.Resize(
                size=COMMON_EVAL_RESIZE_SHORTER,
                interpolation=InterpolationMode.BICUBIC,
                antialias=True,
            ),
            transforms.CenterCrop(
                size=COMMON_INPUT_SIZE
            ),
            transforms.PILToTensor(),
            transforms.ConvertImageDtype(
                torch.float32
            ),
            transforms.Normalize(
                mean=mean,
                std=std,
            ),
        ]
    )


PHASE2_TRAIN_TRANSFORMS = {}
PHASE2_VALIDATION_TRANSFORMS = {}

for student_id, student_spec in PHASE2_STUDENT_SPECS.items():
    PHASE2_TRAIN_TRANSFORMS[student_id] = (
        create_train_transform(
            mean=student_spec["pretrained_mean"],
            std=student_spec["pretrained_std"],
        )
    )
    PHASE2_VALIDATION_TRANSFORMS[student_id] = (
        create_validation_transform(
            mean=student_spec["pretrained_mean"],
            std=student_spec["pretrained_std"],
        )
    )


def deterministic_synthetic_rgb_image(
    width=371,
    height=293,
):
    horizontal = np.linspace(
        0,
        255,
        width,
        dtype=np.uint8,
    )
    vertical = np.linspace(
        0,
        255,
        height,
        dtype=np.uint8,
    )

    red = np.tile(horizontal, (height, 1))
    green = np.tile(
        vertical[:, None],
        (1, width),
    )
    blue = (
        (
            red.astype(np.uint16)
            + green.astype(np.uint16)
        )
        // 2
    ).astype(np.uint8)

    rgb_array = np.stack(
        [red, green, blue],
        axis=-1,
    )

    return Image.fromarray(rgb_array)


synthetic_image = deterministic_synthetic_rgb_image()

preprocessing_audit_rows = []

for student_id in EXPECTED_STUDENT_IDS:
    train_transform = PHASE2_TRAIN_TRANSFORMS[
        student_id
    ]
    validation_transform = (
        PHASE2_VALIDATION_TRANSFORMS[student_id]
    )

    validation_first = validation_transform(
        synthetic_image
    )
    validation_second = validation_transform(
        synthetic_image
    )

    validation_repeat_max_error = (
        validation_first
        .sub(validation_second)
        .abs()
        .max()
        .item()
    )

    # Reset cùng seed phải tái tạo đúng augmentation.
    torch.manual_seed(PHASE2_SEED)
    train_first = train_transform(synthetic_image)
    torch.manual_seed(PHASE2_SEED)
    train_second = train_transform(synthetic_image)

    seeded_train_repeat_max_error = (
        train_first
        .sub(train_second)
        .abs()
        .max()
        .item()
    )

    for tensor_name, tensor in [
        ("train", train_first),
        ("validation", validation_first),
    ]:
        assert tensor.dtype == torch.float32
        assert tuple(tensor.shape) == (
            3,
            COMMON_INPUT_SIZE,
            COMMON_INPUT_SIZE,
        )
        assert torch.isfinite(tensor).all(), (
            f"{student_id} {tensor_name} tensor "
            "có NaN/Inf."
        )

    assert validation_repeat_max_error == 0.0
    assert seeded_train_repeat_max_error == 0.0

    preprocessing_audit_rows.append(
        {
            "student_id": student_id,
            "student_name": (
                PHASE2_STUDENT_SPECS[student_id]
                ["student_name"]
            ),
            "model_name": (
                PHASE2_STUDENT_SPECS[student_id]
                ["model_name"]
            ),
            "train_shape": list(train_first.shape),
            "validation_shape": list(
                validation_first.shape
            ),
            "train_minimum": train_first.min().item(),
            "train_maximum": train_first.max().item(),
            "validation_minimum":
                validation_first.min().item(),
            "validation_maximum":
                validation_first.max().item(),
            "validation_repeat_max_error":
                validation_repeat_max_error,
            "seeded_train_repeat_max_error":
                seeded_train_repeat_max_error,
            "outputs_finite": True,
        }
    )


# Smoke test một số ảnh Validation thực; không đọc caption.
validation_sample_items = list(
    PHASE2_VALIDATION_IMAGE_PATH_BY_ID.items()
)[:3]

assert len(validation_sample_items) == 3

real_image_audit = []

for image_id, image_path in validation_sample_items:
    assert image_id not in PHASE2_HELD_OUT_TEST_IMAGE_IDS
    assert Path(image_path).is_file()

    with Image.open(image_path) as source_image:
        rgb_image = source_image.convert("RGB")
        source_size = list(rgb_image.size)

        for student_id in EXPECTED_STUDENT_IDS:
            first_tensor = (
                PHASE2_VALIDATION_TRANSFORMS[
                    student_id
                ](rgb_image)
            )
            second_tensor = (
                PHASE2_VALIDATION_TRANSFORMS[
                    student_id
                ](rgb_image)
            )

            repeat_max_error = (
                first_tensor
                .sub(second_tensor)
                .abs()
                .max()
                .item()
            )

            assert tuple(first_tensor.shape) == (
                3,
                COMMON_INPUT_SIZE,
                COMMON_INPUT_SIZE,
            )
            assert torch.isfinite(first_tensor).all()
            assert repeat_max_error == 0.0

            real_image_audit.append(
                {
                    "image_id": image_id,
                    "student_id": student_id,
                    "source_size": source_size,
                    "output_shape": list(
                        first_tensor.shape
                    ),
                    "minimum":
                        first_tensor.min().item(),
                    "maximum":
                        first_tensor.max().item(),
                    "mean":
                        first_tensor.mean().item(),
                    "repeat_max_error":
                        repeat_max_error,
                    "finite": True,
                }
            )


PHASE2_PREPROCESSING_DIR = (
    Path(PHASE2_CONFIG_DIR)
    / "preprocessing"
)
PHASE2_PREPROCESSING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREPROCESSING_CONTRACT_PATHS = {}

for student_id, student_spec in PHASE2_STUDENT_SPECS.items():
    contract = {
        "schema_version": "1.0.0",
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "student_id": student_id,
        **student_spec,
        "shared_geometry": {
            "input_size": [
                3,
                COMMON_INPUT_SIZE,
                COMMON_INPUT_SIZE,
            ],
            "color_mode": "RGB",
            "interpolation": COMMON_INTERPOLATION,
            "train": {
                "random_resized_crop_scale":
                    TRAIN_RANDOM_RESIZED_CROP_SCALE,
                "random_resized_crop_ratio":
                    TRAIN_RANDOM_RESIZED_CROP_RATIO,
                "horizontal_flip_probability":
                    TRAIN_HORIZONTAL_FLIP_PROBABILITY,
            },
            "validation": {
                "resize_shorter":
                    COMMON_EVAL_RESIZE_SHORTER,
                "center_crop": COMMON_INPUT_SIZE,
                "random_operation": False,
            },
        },
        "normalization": {
            "mean": list(
                student_spec["pretrained_mean"]
            ),
            "std": list(
                student_spec["pretrained_std"]
            ),
            "source": "pretrained model configuration",
        },
        "tensor": {
            "dtype": "float32",
            "layout": "CHW",
            "batch_layout": "NCHW",
        },
        "policy": {
            "validation_deterministic": True,
            "test_transform_if_unsealed":
                "identical_to_validation",
            "test_images_read": False,
            "caption_read": False,
            "training_started": False,
        },
    }

    contract_path = (
        PHASE2_PREPROCESSING_DIR
        / f"{student_id.lower()}_preprocessing_contract.json"
    )

    with contract_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            contract,
            file,
            ensure_ascii=False,
            indent=2,
        )

    PREPROCESSING_CONTRACT_PATHS[
        student_id
    ] = contract_path


cell4_elapsed_seconds = (
    time.perf_counter()
    - CELL4_STARTED_AT
)

preprocessing_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "shared_geometry": {
        "input_size": COMMON_INPUT_SIZE,
        "eval_resize_shorter":
            COMMON_EVAL_RESIZE_SHORTER,
        "interpolation": COMMON_INTERPOLATION,
        "train_crop_scale":
            TRAIN_RANDOM_RESIZED_CROP_SCALE,
        "train_crop_ratio":
            TRAIN_RANDOM_RESIZED_CROP_RATIO,
        "horizontal_flip_probability":
            TRAIN_HORIZONTAL_FLIP_PROBABILITY,
    },
    "selected_models": {
        student_id: {
            "student_name":
                student_spec["student_name"],
            "library": student_spec["library"],
            "model_name": student_spec["model_name"],
            "pretrained_identifier":
                student_spec["pretrained_identifier"],
            "mean": list(
                student_spec["pretrained_mean"]
            ),
            "std": list(
                student_spec["pretrained_std"]
            ),
        }
        for student_id, student_spec
        in PHASE2_STUDENT_SPECS.items()
    },
    "synthetic_image_audit":
        preprocessing_audit_rows,
    "real_validation_image_audit":
        real_image_audit,
    "contracts": {
        student_id: {
            "path": str(contract_path),
            "sha256": sha256_file(contract_path),
        }
        for student_id, contract_path
        in PREPROCESSING_CONTRACT_PATHS.items()
    },
    "gates": {
        "three_students_registered": True,
        "common_input_geometry": True,
        "validation_deterministic": True,
        "seeded_training_transform_repeatable": True,
        "all_output_shapes_valid": True,
        "all_outputs_finite": True,
        "real_validation_samples_passed": True,
    },
    "policy": {
        "validation_images_read": 3,
        "validation_captions_read": False,
        "test_images_read": False,
        "test_captions_read": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "training_started": False,
    },
    "elapsed_seconds": cell4_elapsed_seconds,
}

PHASE2_PREPROCESSING_AUDIT_PATH = (
    Path(PHASE2_AUDIT_DIR)
    / "phase2_preprocessing_contract_audit.json"
)

with PHASE2_PREPROCESSING_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preprocessing_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


for path in [
    *PREPROCESSING_CONTRACT_PATHS.values(),
    PHASE2_PREPROCESSING_AUDIT_PATH,
]:
    assert path.is_file()
    assert path.stat().st_size > 0


print("=" * 72)
print("PHASE 2 PREPROCESSING CONTRACTS")
print("=" * 72)
print("Shared input geometry : RGB 224 x 224")
print("Train crop scale      : [0.80, 1.00]")
print("Train flip probability: 0.50")
print("Validation geometry   : resize 256 + center crop 224")
print("Interpolation         : bicubic")

for student_id in EXPECTED_STUDENT_IDS:
    student_spec = PHASE2_STUDENT_SPECS[student_id]
    synthetic_row = next(
        row
        for row in preprocessing_audit_rows
        if row["student_id"] == student_id
    )

    print(f"\n{student_id} — {student_spec['student_name']}")
    print("  Library            :", student_spec["library"])
    print("  Model name         :", student_spec["model_name"])
    print("  Mean               :", student_spec["pretrained_mean"])
    print("  Std                :", student_spec["pretrained_std"])
    print("  Output shape       :", synthetic_row["validation_shape"])
    print(
        "  Eval repeat error  :",
        synthetic_row["validation_repeat_max_error"],
    )
    print(
        "  Seeded train error :",
        synthetic_row["seeded_train_repeat_max_error"],
    )
    print(
        "  Contract           :",
        PREPROCESSING_CONTRACT_PATHS[student_id],
    )


print("\n" + "=" * 72)
print("PHASE 2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student encoders          : 3")
print("S1                        : MobileNetV3-Large")
print("S2                        : MobileViT-S")
print("S3                        : TinyViT-5M")
print("Common input size         : 224 x 224")
print("Common train geometry     : LOCKED")
print("Validation deterministic  : True")
print("Seeded train repeatable   : True")
print("Output tensors finite     : True")
print("Validation images audited : 3")
print("Caption read              : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Audit                     :", PHASE2_PREPROCESSING_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell4_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "build Parquet-backed Dataset and dynamic collator"
)


PHASE 2 PREPROCESSING CONTRACTS
Shared input geometry : RGB 224 x 224
Train crop scale      : [0.80, 1.00]
Train flip probability: 0.50
Validation geometry   : resize 256 + center crop 224
Interpolation         : bicubic

S1 — MobileNetV3-Large
  Library            : torchvision
  Model name         : mobilenet_v3_large
  Mean               : (0.485, 0.456, 0.406)
  Std                : (0.229, 0.224, 0.225)
  Output shape       : [3, 224, 224]
  Eval repeat error  : 0.0
  Seeded train error : 0.0
  Contract           : /kaggle/working//phase2_student_baselines/configs/preprocessing/s1_preprocessing_contract.json

S2 — MobileViT-S
  Library            : timm
  Model name         : mobilevit_s.cvnets_in1k
  Mean               : (0.0, 0.0, 0.0)
  Std                : (1.0, 1.0, 1.0)
  Output shape       : [3, 224, 224]
  Eval repeat error  : 0.0
  Seeded train error : 0.0
  Contract           : /kaggle/working//phase2_student_baselines/configs/preprocessing/s2_preprocessing_contract.json

## Cell 5: Parquet Dataset và Dynamic Collator

### Mục tiêu

- Chỉ đọc các cột annotation ID, image ID, split, token IDs và truncation.
- Không đọc cột caption text.
- Không mở Test Parquet.
- Xây Parquet-backed caption index.
- Xây ImageCaptionDataset.
- Tạo decoder input và label bằng phép shift:
  - decoder_input = full_sequence[:-1]
  - label = full_sequence[1:]
- Dynamic right-padding theo batch.
- PAD ID và loss ignore index bằng 0.
- Tạo attention mask: real token = 1, padding = 0.
- Khóa worker seed và DataLoader generator.
- Smoke test Training/Validation cho S1, S2 và S3.
- Kiểm tra hai DataLoader cùng seed cho kết quả giống nhau.
- Chưa huấn luyện.

In [5]:
import json
import random
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


CELL5_STARTED_AT = time.perf_counter()

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_MAX_FULL_SEQUENCE = 32
EXPECTED_MAX_DECODER_STEPS = 31

EXPECTED_TRAIN_CAPTIONS = 566_747
EXPECTED_TRAIN_IMAGES = 113_287
EXPECTED_VALIDATION_CAPTIONS = 25_010
EXPECTED_VALIDATION_IMAGES = 5_000

SMOKE_BATCH_SIZE = 4
SMOKE_NUM_WORKERS = 2


required_runtime_names = [
    "PHASE2_SEED",
    "PHASE2_AUDIT_DIR",
    "PHASE1_TRAIN_PARQUET_PATH",
    "PHASE1_VALIDATION_PARQUET_PATH",
    "PHASE2_TRAIN_IMAGE_PATH_BY_ID",
    "PHASE2_VALIDATION_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_TRAIN_TRANSFORMS",
    "PHASE2_VALIDATION_TRANSFORMS",
    "PHASE2_VOCABULARY_SIZE",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–4: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)

assert PHASE2_VOCABULARY_SIZE == EXPECTED_VOCABULARY_SIZE


class CaptionParquetIndex:
    """
    Metadata/token index nằm trong RAM.

    Không đọc cột caption. Ảnh chỉ được mở trong Dataset.__getitem__.
    """

    REQUIRED_COLUMNS = [
        "annotation_id",
        "image_id",
        "karpathy_split",
        "token_ids",
        "was_truncated",
    ]

    def __init__(
        self,
        parquet_path,
        allowed_splits,
        expected_rows,
        expected_images,
        held_out_test_ids,
    ):
        self.parquet_path = Path(parquet_path)
        self.allowed_splits = set(allowed_splits)
        self.held_out_test_ids = set(
            held_out_test_ids
        )

        table = pq.read_table(
            self.parquet_path,
            columns=self.REQUIRED_COLUMNS,
        )

        assert table.num_rows == expected_rows

        self.annotation_ids = (
            table["annotation_id"]
            .combine_chunks()
            .to_numpy(zero_copy_only=False)
        )
        self.image_ids = (
            table["image_id"]
            .combine_chunks()
            .to_numpy(zero_copy_only=False)
        )
        self.karpathy_splits = (
            table["karpathy_split"]
            .combine_chunks()
        )
        self.token_ids = (
            table["token_ids"]
            .combine_chunks()
        )
        self.was_truncated = (
            table["was_truncated"]
            .combine_chunks()
            .to_numpy(zero_copy_only=False)
        )

        split_counts = Counter(
            self.karpathy_splits.to_pylist()
        )
        unique_image_ids = set(
            int(image_id)
            for image_id in self.image_ids
        )
        test_overlap = (
            unique_image_ids
            & self.held_out_test_ids
        )

        assert set(split_counts).issubset(
            self.allowed_splits
        ), (
            f"Unexpected splits: {split_counts}"
        )
        assert len(unique_image_ids) == expected_images
        assert not test_overlap

        self.row_count = table.num_rows
        self.image_count = len(unique_image_ids)
        self.split_counts = dict(split_counts)
        self.truncated_count = int(
            np.asarray(self.was_truncated).sum()
        )

        del table

    def __len__(self):
        return self.row_count

    def record(self, index):
        token_ids = self.token_ids[index].as_py()

        assert isinstance(token_ids, list)
        assert 2 <= len(token_ids) <= EXPECTED_MAX_FULL_SEQUENCE
        assert token_ids[0] == BOS_ID
        assert token_ids[-1] == EOS_ID
        assert PAD_ID not in token_ids
        assert all(
            isinstance(token_id, int)
            and 0 <= token_id < EXPECTED_VOCABULARY_SIZE
            for token_id in token_ids
        )

        return {
            "annotation_id": int(
                self.annotation_ids[index]
            ),
            "image_id": int(self.image_ids[index]),
            "karpathy_split":
                self.karpathy_splits[index].as_py(),
            "token_ids": token_ids,
            "was_truncated": bool(
                self.was_truncated[index]
            ),
        }


class ImageCaptionDataset(Dataset):
    def __init__(
        self,
        caption_index,
        image_path_by_id,
        image_transform,
        held_out_test_ids,
    ):
        self.caption_index = caption_index
        self.image_path_by_id = image_path_by_id
        self.image_transform = image_transform
        self.held_out_test_ids = set(
            held_out_test_ids
        )

        assert not (
            set(self.image_path_by_id)
            & self.held_out_test_ids
        )

    def __len__(self):
        return len(self.caption_index)

    def __getitem__(self, index):
        record = self.caption_index.record(index)
        image_id = record["image_id"]

        assert image_id not in self.held_out_test_ids

        image_path = self.image_path_by_id.get(
            image_id
        )

        if image_path is None:
            raise KeyError(
                f"Không có image path cho image_id={image_id}"
            )

        with Image.open(image_path) as source_image:
            rgb_image = source_image.convert("RGB")
            image_tensor = self.image_transform(
                rgb_image
            )

        token_ids = torch.tensor(
            record["token_ids"],
            dtype=torch.long,
        )

        assert image_tensor.dtype == torch.float32
        assert tuple(image_tensor.shape) == (
            3,
            224,
            224,
        )
        assert torch.isfinite(image_tensor).all()

        return {
            "image": image_tensor,
            "token_ids": token_ids,
            "image_id": image_id,
            "annotation_id": record["annotation_id"],
            "karpathy_split":
                record["karpathy_split"],
            "was_truncated":
                record["was_truncated"],
        }


def dynamic_caption_collator(samples):
    """
    Right-pad theo batch.

    full=[BOS,...,EOS]
    decoder_input=full[:-1]
    labels=full[1:]
    """

    assert samples

    images = torch.stack(
        [sample["image"] for sample in samples],
        dim=0,
    )

    decoder_input_sequences = [
        sample["token_ids"][:-1]
        for sample in samples
    ]
    label_sequences = [
        sample["token_ids"][1:]
        for sample in samples
    ]

    decoder_input_ids = pad_sequence(
        decoder_input_sequences,
        batch_first=True,
        padding_value=PAD_ID,
    )
    labels = pad_sequence(
        label_sequences,
        batch_first=True,
        padding_value=PAD_ID,
    )
    decoder_attention_mask = (
        decoder_input_ids != PAD_ID
    )

    sequence_lengths = torch.tensor(
        [
            sequence.numel()
            for sequence in decoder_input_sequences
        ],
        dtype=torch.long,
    )

    assert decoder_input_ids.shape == labels.shape
    assert decoder_input_ids.shape[1] <= EXPECTED_MAX_DECODER_STEPS
    assert torch.all(
        decoder_input_ids[:, 0] == BOS_ID
    )
    assert torch.all(
        sequence_lengths >= 1
    )

    for row_index, sequence_length in enumerate(
        sequence_lengths.tolist()
    ):
        # Label cuối cùng trước padding phải là EOS.
        assert (
            labels[
                row_index,
                sequence_length - 1,
            ].item()
            == EOS_ID
        )

    return {
        "images": images,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask":
            decoder_attention_mask,
        "labels": labels,
        "sequence_lengths": sequence_lengths,
        "image_ids": torch.tensor(
            [
                sample["image_id"]
                for sample in samples
            ],
            dtype=torch.long,
        ),
        "annotation_ids": torch.tensor(
            [
                sample["annotation_id"]
                for sample in samples
            ],
            dtype=torch.long,
        ),
        "karpathy_splits": [
            sample["karpathy_split"]
            for sample in samples
        ],
        "was_truncated": torch.tensor(
            [
                sample["was_truncated"]
                for sample in samples
            ],
            dtype=torch.bool,
        ),
    }


def seed_dataloader_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def create_dataloader(
    dataset,
    batch_size,
    shuffle,
    seed,
    num_workers,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        collate_fn=dynamic_caption_collator,
        worker_init_fn=seed_dataloader_worker,
        generator=generator,
        persistent_workers=False,
    )


def audit_batch(
    batch,
    allowed_splits,
    held_out_test_ids,
):
    batch_size = batch["images"].shape[0]

    assert tuple(batch["images"].shape[1:]) == (
        3,
        224,
        224,
    )
    assert batch["images"].dtype == torch.float32
    assert torch.isfinite(batch["images"]).all()

    assert batch["decoder_input_ids"].dtype == torch.long
    assert batch["labels"].dtype == torch.long
    assert (
        batch["decoder_attention_mask"].dtype
        == torch.bool
    )
    assert batch["decoder_input_ids"].shape == (
        batch_size,
        batch["decoder_input_ids"].shape[1],
    )
    assert (
        batch["decoder_input_ids"].shape
        == batch["labels"].shape
        == batch["decoder_attention_mask"].shape
    )
    assert (
        batch["decoder_input_ids"].shape[1]
        <= EXPECTED_MAX_DECODER_STEPS
    )
    assert batch["decoder_input_ids"].min().item() >= 0
    assert (
        batch["decoder_input_ids"].max().item()
        < EXPECTED_VOCABULARY_SIZE
    )

    non_padding_labels = batch["labels"][
        batch["labels"] != PAD_ID
    ]
    assert non_padding_labels.numel() > 0
    assert non_padding_labels.min().item() >= 0
    assert (
        non_padding_labels.max().item()
        < EXPECTED_VOCABULARY_SIZE
    )

    image_id_set = set(
        batch["image_ids"].tolist()
    )
    assert not (
        image_id_set
        & set(held_out_test_ids)
    )
    assert set(
        batch["karpathy_splits"]
    ).issubset(set(allowed_splits))

    mask_expected = (
        batch["decoder_input_ids"] != PAD_ID
    )
    assert torch.equal(
        batch["decoder_attention_mask"],
        mask_expected,
    )

    return {
        "batch_size": batch_size,
        "image_shape": list(batch["images"].shape),
        "decoder_shape": list(
            batch["decoder_input_ids"].shape
        ),
        "minimum_sequence_length": int(
            batch["sequence_lengths"].min().item()
        ),
        "maximum_sequence_length": int(
            batch["sequence_lengths"].max().item()
        ),
        "padding_token_count": int(
            (
                batch["decoder_input_ids"]
                == PAD_ID
            ).sum().item()
        ),
        "test_overlap": 0,
        "finite": True,
    }


print("=" * 72)
print("LOAD PARQUET CAPTION INDEXES")
print("=" * 72)
print("Caption column requested : False")
print("Test Parquet opened      : False")

PHASE2_TRAIN_CAPTION_INDEX = CaptionParquetIndex(
    parquet_path=PHASE1_TRAIN_PARQUET_PATH,
    allowed_splits={"train", "restval"},
    expected_rows=EXPECTED_TRAIN_CAPTIONS,
    expected_images=EXPECTED_TRAIN_IMAGES,
    held_out_test_ids=PHASE2_HELD_OUT_TEST_IMAGE_IDS,
)

PHASE2_VALIDATION_CAPTION_INDEX = CaptionParquetIndex(
    parquet_path=PHASE1_VALIDATION_PARQUET_PATH,
    allowed_splits={"val"},
    expected_rows=EXPECTED_VALIDATION_CAPTIONS,
    expected_images=EXPECTED_VALIDATION_IMAGES,
    held_out_test_ids=PHASE2_HELD_OUT_TEST_IMAGE_IDS,
)

print(
    "Training index   :",
    len(PHASE2_TRAIN_CAPTION_INDEX),
    "captions /",
    PHASE2_TRAIN_CAPTION_INDEX.image_count,
    "images",
)
print(
    "Training splits  :",
    PHASE2_TRAIN_CAPTION_INDEX.split_counts,
)
print(
    "Validation index :",
    len(PHASE2_VALIDATION_CAPTION_INDEX),
    "captions /",
    PHASE2_VALIDATION_CAPTION_INDEX.image_count,
    "images",
)
print(
    "Validation splits:",
    PHASE2_VALIDATION_CAPTION_INDEX.split_counts,
)


print("\n" + "=" * 72)
print("DATALOADER AND COLLATOR SMOKE TEST")
print("=" * 72)

student_batch_audits = {}

for student_id in ["S1", "S2", "S3"]:
    train_dataset = ImageCaptionDataset(
        caption_index=PHASE2_TRAIN_CAPTION_INDEX,
        image_path_by_id=(
            PHASE2_TRAIN_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_TRAIN_TRANSFORMS[student_id]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )

    validation_dataset = ImageCaptionDataset(
        caption_index=(
            PHASE2_VALIDATION_CAPTION_INDEX
        ),
        image_path_by_id=(
            PHASE2_VALIDATION_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_VALIDATION_TRANSFORMS[
                student_id
            ]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )

    train_loader = create_dataloader(
        dataset=train_dataset,
        batch_size=SMOKE_BATCH_SIZE,
        shuffle=True,
        seed=PHASE2_SEED,
        num_workers=SMOKE_NUM_WORKERS,
    )
    validation_loader = create_dataloader(
        dataset=validation_dataset,
        batch_size=SMOKE_BATCH_SIZE,
        shuffle=False,
        seed=PHASE2_SEED,
        num_workers=SMOKE_NUM_WORKERS,
    )

    train_batch = next(iter(train_loader))
    validation_batch = next(
        iter(validation_loader)
    )

    train_audit = audit_batch(
        batch=train_batch,
        allowed_splits={"train", "restval"},
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
    validation_audit = audit_batch(
        batch=validation_batch,
        allowed_splits={"val"},
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )

    student_batch_audits[student_id] = {
        "train": train_audit,
        "validation": validation_audit,
    }

    print(f"\n{student_id}")
    print(
        "  Train image shape :",
        train_audit["image_shape"],
    )
    print(
        "  Train text shape  :",
        train_audit["decoder_shape"],
    )
    print(
        "  Val image shape   :",
        validation_audit["image_shape"],
    )
    print(
        "  Val text shape    :",
        validation_audit["decoder_shape"],
    )
    print("  Test overlap      : 0")


print("\n" + "=" * 72)
print("TWO-LOADER DETERMINISM AUDIT — S1")
print("=" * 72)

determinism_dataset = ImageCaptionDataset(
    caption_index=PHASE2_TRAIN_CAPTION_INDEX,
    image_path_by_id=PHASE2_TRAIN_IMAGE_PATH_BY_ID,
    image_transform=PHASE2_TRAIN_TRANSFORMS["S1"],
    held_out_test_ids=PHASE2_HELD_OUT_TEST_IMAGE_IDS,
)

loader_first = create_dataloader(
    dataset=determinism_dataset,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    seed=PHASE2_SEED,
    num_workers=SMOKE_NUM_WORKERS,
)
loader_second = create_dataloader(
    dataset=determinism_dataset,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    seed=PHASE2_SEED,
    num_workers=SMOKE_NUM_WORKERS,
)

batch_first = next(iter(loader_first))
batch_second = next(iter(loader_second))

image_id_match = torch.equal(
    batch_first["image_ids"],
    batch_second["image_ids"],
)
annotation_id_match = torch.equal(
    batch_first["annotation_ids"],
    batch_second["annotation_ids"],
)
decoder_input_match = torch.equal(
    batch_first["decoder_input_ids"],
    batch_second["decoder_input_ids"],
)
label_match = torch.equal(
    batch_first["labels"],
    batch_second["labels"],
)
image_max_error = (
    batch_first["images"]
    .sub(batch_second["images"])
    .abs()
    .max()
    .item()
)

assert image_id_match
assert annotation_id_match
assert decoder_input_match
assert label_match
assert image_max_error == 0.0

print("Image IDs match      :", image_id_match)
print("Annotation IDs match :", annotation_id_match)
print("Decoder inputs match :", decoder_input_match)
print("Labels match         :", label_match)
print("Image maximum error  :", image_max_error)


# Runtime factories cho các cell model/smoke test tiếp theo.
PHASE2_DATASET_CLASS = ImageCaptionDataset
PHASE2_COLLATOR = dynamic_caption_collator
PHASE2_DATALOADER_FACTORY = create_dataloader


cell5_elapsed_seconds = (
    time.perf_counter()
    - CELL5_STARTED_AT
)

dataset_collator_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "parquet_indexes": {
        "training": {
            "caption_count":
                len(PHASE2_TRAIN_CAPTION_INDEX),
            "image_count":
                PHASE2_TRAIN_CAPTION_INDEX.image_count,
            "split_counts":
                PHASE2_TRAIN_CAPTION_INDEX.split_counts,
            "truncated_count":
                PHASE2_TRAIN_CAPTION_INDEX.truncated_count,
        },
        "validation": {
            "caption_count":
                len(PHASE2_VALIDATION_CAPTION_INDEX),
            "image_count":
                PHASE2_VALIDATION_CAPTION_INDEX.image_count,
            "split_counts":
                PHASE2_VALIDATION_CAPTION_INDEX.split_counts,
            "truncated_count":
                PHASE2_VALIDATION_CAPTION_INDEX.truncated_count,
        },
    },
    "collator": {
        "padding": "dynamic_right_padding",
        "pad_id": PAD_ID,
        "bos_id": BOS_ID,
        "eos_id": EOS_ID,
        "label_padding": PAD_ID,
        "loss_ignore_index": PAD_ID,
        "attention_mask_real": True,
        "maximum_decoder_steps":
            EXPECTED_MAX_DECODER_STEPS,
    },
    "smoke_batches": student_batch_audits,
    "determinism": {
        "student_id": "S1",
        "seed": PHASE2_SEED,
        "num_workers": SMOKE_NUM_WORKERS,
        "image_id_match": image_id_match,
        "annotation_id_match": annotation_id_match,
        "decoder_input_match": decoder_input_match,
        "label_match": label_match,
        "image_maximum_error": image_max_error,
    },
    "policy": {
        "parquet_caption_column_read": False,
        "test_parquet_opened": False,
        "test_image_read": False,
        "test_caption_read": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "training_started": False,
    },
    "elapsed_seconds": cell5_elapsed_seconds,
}

PHASE2_DATASET_COLLATOR_AUDIT_PATH = (
    Path(PHASE2_AUDIT_DIR)
    / "phase2_dataset_collator_audit.json"
)

with PHASE2_DATASET_COLLATOR_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        dataset_collator_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

assert PHASE2_DATASET_COLLATOR_AUDIT_PATH.is_file()
assert PHASE2_DATASET_COLLATOR_AUDIT_PATH.stat().st_size > 0


print("\n" + "=" * 72)
print("PHASE 2 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training caption index    : 566747")
print("Training image IDs        : 113287")
print("Validation caption index  : 25010")
print("Validation image IDs      : 5000")
print("Caption text column read  : False")
print("Dynamic right padding     : True")
print("Maximum decoder steps     : 31")
print("Label ignore index        : 0")
print("Student smoke batches     : 3 / 3 PASSED")
print("Two-loader determinism    : PASSED")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Audit                     :", PHASE2_DATASET_COLLATOR_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell5_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "define shared Transformer decoder contract"
)

LOAD PARQUET CAPTION INDEXES
Caption column requested : False
Test Parquet opened      : False
Training index   : 566747 captions / 113287 images
Training splits  : {'train': 414113, 'restval': 152634}
Validation index : 25010 captions / 5000 images
Validation splits: {'val': 25010}

DATALOADER AND COLLATOR SMOKE TEST

S1
  Train image shape : [4, 3, 224, 224]
  Train text shape  : [4, 14]
  Val image shape   : [4, 3, 224, 224]
  Val text shape    : [4, 13]
  Test overlap      : 0

S2
  Train image shape : [4, 3, 224, 224]
  Train text shape  : [4, 14]
  Val image shape   : [4, 3, 224, 224]
  Val text shape    : [4, 13]
  Test overlap      : 0

S3
  Train image shape : [4, 3, 224, 224]
  Train text shape  : [4, 14]
  Val image shape   : [4, 3, 224, 224]
  Val text shape    : [4, 13]
  Test overlap      : 0

TWO-LOADER DETERMINISM AUDIT — S1
Image IDs match      : True
Annotation IDs match : True
Decoder inputs match : True
Labels match         : True
Image maximum error  : 0.0

PHASE 2

## Cell 6: Shared Transformer Decoder Contract

### Mục tiêu

Cell này xây dựng một Transformer Decoder dùng chung cho cả ba Student:

- S1: MobileNetV3-Large
- S2: MobileViT-S
- S3: TinyViT-5M

Việc dùng chung decoder giúp phép so sánh tập trung vào năng lực của visual
encoder, tránh sai lệch do mỗi Student sử dụng một decoder khác nhau.

### Cấu hình khóa

- Vocabulary size: 10.080
- `d_model`: 256
- Attention heads: 8
- Dimension mỗi head: 32
- Decoder layers: 3
- Feed-forward dimension: 1.024
- Dropout: 0,1
- Activation: GELU
- Maximum decoder steps: 31
- Causal self-attention
- Cross-attention tới visual memory `[B,N,256]`
- Dynamic padding mask
- Weight tying giữa token embedding và output projection
- Decoder parameter budget: tối đa 7 triệu tham số

### Kiểm tra thực hiện

1. Kiểm tra cấu hình khớp Phase 1.
2. Kiểm tra causal mask và padding mask.
3. Chạy forward trên batch thật từ Cell 5.
4. Kiểm tra logits `[B,T,10080]`.
5. Tính Cross-Entropy với PAD được bỏ qua.
6. Chạy backward và kiểm tra gradient hữu hạn.
7. Kiểm tra eval determinism.
8. Kiểm tra visual-memory length 1, 49 và 196.
9. Lưu contract, checkpoint khởi tạo và audit.

### Quy tắc dữ liệu

- Không đọc Test Parquet.
- Không đọc Test image hoặc Test caption.
- Không sử dụng Flickr.
- Không bắt đầu huấn luyện full.

In [6]:
import hashlib
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


CELL6_STARTED_AT = time.perf_counter()

SHARED_DECODER_CONFIG = {
    "vocabulary_size": 10_080,
    "pad_id": 0,
    "bos_id": 1,
    "eos_id": 2,
    "unk_id": 3,
    "d_model": 256,
    "nhead": 8,
    "head_dimension": 32,
    "num_layers": 3,
    "dim_feedforward": 1_024,
    "dropout": 0.10,
    "activation": "gelu",
    "maximum_decoder_steps": 31,
    "memory_dimension": 256,
    "norm_first": True,
    "batch_first": True,
    "tie_input_output_embeddings": True,
}

MAXIMUM_DECODER_PARAMETER_BUDGET = 7_000_000


required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_CHECKPOINT_DIR",
    "PHASE2_TRAIN_CAPTION_INDEX",
    "PHASE2_DATASET_CLASS",
    "PHASE2_DATALOADER_FACTORY",
    "PHASE2_TRAIN_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_TRAIN_TRANSFORMS",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–5: "
    f"{missing_runtime_names}. "
    "Không restart runtime; hãy chạy lại cell còn thiếu."
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


class SharedCaptionTransformerDecoder(nn.Module):
    """
    Autoregressive Transformer decoder với cross-attention.

    Input:
      decoder_input_ids: [B, T]
      visual_memory: [B, N, 256]
      decoder_attention_mask: [B, T], True cho token thật

    Output:
      logits: [B, T, vocabulary_size]
    """

    def __init__(self, config):
        super().__init__()

        self.config = dict(config)
        self.vocabulary_size = int(
            config["vocabulary_size"]
        )
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )

        assert (
            self.d_model % int(config["nhead"])
            == 0
        )
        assert (
            self.d_model // int(config["nhead"])
            == int(config["head_dimension"])
        )
        assert (
            int(config["memory_dimension"])
            == self.d_model
        )

        self.token_embedding = nn.Embedding(
            num_embeddings=self.vocabulary_size,
            embedding_dim=self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=self.maximum_decoder_steps,
            embedding_dim=self.d_model,
        )
        self.embedding_dropout = nn.Dropout(
            p=float(config["dropout"])
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(
                config["dim_feedforward"]
            ),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=bool(
                config["batch_first"]
            ),
            norm_first=bool(config["norm_first"]),
            bias=True,
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer=decoder_layer,
            num_layers=int(config["num_layers"]),
            norm=None,
        )
        self.final_norm = nn.LayerNorm(self.d_model)

        # Weight tying: logits dùng chính token_embedding.weight.
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=self.d_model ** -0.5,
        )
        nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=0.02,
        )

        with torch.no_grad():
            self.token_embedding.weight[
                self.pad_id
            ].zero_()
            self.output_bias.zero_()

    def build_causal_mask(self, sequence_length, target_device):
        return torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=target_device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
        memory_attention_mask=None,
    ):
        assert decoder_input_ids.ndim == 2
        assert visual_memory.ndim == 3
        assert (
            decoder_input_ids.shape[0]
            == visual_memory.shape[0]
        )
        assert (
            decoder_input_ids.shape[1]
            <= self.maximum_decoder_steps
        )
        assert (
            visual_memory.shape[-1]
            == self.d_model
        )

        batch_size, sequence_length = (
            decoder_input_ids.shape
        )

        position_ids = torch.arange(
            sequence_length,
            device=decoder_input_ids.device,
            dtype=torch.long,
        ).unsqueeze(0).expand(batch_size, -1)

        target_embeddings = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(position_ids)
        )
        target_embeddings = self.embedding_dropout(
            target_embeddings
        )

        causal_mask = self.build_causal_mask(
            sequence_length=sequence_length,
            target_device=decoder_input_ids.device,
        )

        target_key_padding_mask = None

        if decoder_attention_mask is not None:
            assert (
                decoder_attention_mask.shape
                == decoder_input_ids.shape
            )
            target_key_padding_mask = (
                ~decoder_attention_mask.bool()
            )

        memory_key_padding_mask = None

        if memory_attention_mask is not None:
            assert (
                memory_attention_mask.shape
                == visual_memory.shape[:2]
            )
            memory_key_padding_mask = (
                ~memory_attention_mask.bool()
            )

        hidden_states = self.transformer_decoder(
            tgt=target_embeddings,
            memory=visual_memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=(
                target_key_padding_mask
            ),
            memory_key_padding_mask=(
                memory_key_padding_mask
            ),
        )
        hidden_states = self.final_norm(hidden_states)

        logits = F.linear(
            hidden_states,
            self.token_embedding.weight,
            self.output_bias,
        )

        assert logits.shape == (
            batch_size,
            sequence_length,
            self.vocabulary_size,
        )

        return logits


def count_unique_parameters(model):
    unique_parameter_ids = set()
    total = 0
    trainable = 0

    for parameter in model.parameters():
        parameter_identity = id(parameter)

        if parameter_identity in unique_parameter_ids:
            continue

        unique_parameter_ids.add(parameter_identity)
        total += parameter.numel()

        if parameter.requires_grad:
            trainable += parameter.numel()

    return total, trainable


def parameter_breakdown(model):
    groups = {
        "token_embedding": 0,
        "position_embedding": 0,
        "transformer_decoder": 0,
        "final_norm": 0,
        "output_bias": 0,
    }

    for name, parameter in model.named_parameters():
        if name.startswith("token_embedding."):
            group = "token_embedding"
        elif name.startswith("position_embedding."):
            group = "position_embedding"
        elif name.startswith("transformer_decoder."):
            group = "transformer_decoder"
        elif name.startswith("final_norm."):
            group = "final_norm"
        elif name == "output_bias":
            group = "output_bias"
        else:
            raise AssertionError(
                f"Unclassified decoder parameter: {name}"
            )

        groups[group] += parameter.numel()

    return groups


torch.manual_seed(PHASE2_SEED)
torch.cuda.manual_seed_all(PHASE2_SEED)

PHASE2_SHARED_DECODER_CLASS = (
    SharedCaptionTransformerDecoder
)
PHASE2_SHARED_DECODER_CONFIG = dict(
    SHARED_DECODER_CONFIG
)

shared_decoder = SharedCaptionTransformerDecoder(
    PHASE2_SHARED_DECODER_CONFIG
).to(device)

total_parameters, trainable_parameters = (
    count_unique_parameters(shared_decoder)
)
decoder_parameter_breakdown = parameter_breakdown(
    shared_decoder
)

assert total_parameters == trainable_parameters
assert (
    total_parameters
    <= MAXIMUM_DECODER_PARAMETER_BUDGET
), (
    f"Decoder vượt parameter budget: "
    f"{total_parameters:,} > "
    f"{MAXIMUM_DECODER_PARAMETER_BUDGET:,}"
)
assert (
    sum(decoder_parameter_breakdown.values())
    == total_parameters
)


print("=" * 72)
print("SHARED TRANSFORMER DECODER CONTRACT")
print("=" * 72)
print("Vocabulary size       :", SHARED_DECODER_CONFIG["vocabulary_size"])
print("d_model               :", SHARED_DECODER_CONFIG["d_model"])
print("Attention heads       :", SHARED_DECODER_CONFIG["nhead"])
print("Head dimension        :", SHARED_DECODER_CONFIG["head_dimension"])
print("Decoder layers        :", SHARED_DECODER_CONFIG["num_layers"])
print("Feed-forward dimension:", SHARED_DECODER_CONFIG["dim_feedforward"])
print("Dropout               :", SHARED_DECODER_CONFIG["dropout"])
print("Activation            :", SHARED_DECODER_CONFIG["activation"])
print("Maximum time steps    :", SHARED_DECODER_CONFIG["maximum_decoder_steps"])
print("Visual memory dim     :", SHARED_DECODER_CONFIG["memory_dimension"])
print("Tied input/output     :", SHARED_DECODER_CONFIG["tie_input_output_embeddings"])
print("Total parameters      :", f"{total_parameters:,}")
print("Trainable parameters  :", f"{trainable_parameters:,}")
print(
    "FP32 parameter size  :",
    f"{total_parameters * 4 / (1024 ** 2):.2f} MiB",
)
print("Parameter breakdown   :", decoder_parameter_breakdown)


# Tạo một batch thật từ Training index nhưng không đọc caption text.
decoder_smoke_dataset = PHASE2_DATASET_CLASS(
    caption_index=PHASE2_TRAIN_CAPTION_INDEX,
    image_path_by_id=PHASE2_TRAIN_IMAGE_PATH_BY_ID,
    image_transform=PHASE2_TRAIN_TRANSFORMS["S1"],
    held_out_test_ids=PHASE2_HELD_OUT_TEST_IMAGE_IDS,
)

decoder_smoke_loader = PHASE2_DATALOADER_FACTORY(
    dataset=decoder_smoke_dataset,
    batch_size=4,
    shuffle=True,
    seed=PHASE2_SEED,
    num_workers=2,
)

decoder_smoke_batch = next(
    iter(decoder_smoke_loader)
)

decoder_input_ids = decoder_smoke_batch[
    "decoder_input_ids"
].to(device, non_blocking=True)
decoder_attention_mask = decoder_smoke_batch[
    "decoder_attention_mask"
].to(device, non_blocking=True)
labels = decoder_smoke_batch[
    "labels"
].to(device, non_blocking=True)

batch_size = decoder_input_ids.shape[0]
visual_token_count = 49

torch.manual_seed(PHASE2_SEED + 600)
torch.cuda.manual_seed_all(PHASE2_SEED + 600)

visual_memory = torch.randn(
    batch_size,
    visual_token_count,
    SHARED_DECODER_CONFIG["memory_dimension"],
    device=device,
    dtype=torch.float32,
)
memory_attention_mask = torch.ones(
    batch_size,
    visual_token_count,
    device=device,
    dtype=torch.bool,
)


print("\n" + "=" * 72)
print("FORWARD / LOSS / BACKWARD SMOKE TEST")
print("=" * 72)

shared_decoder.train()
shared_decoder.zero_grad(set_to_none=True)

logits = shared_decoder(
    decoder_input_ids=decoder_input_ids,
    visual_memory=visual_memory,
    decoder_attention_mask=decoder_attention_mask,
    memory_attention_mask=memory_attention_mask,
)

loss = F.cross_entropy(
    logits.reshape(
        -1,
        SHARED_DECODER_CONFIG["vocabulary_size"],
    ),
    labels.reshape(-1),
    ignore_index=SHARED_DECODER_CONFIG["pad_id"],
)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

loss.backward()

gradient_squared_sum = 0.0
gradient_parameter_count = 0
missing_gradient_parameters = []
nonfinite_gradient_parameters = []

for name, parameter in shared_decoder.named_parameters():
    if not parameter.requires_grad:
        continue

    if parameter.grad is None:
        missing_gradient_parameters.append(name)
        continue

    gradient_parameter_count += parameter.numel()

    if not torch.isfinite(parameter.grad).all():
        nonfinite_gradient_parameters.append(name)
        continue

    gradient_squared_sum += (
        parameter.grad.detach().float().pow(2).sum().item()
    )

gradient_l2_norm = math.sqrt(
    gradient_squared_sum
)

assert not missing_gradient_parameters, (
    f"Parameters without gradient: "
    f"{missing_gradient_parameters}"
)
assert not nonfinite_gradient_parameters, (
    f"Non-finite gradients: "
    f"{nonfinite_gradient_parameters}"
)
assert gradient_l2_norm > 0.0
assert math.isfinite(gradient_l2_norm)

print("Decoder input shape :", list(decoder_input_ids.shape))
print("Visual memory shape :", list(visual_memory.shape))
print("Logits shape        :", list(logits.shape))
print("CE loss             :", f"{loss.item():.8f}")
print("Gradient L2 norm    :", f"{gradient_l2_norm:.8f}")
print("Missing gradients   :", len(missing_gradient_parameters))
print("Nonfinite gradients :", len(nonfinite_gradient_parameters))


print("\n" + "=" * 72)
print("EVAL DETERMINISM AND MEMORY-LENGTH AUDIT")
print("=" * 72)

shared_decoder.eval()

with torch.inference_mode():
    eval_logits_first = shared_decoder(
        decoder_input_ids=decoder_input_ids,
        visual_memory=visual_memory,
        decoder_attention_mask=decoder_attention_mask,
        memory_attention_mask=memory_attention_mask,
    )
    eval_logits_second = shared_decoder(
        decoder_input_ids=decoder_input_ids,
        visual_memory=visual_memory,
        decoder_attention_mask=decoder_attention_mask,
        memory_attention_mask=memory_attention_mask,
    )

eval_repeat_max_error = (
    eval_logits_first
    .sub(eval_logits_second)
    .abs()
    .max()
    .item()
)

memory_length_shapes = {}

with torch.inference_mode():
    for memory_length in [1, 49, 196]:
        memory = torch.randn(
            batch_size,
            memory_length,
            SHARED_DECODER_CONFIG["memory_dimension"],
            device=device,
            dtype=torch.float32,
        )
        memory_mask = torch.ones(
            batch_size,
            memory_length,
            device=device,
            dtype=torch.bool,
        )
        memory_logits = shared_decoder(
            decoder_input_ids=decoder_input_ids,
            visual_memory=memory,
            decoder_attention_mask=decoder_attention_mask,
            memory_attention_mask=memory_mask,
        )

        assert torch.isfinite(memory_logits).all()

        memory_length_shapes[str(memory_length)] = list(
            memory_logits.shape
        )

assert eval_repeat_max_error == 0.0

print("Eval repeat max error:", eval_repeat_max_error)
print("Memory-length shapes :", memory_length_shapes)


PHASE2_DECODER_DIR = (
    Path(PHASE2_CONFIG_DIR)
    / "decoder"
)
PHASE2_DECODER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SHARED_DECODER_CONTRACT_PATH = (
    PHASE2_DECODER_DIR
    / "shared_transformer_decoder_contract.json"
)

shared_decoder_contract = {
    "schema_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "architecture": PHASE2_SHARED_DECODER_CONFIG,
    "parameter_budget": {
        "maximum_parameters":
            MAXIMUM_DECODER_PARAMETER_BUDGET,
        "actual_total_parameters":
            total_parameters,
        "actual_trainable_parameters":
            trainable_parameters,
        "fp32_parameter_size_mib": (
            total_parameters * 4 / (1024 ** 2)
        ),
        "breakdown": decoder_parameter_breakdown,
    },
    "formulas": {
        "token_input": (
            "sqrt(d_model) * token_embedding + "
            "learned_position_embedding"
        ),
        "causal_mask": (
            "mask[i,j] = True when j > i"
        ),
        "output_logits": (
            "hidden @ token_embedding.weight.T + output_bias"
        ),
        "cross_entropy": (
            "-sum(valid positions) log p(target_t | target_<t, image)"
        ),
    },
    "design_rationale": {
        "d_model_256": (
            "matches visual-memory projection and mobile budget"
        ),
        "eight_heads": (
            "32 dimensions per head; divides d_model exactly"
        ),
        "three_layers": (
            "capacity/latency compromise shared by all students"
        ),
        "ffn_1024": (
            "standard 4x expansion from d_model"
        ),
        "dropout_0_1": (
            "regularization used only in training mode"
        ),
        "weight_tying": (
            "reduces vocabulary projection parameters and "
            "keeps input/output token semantics aligned"
        ),
    },
    "fairness_policy": {
        "same_decoder_for_s1_s2_s3": True,
        "decoder_hyperparameters_may_not_change_per_encoder":
            True,
        "visual_memory_contract": "[B,N,256]",
    },
    "data_policy": {
        "caption_text_read": False,
        "test_data_read": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}

with SHARED_DECODER_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        shared_decoder_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )


SHARED_DECODER_INITIAL_CHECKPOINT_PATH = (
    Path(PHASE2_CHECKPOINT_DIR)
    / "shared_decoder_initial_seed42.pt"
)

cpu_state_dict = {
    name: tensor.detach().cpu()
    for name, tensor in shared_decoder.state_dict().items()
}

torch.save(
    {
        "schema_version": "1.0.0",
        "seed": PHASE2_SEED,
        "config": PHASE2_SHARED_DECODER_CONFIG,
        "state_dict": cpu_state_dict,
    },
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH,
)

shared_decoder_contract_sha256 = sha256_file(
    SHARED_DECODER_CONTRACT_PATH
)
shared_decoder_checkpoint_sha256 = sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
)


cell6_elapsed_seconds = (
    time.perf_counter()
    - CELL6_STARTED_AT
)

decoder_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "architecture": PHASE2_SHARED_DECODER_CONFIG,
    "parameters": {
        "total": total_parameters,
        "trainable": trainable_parameters,
        "maximum_budget":
            MAXIMUM_DECODER_PARAMETER_BUDGET,
        "breakdown": decoder_parameter_breakdown,
    },
    "smoke_test": {
        "decoder_input_shape":
            list(decoder_input_ids.shape),
        "visual_memory_shape":
            list(visual_memory.shape),
        "logits_shape": list(logits.shape),
        "loss": loss.item(),
        "gradient_l2_norm": gradient_l2_norm,
        "missing_gradient_parameters":
            missing_gradient_parameters,
        "nonfinite_gradient_parameters":
            nonfinite_gradient_parameters,
        "logits_finite": True,
        "loss_finite": True,
        "gradients_finite": True,
    },
    "determinism": {
        "eval_repeat_max_error":
            eval_repeat_max_error,
        "passed": eval_repeat_max_error == 0.0,
    },
    "dynamic_memory": {
        "tested_lengths": [1, 49, 196],
        "logit_shapes": memory_length_shapes,
        "passed": True,
    },
    "artifacts": {
        "contract": {
            "path": str(
                SHARED_DECODER_CONTRACT_PATH
            ),
            "sha256":
                shared_decoder_contract_sha256,
        },
        "initial_checkpoint": {
            "path": str(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "sha256":
                shared_decoder_checkpoint_sha256,
            "size_bytes":
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
                .stat().st_size,
        },
    },
    "policy": {
        "encoder_loaded": False,
        "caption_text_read": False,
        "test_data_read": False,
        "test_set_status": "sealed",
        "test_used": False,
        "flickr_used": False,
        "optimizer_step_performed": False,
        "training_started": False,
    },
    "elapsed_seconds": cell6_elapsed_seconds,
}

PHASE2_DECODER_AUDIT_PATH = (
    Path(PHASE2_AUDIT_DIR)
    / "phase2_shared_decoder_audit.json"
)

with PHASE2_DECODER_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decoder_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )

for path in [
    SHARED_DECODER_CONTRACT_PATH,
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH,
    PHASE2_DECODER_AUDIT_PATH,
]:
    assert path.is_file()
    assert path.stat().st_size > 0


# Giải phóng smoke model/GPU; giữ class và config cho Cell 7.
del shared_decoder
del logits
del loss
del eval_logits_first
del eval_logits_second
del visual_memory
del memory_attention_mask
del cpu_state_dict

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("\n" + "=" * 72)
print("PHASE 2 CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Shared across S1/S2/S3    : True")
print("Vocabulary size           : 10080")
print("d_model / heads           : 256 / 8")
print("Decoder layers / FFN      : 3 / 1024")
print("Maximum decoder steps     : 31")
print("Visual memory contract    : [B, N, 256]")
print("Tied input/output weights : True")
print("Total decoder parameters  :", total_parameters)
print("Parameter budget passed   : True")
print("Forward logits finite     : True")
print("Cross-entropy finite      : True")
print("Backward gradients finite : True")
print("Eval deterministic        : True")
print("Dynamic memory lengths    : 1 / 49 / 196 PASSED")
print("Encoder loaded            : False")
print("Caption text read         : False")
print("Test data read            : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Contract                  :", SHARED_DECODER_CONTRACT_PATH)
print("Contract SHA-256          :", shared_decoder_contract_sha256)
print(
    "Initial checkpoint        :",
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH,
)
print("Checkpoint SHA-256        :", shared_decoder_checkpoint_sha256)
print("Audit                     :", PHASE2_DECODER_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell6_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : "
    "define encoder adapters and visual-memory contracts"
)

SHARED TRANSFORMER DECODER CONTRACT
Vocabulary size       : 10080
d_model               : 256
Attention heads       : 8
Head dimension        : 32
Decoder layers        : 3
Feed-forward dimension: 1024
Dropout               : 0.1
Activation            : gelu
Maximum time steps    : 31
Visual memory dim     : 256
Tied input/output     : True
Total parameters      : 5,759,328
Trainable parameters  : 5,759,328
FP32 parameter size  : 21.97 MiB
Parameter breakdown   : {'token_embedding': 2580480, 'position_embedding': 7936, 'transformer_decoder': 3160320, 'final_norm': 512, 'output_bias': 10080}

FORWARD / LOSS / BACKWARD SMOKE TEST


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Decoder input shape : [4, 14]
Visual memory shape : [4, 49, 256]
Logits shape        : [4, 14, 10080]
CE loss             : 11.67737198
Gradient L2 norm    : 6.89811130
Missing gradients   : 0
Nonfinite gradients : 0

EVAL DETERMINISM AND MEMORY-LENGTH AUDIT
Eval repeat max error: 0.0
Memory-length shapes : {'1': [4, 14, 10080], '49': [4, 14, 10080], '196': [4, 14, 10080]}

PHASE 2 CELL 6 STATUS
Status                    : PASSED
Shared across S1/S2/S3    : True
Vocabulary size           : 10080
d_model / heads           : 256 / 8
Decoder layers / FFN      : 3 / 1024
Maximum decoder steps     : 31
Visual memory contract    : [B, N, 256]
Tied input/output weights : True
Total decoder parameters  : 5759328
Parameter budget passed   : True
Forward logits finite     : True
Cross-entropy finite      : True
Backward gradients finite : True
Eval deterministic        : True
Dynamic memory lengths    : 1 / 49 / 196 PASSED
Encoder loaded            : False
Caption text read         : False
Test 

## Cell 7: Pretrained Encoder Adapters

### Mục tiêu

Xây dựng ba visual encoder pretrained:

- S1: MobileNetV3-Large
- S2: MobileViT-S
- S3: TinyViT-5M

Mỗi encoder phải trả về cùng một visual-memory contract:

`[batch, visual_tokens, 256]`

### Thành phần adapter

1. Pretrained visual backbone.
2. Loại bỏ classification head.
3. Trích xuất spatial feature hoặc visual token.
4. Chuyển feature về định dạng `[B,N,C]`.
5. Linear projection từ `C` về 256.
6. Learned visual-position embedding.
7. LayerNorm đầu ra.

### Kiểm tra

- Tải đúng pretrained weights.
- Dùng ảnh Validation thật.
- Không sử dụng Test image hoặc Test caption.
- Kiểm tra raw feature shape và layout.
- Kiểm tra visual memory `[B,N,256]`.
- Kiểm tra output hữu hạn.
- Chạy backward smoke test.
- Kiểm tra gradient hữu hạn.
- Kiểm tra hai forward eval giống nhau.
- Audit số lượng tham số.
- Giới hạn encoder + adapter dưới 8 triệu tham số.
- Lưu contract và checkpoint khởi tạo cho từng Student.

### Trạng thái dữ liệu

- Test Parquet không được mở.
- Test image không được đọc.
- Caption text không được đọc.
- Không sử dụng Flickr.
- Không thực hiện optimizer step.
- Chưa bắt đầu full training.

In [7]:
import gc
import hashlib
import json
import time
from datetime import datetime, timezone

import timm
import torch
import torch.nn as nn
import torchvision
from torchvision.models import (
    MobileNet_V3_Large_Weights,
    mobilenet_v3_large,
)


CELL7_STARTED_AT = time.perf_counter()

VISUAL_MEMORY_DIMENSION = 256
MAXIMUM_ENCODER_ADAPTER_PARAMETER_BUDGET = 8_000_000
SMOKE_BATCH_SIZE = 2
EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]

required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_CHECKPOINT_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_VALIDATION_TRANSFORMS",
    "PHASE2_VALIDATION_CAPTION_INDEX",
    "PHASE2_VALIDATION_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_DATASET_CLASS",
    "PHASE2_DATALOADER_FACTORY",
    "SHARED_DECODER_CONFIG",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–6: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 6."
)

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS
assert SHARED_DECODER_CONFIG["memory_dimension"] == (
    VISUAL_MEMORY_DIMENSION
)
assert SHARED_DECODER_CONFIG["d_model"] == (
    VISUAL_MEMORY_DIMENSION
)
assert not (
    set(PHASE2_VALIDATION_IMAGE_PATH_BY_ID)
    & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def count_unique_parameters(model):
    seen_storages = set()
    total = 0
    trainable = 0

    for parameter in model.parameters():
        storage_key = (
            parameter.untyped_storage().data_ptr(),
            parameter.storage_offset(),
            parameter.numel(),
        )

        if storage_key in seen_storages:
            continue

        seen_storages.add(storage_key)
        total += parameter.numel()

        if parameter.requires_grad:
            trainable += parameter.numel()

    return int(total), int(trainable)


def select_tensor_from_feature_output(feature_output):
    """
    timm có thể trả Tensor, tuple/list hoặc dict tùy backbone.
    Chọn tensor đặc trưng cuối cùng một cách tường minh.
    """

    if torch.is_tensor(feature_output):
        return feature_output

    if isinstance(feature_output, dict):
        tensor_values = [
            value
            for value in feature_output.values()
            if torch.is_tensor(value)
        ]
        assert tensor_values, (
            "Backbone trả dict nhưng không có tensor đặc trưng."
        )
        return tensor_values[-1]

    if isinstance(feature_output, (tuple, list)):
        tensor_values = [
            value
            for value in feature_output
            if torch.is_tensor(value)
        ]
        assert tensor_values, (
            "Backbone trả tuple/list nhưng không có tensor đặc trưng."
        )
        return tensor_values[-1]

    raise TypeError(
        "Kiểu output backbone không được hỗ trợ: "
        f"{type(feature_output).__name__}"
    )


def infer_feature_layout(feature_tensor, expected_feature_dim=None):
    shape = tuple(feature_tensor.shape)

    if feature_tensor.ndim == 4:
        if (
            expected_feature_dim is not None
            and shape[1] == expected_feature_dim
        ):
            return "BCHW"

        if (
            expected_feature_dim is not None
            and shape[-1] == expected_feature_dim
        ):
            return "BHWC"

        # Với feature map ảnh, channel thường lớn hơn kích thước không gian.
        if shape[1] > shape[-1]:
            return "BCHW"

        return "BHWC"

    if feature_tensor.ndim == 3:
        if (
            expected_feature_dim is not None
            and shape[-1] == expected_feature_dim
        ):
            return "BNC"

        if (
            expected_feature_dim is not None
            and shape[1] == expected_feature_dim
        ):
            return "BCN"

        return "BNC"

    if feature_tensor.ndim == 2:
        return "BC"

    raise AssertionError(
        "Feature backbone phải có rank 2, 3 hoặc 4; "
        f"nhận được shape={shape}."
    )


def feature_tensor_to_tokens(feature_tensor, layout):
    """
    Trả visual tokens theo format [B, N, C].
    """

    if layout == "BCHW":
        tokens = feature_tensor.flatten(2).transpose(1, 2)
    elif layout == "BHWC":
        tokens = feature_tensor.flatten(1, 2)
    elif layout == "BNC":
        tokens = feature_tensor
    elif layout == "BCN":
        tokens = feature_tensor.transpose(1, 2)
    elif layout == "BC":
        tokens = feature_tensor.unsqueeze(1)
    else:
        raise ValueError(f"Feature layout không hợp lệ: {layout}")

    assert tokens.ndim == 3
    return tokens.contiguous()


class StudentVisualEncoderAdapter(nn.Module):
    """
    Backbone pretrained + linear projection + learned visual positions + LN.

    Output duy nhất:
        visual_memory: float tensor [B, N, 256]
    """

    def __init__(
        self,
        student_id,
        student_name,
        backbone,
        extraction_method,
        feature_layout,
        source_feature_dimension,
        visual_token_count,
        visual_memory_dimension=256,
    ):
        super().__init__()

        self.student_id = student_id
        self.student_name = student_name
        self.backbone = backbone
        self.extraction_method = extraction_method
        self.feature_layout = feature_layout
        self.source_feature_dimension = int(
            source_feature_dimension
        )
        self.visual_token_count = int(visual_token_count)
        self.visual_memory_dimension = int(
            visual_memory_dimension
        )

        self.projection = nn.Linear(
            self.source_feature_dimension,
            self.visual_memory_dimension,
        )
        self.visual_position_embedding = nn.Parameter(
            torch.empty(
                1,
                self.visual_token_count,
                self.visual_memory_dimension,
            )
        )
        self.output_norm = nn.LayerNorm(
            self.visual_memory_dimension
        )

        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(
            self.visual_position_embedding,
            mean=0.0,
            std=0.02,
        )

    def extract_features(self, images):
        if self.extraction_method == "torchvision_features":
            feature_output = self.backbone(images)
        elif self.extraction_method == "timm_forward_features":
            feature_output = self.backbone.forward_features(images)
        else:
            raise ValueError(
                "Extraction method không hợp lệ: "
                f"{self.extraction_method}"
            )

        return select_tensor_from_feature_output(
            feature_output
        )

    def forward(self, images):
        feature_tensor = self.extract_features(images)
        visual_tokens = feature_tensor_to_tokens(
            feature_tensor,
            self.feature_layout,
        )

        assert visual_tokens.shape[-1] == (
            self.source_feature_dimension
        ), (
            f"{self.student_id}: source feature dimension thay đổi, "
            f"expected={self.source_feature_dimension}, "
            f"actual={visual_tokens.shape[-1]}"
        )
        assert visual_tokens.shape[1] == (
            self.visual_token_count
        ), (
            f"{self.student_id}: visual token count thay đổi, "
            f"expected={self.visual_token_count}, "
            f"actual={visual_tokens.shape[1]}"
        )

        visual_memory = self.projection(visual_tokens)
        visual_memory = (
            visual_memory
            + self.visual_position_embedding
        )
        visual_memory = self.output_norm(visual_memory)

        assert visual_memory.shape[-1] == (
            self.visual_memory_dimension
        )

        return visual_memory


def build_pretrained_backbone(student_id, student_spec):
    """
    Tải đúng pretrained backbone đã khóa ở Cell 4.
    Classification head bị loại khỏi adapter.
    """

    try:
        if student_id == "S1":
            weights = (
                MobileNet_V3_Large_Weights.IMAGENET1K_V2
            )
            full_model = mobilenet_v3_large(
                weights=weights
            )
            backbone = full_model.features
            pretrained_source = weights.url
            expected_feature_dim = None
            extraction_method = "torchvision_features"

            del full_model

        else:
            backbone = timm.create_model(
                student_spec["model_name"],
                pretrained=True,
            )

            if hasattr(backbone, "reset_classifier"):
                backbone.reset_classifier(
                    num_classes=0,
                    global_pool="",
                )

            # forward_features() bỏ qua toàn bộ classifier head. Một số
            # kiến trúc (đặc biệt TinyViT) vẫn giữ các affine parameter
            # trong NormMlpClassifierHead sau reset_classifier(0, "").
            # Xóa hẳn các head này để checkpoint/budget không chứa tham số
            # trainable nhưng không bao giờ tham gia captioning forward.
            removed_head_modules = []
            for head_name in [
                "head",
                "head_dist",
                "classifier",
                "fc",
            ]:
                head_module = getattr(
                    backbone,
                    head_name,
                    None,
                )
                if (
                    isinstance(head_module, nn.Module)
                    and not isinstance(
                        head_module,
                        nn.Identity,
                    )
                ):
                    setattr(
                        backbone,
                        head_name,
                        nn.Identity(),
                    )
                    removed_head_modules.append(
                        head_name
                    )

            pretrained_cfg = (
                getattr(backbone, "pretrained_cfg", {})
                or {}
            )
            pretrained_source = (
                pretrained_cfg.get("hf_hub_id")
                or pretrained_cfg.get("url")
                or student_spec[
                    "pretrained_identifier"
                ]
            )
            expected_feature_dim = getattr(
                backbone,
                "num_features",
                None,
            )
            extraction_method = "timm_forward_features"

    except Exception as error:
        raise RuntimeError(
            f"Không tải được pretrained weights cho {student_id} "
            f"({student_spec['model_name']}). "
            "Hãy bật Internet cho Kaggle hoặc gắn weights như một "
            "Kaggle Input rồi chạy lại cell."
        ) from error

    return {
        "backbone": backbone,
        "pretrained_source": str(pretrained_source),
        "expected_feature_dim": (
            int(expected_feature_dim)
            if expected_feature_dim is not None
            else None
        ),
        "extraction_method": extraction_method,
        "removed_head_modules": (
            removed_head_modules
            if student_id != "S1"
            else ["classifier"]
        ),
    }


def build_validation_smoke_batch(student_id):
    validation_dataset = PHASE2_DATASET_CLASS(
        caption_index=PHASE2_VALIDATION_CAPTION_INDEX,
        image_path_by_id=(
            PHASE2_VALIDATION_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_VALIDATION_TRANSFORMS[student_id]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
    validation_loader = PHASE2_DATALOADER_FACTORY(
        dataset=validation_dataset,
        batch_size=SMOKE_BATCH_SIZE,
        shuffle=False,
        seed=PHASE2_SEED,
        num_workers=0,
    )
    batch = next(iter(validation_loader))

    assert not (
        set(batch["image_ids"].tolist())
        & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
    )

    return batch


def build_adapter_from_probe(
    student_id,
    student_spec,
    images,
):
    torch.manual_seed(PHASE2_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(PHASE2_SEED)

    backbone_bundle = build_pretrained_backbone(
        student_id,
        student_spec,
    )
    backbone = backbone_bundle["backbone"].to(device)
    backbone.eval()

    with torch.inference_mode():
        if (
            backbone_bundle["extraction_method"]
            == "torchvision_features"
        ):
            raw_output = backbone(images[:1])
        else:
            raw_output = backbone.forward_features(
                images[:1]
            )

        feature_tensor = select_tensor_from_feature_output(
            raw_output
        )
        feature_layout = infer_feature_layout(
            feature_tensor,
            backbone_bundle["expected_feature_dim"],
        )
        visual_tokens = feature_tensor_to_tokens(
            feature_tensor,
            feature_layout,
        )

    source_feature_shape = list(feature_tensor.shape)
    source_feature_dimension = int(
        visual_tokens.shape[-1]
    )
    visual_token_count = int(visual_tokens.shape[1])

    del raw_output
    del feature_tensor
    del visual_tokens

    adapter = StudentVisualEncoderAdapter(
        student_id=student_id,
        student_name=student_spec["student_name"],
        backbone=backbone,
        extraction_method=(
            backbone_bundle["extraction_method"]
        ),
        feature_layout=feature_layout,
        source_feature_dimension=(
            source_feature_dimension
        ),
        visual_token_count=visual_token_count,
        visual_memory_dimension=(
            VISUAL_MEMORY_DIMENSION
        ),
    ).to(device)

    return adapter, {
        "source_feature_shape": source_feature_shape,
        "source_feature_layout": feature_layout,
        "source_feature_dimension": (
            source_feature_dimension
        ),
        "visual_token_count": visual_token_count,
        "pretrained_source": (
            backbone_bundle["pretrained_source"]
        ),
        "extraction_method": (
            backbone_bundle["extraction_method"]
        ),
        "removed_head_modules": backbone_bundle[
            "removed_head_modules"
        ],
    }


def gradient_audit(model):
    missing_names = []
    nonfinite_names = []
    gradient_square_sum = 0.0
    parameters_with_gradient = 0

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue

        if parameter.grad is None:
            missing_names.append(name)
            continue

        parameters_with_gradient += parameter.numel()

        if not torch.isfinite(parameter.grad).all():
            nonfinite_names.append(name)
            continue

        gradient_square_sum += (
            parameter.grad.detach().float().square().sum().item()
        )

    return {
        "missing_gradient_names": missing_names,
        "nonfinite_gradient_names": nonfinite_names,
        "parameters_with_gradient": int(
            parameters_with_gradient
        ),
        "gradient_l2_norm": float(
            gradient_square_sum ** 0.5
        ),
    }


def enable_training_graph_with_frozen_batchnorm(model):
    """
    Dùng training graph thật nhưng không cập nhật BatchNorm running stats.

    TinyViT chỉ nối attention_biases trực tiếp vào graph khi Attention ở
    training mode; eval mode dùng attention_bias_cache đã detach khỏi graph.
    """

    model.train()
    frozen_batchnorm_modules = 0

    for module in model.modules():
        if isinstance(
            module,
            nn.modules.batchnorm._BatchNorm,
        ):
            module.eval()
            frozen_batchnorm_modules += 1

    return frozen_batchnorm_modules


PHASE2_ENCODER_CONFIG_DIR = (
    PHASE2_CONFIG_DIR / "encoders"
)
PHASE2_ENCODER_CHECKPOINT_DIR = (
    PHASE2_CHECKPOINT_DIR / "encoder_initial"
)

PHASE2_ENCODER_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
PHASE2_ENCODER_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PHASE2_ENCODER_ADAPTER_CLASS = (
    StudentVisualEncoderAdapter
)
PHASE2_ENCODER_CONTRACTS = {}
PHASE2_ENCODER_CONTRACT_PATHS = {}
PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS = {}

student_audits = {}

print("=" * 72)
print("PRETRAINED STUDENT ENCODER ADAPTER CONTRACTS")
print("=" * 72)
print()
print("Target visual memory : [B, N, 256]")
print("Pretrained backbones : True")
print("Backbone trainable   : True")
print("Projection trainable : True")
print("Test data read       : False")


for student_index, student_id in enumerate(
    EXPECTED_STUDENT_IDS,
    start=1,
):
    student_started_at = time.perf_counter()
    student_spec = PHASE2_STUDENT_SPECS[student_id]

    print()
    print(
        f"{student_id} — {student_spec['student_name']}"
    )
    print("-" * 72)

    smoke_batch = build_validation_smoke_batch(
        student_id
    )
    images = smoke_batch["images"].to(
        device,
        non_blocking=True,
    )

    assert tuple(images.shape) == (
        SMOKE_BATCH_SIZE,
        3,
        224,
        224,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    adapter, probe = build_adapter_from_probe(
        student_id=student_id,
        student_spec=student_spec,
        images=images,
    )

    total_parameters, trainable_parameters = (
        count_unique_parameters(adapter)
    )
    parameter_budget_passed = (
        total_parameters
        <= MAXIMUM_ENCODER_ADAPTER_PARAMETER_BUDGET
    )
    assert parameter_budget_passed, (
        f"{student_id}: encoder + adapter có "
        f"{total_parameters:,} parameters, vượt budget "
        f"{MAXIMUM_ENCODER_ADAPTER_PARAMETER_BUDGET:,}."
    )
    assert trainable_parameters == total_parameters

    # Backward phải dùng training graph để TinyViT attention_biases không
    # đi qua eval cache. Chỉ BatchNorm được giữ ở eval nhằm bảo toàn running
    # statistics của pretrained checkpoint.
    frozen_batchnorm_modules = (
        enable_training_graph_with_frozen_batchnorm(
            adapter
        )
    )
    adapter.zero_grad(set_to_none=True)

    torch.manual_seed(PHASE2_SEED + student_index)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            PHASE2_SEED + student_index
        )

    visual_memory = adapter(images)

    expected_memory_shape = [
        SMOKE_BATCH_SIZE,
        probe["visual_token_count"],
        VISUAL_MEMORY_DIMENSION,
    ]
    assert list(visual_memory.shape) == (
        expected_memory_shape
    )
    assert visual_memory.dtype == torch.float32
    assert torch.isfinite(visual_memory).all()

    # Không dùng mean(x^2) đơn thuần sau LayerNorm vì đại lượng đó gần
    # hằng số. Trọng số theo channel tạo tín hiệu gradient có ý nghĩa.
    audit_channel_weights = torch.linspace(
        -1.0,
        1.0,
        steps=VISUAL_MEMORY_DIMENSION,
        device=visual_memory.device,
        dtype=visual_memory.dtype,
    ).view(1, 1, -1)
    smoke_loss = (
        visual_memory
        * audit_channel_weights
    ).mean()
    assert torch.isfinite(smoke_loss)
    smoke_loss.backward()

    gradients = gradient_audit(adapter)
    assert not gradients["missing_gradient_names"], (
        f"{student_id}: còn trainable parameters không nhận gradient: "
        f"{gradients['missing_gradient_names']}. "
        "Không được chuyển sang end-to-end smoke test."
    )
    assert not gradients["nonfinite_gradient_names"]
    assert gradients["gradient_l2_norm"] > 0.0
    assert adapter.projection.weight.grad is not None
    assert torch.isfinite(
        adapter.projection.weight.grad
    ).all()

    # Determinism chỉ audit forward eval; không optimizer step.
    adapter.eval()
    adapter.zero_grad(set_to_none=True)
    with torch.inference_mode():
        repeat_memory_1 = adapter(images)
        repeat_memory_2 = adapter(images)

    repeat_maximum_error = float(
        (
            repeat_memory_1
            - repeat_memory_2
        ).abs().max().item()
    )
    assert repeat_maximum_error == 0.0

    output_mean = float(
        repeat_memory_1.float().mean().item()
    )
    output_std = float(
        repeat_memory_1.float().std().item()
    )
    assert output_std > 0.0

    peak_gpu_memory_bytes = (
        int(torch.cuda.max_memory_allocated())
        if torch.cuda.is_available()
        else 0
    )

    contract = {
        "contract_version": "phase2_encoder_adapter_v1",
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "student_id": student_id,
        "student_name": student_spec["student_name"],
        "library": student_spec["library"],
        "library_versions": {
            "torch": torch.__version__,
            "torchvision": torchvision.__version__,
            "timm": timm.__version__,
        },
        "model_name": student_spec["model_name"],
        "pretrained": True,
        "pretrained_identifier": student_spec[
            "pretrained_identifier"
        ],
        "pretrained_source": probe[
            "pretrained_source"
        ],
        "input_contract": {
            "shape": ["B", 3, 224, 224],
            "dtype": "float32",
            "preprocessing_contract": student_id,
        },
        "feature_extraction": {
            "method": probe["extraction_method"],
            "raw_layout": probe[
                "source_feature_layout"
            ],
            "raw_probe_shape": probe[
                "source_feature_shape"
            ],
            "source_feature_dimension": probe[
                "source_feature_dimension"
            ],
        },
        "adapter": {
            "removed_classification_heads": probe[
                "removed_head_modules"
            ],
            "projection": (
                f"{probe['source_feature_dimension']}"
                f"->{VISUAL_MEMORY_DIMENSION}"
            ),
            "learned_visual_position_embedding": True,
            "normalization": "LayerNorm",
            "backbone_trainable": True,
            "projection_trainable": True,
        },
        "output_contract": {
            "name": "visual_memory",
            "shape": [
                "B",
                probe["visual_token_count"],
                VISUAL_MEMORY_DIMENSION,
            ],
            "general_shape": [
                "B",
                "N",
                VISUAL_MEMORY_DIMENSION,
            ],
            "dtype": "float32",
            "finite_required": True,
        },
        "parameters": {
            "total": total_parameters,
            "trainable": trainable_parameters,
            "maximum_budget": (
                MAXIMUM_ENCODER_ADAPTER_PARAMETER_BUDGET
            ),
            "budget_passed": parameter_budget_passed,
        },
        "smoke_test": {
            "split": "validation",
            "backward_mode": (
                "train_with_batchnorm_eval"
            ),
            "frozen_batchnorm_module_count": (
                frozen_batchnorm_modules
            ),
            "batch_size": SMOKE_BATCH_SIZE,
            "image_ids": smoke_batch[
                "image_ids"
            ].tolist(),
            "test_overlap": 0,
            "visual_memory_shape": (
                expected_memory_shape
            ),
            "loss": float(smoke_loss.item()),
            "gradient_l2_norm": gradients[
                "gradient_l2_norm"
            ],
            "missing_gradient_parameter_count": len(
                gradients["missing_gradient_names"]
            ),
            "nonfinite_gradient_parameter_count": len(
                gradients["nonfinite_gradient_names"]
            ),
            "eval_repeat_maximum_error": (
                repeat_maximum_error
            ),
            "output_mean": output_mean,
            "output_std": output_std,
            "peak_gpu_memory_bytes": (
                peak_gpu_memory_bytes
            ),
        },
        "data_policy": {
            "caption_text_read": False,
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "flickr_used": False,
            "optimizer_step_performed": False,
            "training_started": False,
        },
        "seed": PHASE2_SEED,
    }

    contract_path = (
        PHASE2_ENCODER_CONFIG_DIR
        / f"{student_id.lower()}_encoder_adapter_contract.json"
    )
    checkpoint_path = (
        PHASE2_ENCODER_CHECKPOINT_DIR
        / f"{student_id.lower()}_encoder_adapter_initial_seed42.pt"
    )

    with open(
        contract_path,
        "w",
        encoding="utf-8",
    ) as contract_file:
        json.dump(
            contract,
            contract_file,
            ensure_ascii=False,
            indent=2,
        )

    cpu_state_dict = {
        name: tensor.detach().cpu().clone()
        for name, tensor in adapter.state_dict().items()
    }
    torch.save(
        {
            "student_id": student_id,
            "student_name": student_spec["student_name"],
            "seed": PHASE2_SEED,
            "contract": contract,
            "state_dict": cpu_state_dict,
        },
        checkpoint_path,
    )

    contract_sha256 = sha256_file(contract_path)
    checkpoint_sha256 = sha256_file(checkpoint_path)
    student_elapsed_seconds = (
        time.perf_counter()
        - student_started_at
    )

    student_audits[student_id] = {
        "status": "passed",
        "student_name": student_spec["student_name"],
        "model_name": student_spec["model_name"],
        "pretrained": True,
        "source_feature_shape": probe[
            "source_feature_shape"
        ],
        "source_feature_layout": probe[
            "source_feature_layout"
        ],
        "source_feature_dimension": probe[
            "source_feature_dimension"
        ],
        "visual_memory_shape": expected_memory_shape,
        "visual_token_count": probe[
            "visual_token_count"
        ],
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "parameter_budget_passed": (
            parameter_budget_passed
        ),
        "forward_finite": True,
        "backward_finite": True,
        "gradient_l2_norm": gradients[
            "gradient_l2_norm"
        ],
        "backward_mode": "train_with_batchnorm_eval",
        "frozen_batchnorm_module_count": (
            frozen_batchnorm_modules
        ),
        "missing_gradient_parameter_count": len(
            gradients["missing_gradient_names"]
        ),
        "nonfinite_gradient_parameter_count": len(
            gradients["nonfinite_gradient_names"]
        ),
        "eval_repeat_maximum_error": (
            repeat_maximum_error
        ),
        "output_mean": output_mean,
        "output_std": output_std,
        "peak_gpu_memory_bytes": (
            peak_gpu_memory_bytes
        ),
        "contract": str(contract_path),
        "contract_sha256": contract_sha256,
        "initial_checkpoint": str(checkpoint_path),
        "initial_checkpoint_sha256": (
            checkpoint_sha256
        ),
        "elapsed_seconds": student_elapsed_seconds,
    }

    PHASE2_ENCODER_CONTRACTS[student_id] = contract
    PHASE2_ENCODER_CONTRACT_PATHS[
        student_id
    ] = contract_path
    PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
        student_id
    ] = checkpoint_path

    print(
        "Model name          :",
        student_spec["model_name"],
    )
    print("Pretrained          : True")
    print(
        "Raw feature shape   :",
        probe["source_feature_shape"],
    )
    print(
        "Raw feature layout  :",
        probe["source_feature_layout"],
    )
    print(
        "Visual memory shape :",
        expected_memory_shape,
    )
    print(
        "Total parameters    :",
        f"{total_parameters:,}",
    )
    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}",
    )
    print(
        "Gradient L2 norm    :",
        f"{gradients['gradient_l2_norm']:.8f}",
    )
    print(
        "Backward mode       : train; BatchNorm eval"
    )
    print(
        "Frozen BatchNorm    :",
        frozen_batchnorm_modules,
    )
    print(
        "Missing gradients   :",
        len(gradients["missing_gradient_names"]),
    )
    print(
        "Nonfinite gradients :",
        len(gradients["nonfinite_gradient_names"]),
    )
    print(
        "Eval repeat error   :",
        repeat_maximum_error,
    )
    print(
        "Peak GPU memory     :",
        f"{peak_gpu_memory_bytes / (1024 ** 2):.2f} MiB",
    )
    print("Contract            :", contract_path)
    print("Contract SHA-256    :", contract_sha256)
    print("Initial checkpoint  :", checkpoint_path)
    print("Checkpoint SHA-256  :", checkpoint_sha256)
    print(
        "Elapsed             :",
        round(student_elapsed_seconds, 2),
        "giây",
    )

    del cpu_state_dict
    del repeat_memory_1
    del repeat_memory_2
    del visual_memory
    del smoke_loss
    del audit_channel_weights
    del images
    del smoke_batch
    del adapter

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


assert set(student_audits) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    audit["status"] == "passed"
    for audit in student_audits.values()
)
assert all(
    audit["visual_memory_shape"][-1]
    == VISUAL_MEMORY_DIMENSION
    for audit in student_audits.values()
)
assert all(
    audit["nonfinite_gradient_parameter_count"] == 0
    for audit in student_audits.values()
)
assert all(
    audit["missing_gradient_parameter_count"] == 0
    for audit in student_audits.values()
)
assert all(
    audit["eval_repeat_maximum_error"] == 0.0
    for audit in student_audits.values()
)

cell7_elapsed_seconds = (
    time.perf_counter()
    - CELL7_STARTED_AT
)

PHASE2_ENCODER_ADAPTER_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_encoder_adapter_contract_audit.json"
)

combined_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "visual_memory_contract": [
        "B",
        "N",
        VISUAL_MEMORY_DIMENSION,
    ],
    "shared_decoder_memory_dimension": (
        SHARED_DECODER_CONFIG["memory_dimension"]
    ),
    "student_ids": EXPECTED_STUDENT_IDS,
    "student_count": len(student_audits),
    "students": student_audits,
    "all_pretrained": True,
    "all_parameter_budgets_passed": True,
    "all_forward_outputs_finite": True,
    "all_backward_gradients_finite": True,
    "all_eval_deterministic": True,
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "optimizer_step_performed": False,
    "training_started": False,
    "elapsed_seconds": cell7_elapsed_seconds,
}

with open(
    PHASE2_ENCODER_ADAPTER_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        combined_audit,
        audit_file,
        ensure_ascii=False,
        indent=2,
    )

combined_audit_sha256 = sha256_file(
    PHASE2_ENCODER_ADAPTER_AUDIT_PATH
)

print()
print("=" * 72)
print("PHASE 2 CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student adapters          : 3 / 3 PASSED")
print("S1                        : MobileNetV3-Large")
print("S2                        : MobileViT-S")
print("S3                        : TinyViT-5M")
print("All pretrained            : True")
print("Visual memory contract    : [B, N, 256]")
print("Learned visual positions  : True")
print("Output LayerNorm          : True")
print("Parameter budgets         : 3 / 3 PASSED")
print("Forward outputs finite    : 3 / 3 PASSED")
print("Backward gradients finite : 3 / 3 PASSED")
print("Unused trainable params   : 0")
print("Eval deterministic        : 3 / 3 PASSED")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Audit                     :", PHASE2_ENCODER_ADAPTER_AUDIT_PATH)
print("Audit SHA-256             :", combined_audit_sha256)
print(
    "Elapsed                   :",
    round(cell7_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : connect each encoder adapter "
    "to the shared decoder and run end-to-end forward/loss/backward"
)


PRETRAINED STUDENT ENCODER ADAPTER CONTRACTS

Target visual memory : [B, N, 256]
Pretrained backbones : True
Backbone trainable   : True
Projection trainable : True
Test data read       : False

S1 — MobileNetV3-Large
------------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 175MB/s]


Model name          : mobilenet_v3_large
Pretrained          : True
Raw feature shape   : [1, 960, 7, 7]
Raw feature layout  : BCHW
Visual memory shape : [2, 49, 256]
Total parameters    : 3,231,024
Trainable parameters: 3,231,024
Gradient L2 norm    : 1.31688850
Backward mode       : train; BatchNorm eval
Frozen BatchNorm    : 46
Missing gradients   : 0
Nonfinite gradients : 0
Eval repeat error   : 0.0
Peak GPU memory     : 171.62 MiB
Contract            : /kaggle/working//phase2_student_baselines/configs/encoders/s1_encoder_adapter_contract.json
Contract SHA-256    : 327035a447a751a487de250797e1930bb2c8341b2b9ce0cb4a46692513b3fb75
Initial checkpoint  : /kaggle/working//phase2_student_baselines/checkpoints/encoder_initial/s1_encoder_adapter_initial_seed42.pt
Checkpoint SHA-256  : f35e5e31710173314366d6c8973ab49bc9293a60b680cae38f69f7b71a69d703
Elapsed             : 1.8 giây

S2 — MobileViT-S
------------------------------------------------------------------------


model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

Model name          : mobilevit_s.cvnets_in1k
Pretrained          : True
Raw feature shape   : [1, 640, 7, 7]
Raw feature layout  : BCHW
Visual memory shape : [2, 49, 256]
Total parameters    : 5,114,784
Trainable parameters: 5,114,784
Gradient L2 norm    : 1.11481918
Backward mode       : train; BatchNorm eval
Frozen BatchNorm    : 32
Missing gradients   : 0
Nonfinite gradients : 0
Eval repeat error   : 0.0
Peak GPU memory     : 358.40 MiB
Contract            : /kaggle/working//phase2_student_baselines/configs/encoders/s2_encoder_adapter_contract.json
Contract SHA-256    : 5759cb8f5980ca05aac1e66c6eff8fb4ff3b3eb4df16b0e286d64326a42220ba
Initial checkpoint  : /kaggle/working//phase2_student_baselines/checkpoints/encoder_initial/s2_encoder_adapter_initial_seed42.pt
Checkpoint SHA-256  : c5c9ab4aa8bb47df2aadb463de7ddb92228ce131e8ed384da6e7a81b63985dd2
Elapsed             : 3.19 giây

S3 — TinyViT-5M
------------------------------------------------------------------------


model.safetensors:   0%|          | 0.00/21.6M [00:00<?, ?B/s]

Model name          : tiny_vit_5m_224.dist_in22k_ft_in1k
Pretrained          : True
Raw feature shape   : [1, 320, 7, 7]
Raw feature layout  : BCHW
Visual memory shape : [2, 49, 256]
Total parameters    : 5,166,356
Trainable parameters: 5,166,356
Gradient L2 norm    : 1.58537531
Backward mode       : train; BatchNorm eval
Frozen BatchNorm    : 27
Missing gradients   : 0
Nonfinite gradients : 0
Eval repeat error   : 0.0
Peak GPU memory     : 275.71 MiB
Contract            : /kaggle/working//phase2_student_baselines/configs/encoders/s3_encoder_adapter_contract.json
Contract SHA-256    : c4ac3027da0f7e0a98ecc681eac5e99e7e24918261388b1f633d980afb406a8d
Initial checkpoint  : /kaggle/working//phase2_student_baselines/checkpoints/encoder_initial/s3_encoder_adapter_initial_seed42.pt
Checkpoint SHA-256  : b431a35542a1ca728cf3a2d2a74c20aa776e6d46b5960d40b806b17bf79fc6da
Elapsed             : 4.41 giây

PHASE 2 CELL 7 STATUS
Status                    : PASSED
Student adapters          : 3 / 3 PAS

## Cell 8: End-to-End Forward, Loss and Backward Audit

### Mục tiêu

Kết nối từng pretrained encoder adapter với cùng một Shared Transformer Decoder:

`Image → Encoder → Adapter → Visual Memory → Decoder → Token Logits`

### Pipeline được kiểm tra

- S1: MobileNetV3-Large + Shared Decoder
- S2: MobileViT-S + Shared Decoder
- S3: TinyViT-5M + Shared Decoder

### Kiểm tra checkpoint

- Xác minh SHA-256 trước khi đọc checkpoint.
- Strict-load ba encoder checkpoint.
- Strict-load shared decoder checkpoint.
- Cả ba Student phải sử dụng đúng cùng một decoder khởi tạo.

### Kiểm tra forward

- Input image: `[B,3,224,224]`
- Visual memory: `[B,49,256]`
- Decoder input: `[B,T]`, với `T <= 31`
- Token logits: `[B,T,10080]`
- Không có NaN hoặc Inf.

### Kiểm tra loss và backward

- Cross-Entropy, bỏ qua PAD=0.
- Backward qua toàn encoder, adapter và decoder.
- Mọi trainable parameter phải nhận gradient.
- Gradient phải hữu hạn.
- Không thực hiện optimizer step.
- BatchNorm của encoder được giữ ở eval để không đổi running statistics.

### Determinism

- Tắt Flash SDPA.
- Tắt Memory-Efficient SDPA.
- Bật Math SDPA.
- Bật deterministic algorithms ở hard-error mode.
- Hai lần forward eval phải có memory và logits repeat error bằng 0.

### Quy tắc dữ liệu

- Chỉ sử dụng Validation batch.
- Không đọc caption text.
- Không mở Test Parquet.
- Không đọc Test image.
- Không sử dụng Flickr.
- Chưa bắt đầu huấn luyện.

In [8]:
import gc
import hashlib
import json
import pickle
import time
from datetime import datetime, timezone

import torch
import torch.nn.functional as F


CELL8_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]
END_TO_END_SMOKE_BATCH_SIZE = 2
EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_VISUAL_MEMORY_DIMENSION = 256
EXPECTED_VISUAL_TOKEN_COUNT = 49
EXPECTED_MAXIMUM_DECODER_STEPS = 31

required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "PHASE2_AUDIT_DIR",
    "PHASE2_CHECKPOINT_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_VALIDATION_TRANSFORMS",
    "PHASE2_VALIDATION_CAPTION_INDEX",
    "PHASE2_VALIDATION_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_DATASET_CLASS",
    "PHASE2_DATALOADER_FACTORY",
    "PHASE2_SHARED_DECODER_CLASS",
    "PHASE2_SHARED_DECODER_CONFIG",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "PHASE2_ENCODER_CONTRACTS",
    "student_audits",
    "shared_decoder_checkpoint_sha256",
    "build_adapter_from_probe",
    "enable_training_graph_with_frozen_batchnorm",
    "gradient_audit",
    "count_unique_parameters",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–7: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 7."
)

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS
assert set(PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS) == set(
    EXPECTED_STUDENT_IDS
)
assert set(PHASE2_ENCODER_CONTRACTS) == set(
    EXPECTED_STUDENT_IDS
)
assert PHASE2_SHARED_DECODER_CONFIG[
    "vocabulary_size"
] == EXPECTED_VOCABULARY_SIZE
assert PHASE2_SHARED_DECODER_CONFIG[
    "memory_dimension"
] == EXPECTED_VISUAL_MEMORY_DIMENSION
assert PHASE2_SHARED_DECODER_CONFIG[
    "maximum_decoder_steps"
] == EXPECTED_MAXIMUM_DECODER_STEPS
assert not (
    set(PHASE2_VALIDATION_IMAGE_PATH_BY_ID)
    & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_owned_checkpoint(path, expected_sha256):
    """
    Các checkpoint này do Cell 6/7 vừa sinh và chỉ chứa tensor + primitive.
    weights_only=True ngăn unpickling object tùy ý.
    """

    path = path.resolve()
    checkpoint_root = PHASE2_CHECKPOINT_DIR.resolve()

    assert path.is_relative_to(checkpoint_root), (
        f"Checkpoint nằm ngoài Phase 2 checkpoint root: {path}"
    )
    assert path.is_file(), f"Không tìm thấy checkpoint: {path}"
    assert path.stat().st_size > 0

    actual_sha256 = sha256_file(path)
    assert actual_sha256 == expected_sha256, (
        f"Checkpoint SHA-256 không khớp: {path}; "
        f"expected={expected_sha256}, actual={actual_sha256}"
    )

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )
        load_mode = "weights_only"
    except pickle.UnpicklingError:
        # Checkpoint do Cell 7 tạo, nằm đúng output root và checksum đã
        # khớp audit. Fallback cần thiết nếu torch.__version__ cũ được
        # pickle dưới kiểu TorchVersion thay vì plain str.
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
        load_mode = "trusted_sha256_fallback"

    return checkpoint, load_mode


def build_validation_smoke_batch(student_id):
    validation_dataset = PHASE2_DATASET_CLASS(
        caption_index=PHASE2_VALIDATION_CAPTION_INDEX,
        image_path_by_id=(
            PHASE2_VALIDATION_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_VALIDATION_TRANSFORMS[student_id]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
    validation_loader = PHASE2_DATALOADER_FACTORY(
        dataset=validation_dataset,
        batch_size=END_TO_END_SMOKE_BATCH_SIZE,
        shuffle=False,
        seed=PHASE2_SEED,
        num_workers=0,
    )
    batch = next(iter(validation_loader))

    assert not (
        set(batch["image_ids"].tolist())
        & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
    )
    assert batch["decoder_input_ids"].shape == (
        batch["labels"].shape
    )
    assert (
        batch["decoder_input_ids"].shape[1]
        <= EXPECTED_MAXIMUM_DECODER_STEPS
    )

    return batch


def configure_strict_deterministic_attention():
    """
    Tắt Flash/Memory-Efficient SDPA trên CUDA và giữ Math SDPA.
    Math backend loại cảnh báo nondeterministic attention backward ở Cell 6.
    """

    backend_state = {
        "cuda_available": torch.cuda.is_available(),
        "flash_sdp_enabled": None,
        "memory_efficient_sdp_enabled": None,
        "math_sdp_enabled": None,
        "deterministic_algorithms_enabled": None,
    }

    if torch.cuda.is_available():
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)

        backend_state.update(
            {
                "flash_sdp_enabled": (
                    torch.backends.cuda.flash_sdp_enabled()
                ),
                "memory_efficient_sdp_enabled": (
                    torch.backends.cuda.mem_efficient_sdp_enabled()
                ),
                "math_sdp_enabled": (
                    torch.backends.cuda.math_sdp_enabled()
                ),
            }
        )

        assert not backend_state["flash_sdp_enabled"]
        assert not backend_state[
            "memory_efficient_sdp_enabled"
        ]
        assert backend_state["math_sdp_enabled"]

    torch.use_deterministic_algorithms(
        True,
        warn_only=False,
    )
    backend_state[
        "deterministic_algorithms_enabled"
    ] = torch.are_deterministic_algorithms_enabled()
    assert backend_state[
        "deterministic_algorithms_enabled"
    ]

    return backend_state


def build_and_load_encoder(student_id, images):
    student_spec = PHASE2_STUDENT_SPECS[student_id]
    encoder, probe = build_adapter_from_probe(
        student_id=student_id,
        student_spec=student_spec,
        images=images,
    )

    checkpoint_path = (
        PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
            student_id
        ]
    )
    checkpoint, load_mode = load_owned_checkpoint(
        checkpoint_path,
        student_audits[student_id][
            "initial_checkpoint_sha256"
        ],
    )

    assert checkpoint["student_id"] == student_id
    assert checkpoint["seed"] == PHASE2_SEED
    assert checkpoint["contract"]["student_id"] == (
        student_id
    )
    assert checkpoint["contract"]["output_contract"][
        "general_shape"
    ] == ["B", "N", EXPECTED_VISUAL_MEMORY_DIMENSION]

    incompatible = encoder.load_state_dict(
        checkpoint["state_dict"],
        strict=True,
    )
    assert not incompatible.missing_keys
    assert not incompatible.unexpected_keys

    return encoder, probe, checkpoint_path, load_mode


def build_and_load_shared_decoder():
    checkpoint, load_mode = load_owned_checkpoint(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH,
        shared_decoder_checkpoint_sha256,
    )

    assert checkpoint["seed"] == PHASE2_SEED
    assert checkpoint["config"] == (
        PHASE2_SHARED_DECODER_CONFIG
    )

    decoder = PHASE2_SHARED_DECODER_CLASS(
        PHASE2_SHARED_DECODER_CONFIG
    ).to(device)
    incompatible = decoder.load_state_dict(
        checkpoint["state_dict"],
        strict=True,
    )
    assert not incompatible.missing_keys
    assert not incompatible.unexpected_keys

    return decoder, load_mode


def named_gradient_audit(model, prefix):
    audit = gradient_audit(model)
    return {
        "component": prefix,
        "missing_gradient_names": [
            f"{prefix}.{name}"
            for name in audit["missing_gradient_names"]
        ],
        "nonfinite_gradient_names": [
            f"{prefix}.{name}"
            for name in audit["nonfinite_gradient_names"]
        ],
        "parameters_with_gradient": audit[
            "parameters_with_gradient"
        ],
        "gradient_l2_norm": audit[
            "gradient_l2_norm"
        ],
    }


def token_accuracy(logits, labels, pad_id):
    predictions = logits.argmax(dim=-1)
    valid_mask = labels != pad_id
    valid_count = int(valid_mask.sum().item())
    correct_count = int(
        (
            predictions.eq(labels)
            & valid_mask
        ).sum().item()
    )

    assert valid_count > 0

    return {
        "valid_token_count": valid_count,
        "correct_token_count": correct_count,
        "accuracy": correct_count / valid_count,
    }


def tensors_maximum_error(first, second):
    return float(
        (first - second).abs().max().item()
    )


deterministic_backend = (
    configure_strict_deterministic_attention()
)

decoder_checkpoint_sha256 = sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
)
encoder_checkpoint_sha256 = {
    student_id: sha256_file(
        PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
            student_id
        ]
    )
    for student_id in EXPECTED_STUDENT_IDS
}

print("=" * 72)
print("END-TO-END STUDENT SMOKE TEST")
print("=" * 72)
print()
print("Pipeline              : image -> encoder -> adapter -> decoder")
print("Validation batch size :", END_TO_END_SMOKE_BATCH_SIZE)
print("Vocabulary size       :", EXPECTED_VOCABULARY_SIZE)
print("Visual memory target  : [B, 49, 256]")
print("Decoder checkpoint    :", decoder_checkpoint_sha256)
print("Strict deterministic  : True")
print("Flash SDPA            :", deterministic_backend["flash_sdp_enabled"])
print(
    "Memory-efficient SDPA:",
    deterministic_backend["memory_efficient_sdp_enabled"],
)
print("Math SDPA             :", deterministic_backend["math_sdp_enabled"])
print("Test data read        : False")


student_results = {}

for student_index, student_id in enumerate(
    EXPECTED_STUDENT_IDS,
    start=1,
):
    student_started_at = time.perf_counter()
    student_spec = PHASE2_STUDENT_SPECS[student_id]

    print()
    print(
        f"{student_id} — {student_spec['student_name']}"
    )
    print("-" * 72)

    smoke_batch = build_validation_smoke_batch(
        student_id
    )

    images = smoke_batch["images"].to(
        device,
        non_blocking=True,
    )
    decoder_input_ids = smoke_batch[
        "decoder_input_ids"
    ].to(
        device,
        non_blocking=True,
    )
    decoder_attention_mask = smoke_batch[
        "decoder_attention_mask"
    ].to(
        device,
        non_blocking=True,
    )
    labels = smoke_batch["labels"].to(
        device,
        non_blocking=True,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    (
        encoder,
        encoder_probe,
        encoder_checkpoint_path,
        encoder_checkpoint_load_mode,
    ) = (
        build_and_load_encoder(
            student_id,
            images,
        )
    )
    decoder, decoder_checkpoint_load_mode = (
        build_and_load_shared_decoder()
    )

    encoder_parameters, encoder_trainable = (
        count_unique_parameters(encoder)
    )
    decoder_parameters, decoder_trainable = (
        count_unique_parameters(decoder)
    )
    assert encoder_parameters == encoder_trainable
    assert decoder_parameters == decoder_trainable

    frozen_batchnorm_modules = (
        enable_training_graph_with_frozen_batchnorm(
            encoder
        )
    )
    decoder.train()
    encoder.zero_grad(set_to_none=True)
    decoder.zero_grad(set_to_none=True)

    torch.manual_seed(
        PHASE2_SEED + 800 + student_index
    )
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            PHASE2_SEED + 800 + student_index
        )

    visual_memory = encoder(images)
    memory_attention_mask = torch.ones(
        visual_memory.shape[:2],
        dtype=torch.bool,
        device=device,
    )
    logits = decoder(
        decoder_input_ids=decoder_input_ids,
        visual_memory=visual_memory,
        decoder_attention_mask=(
            decoder_attention_mask
        ),
        memory_attention_mask=(
            memory_attention_mask
        ),
    )

    expected_visual_memory_shape = [
        END_TO_END_SMOKE_BATCH_SIZE,
        EXPECTED_VISUAL_TOKEN_COUNT,
        EXPECTED_VISUAL_MEMORY_DIMENSION,
    ]
    expected_logits_shape = [
        END_TO_END_SMOKE_BATCH_SIZE,
        decoder_input_ids.shape[1],
        EXPECTED_VOCABULARY_SIZE,
    ]

    assert list(visual_memory.shape) == (
        expected_visual_memory_shape
    )
    assert list(logits.shape) == expected_logits_shape
    assert torch.isfinite(visual_memory).all()
    assert torch.isfinite(logits).all()

    cross_entropy_loss = F.cross_entropy(
        logits.reshape(
            -1,
            EXPECTED_VOCABULARY_SIZE,
        ).float(),
        labels.reshape(-1),
        ignore_index=PHASE2_SHARED_DECODER_CONFIG[
            "pad_id"
        ],
    )
    assert torch.isfinite(cross_entropy_loss)
    cross_entropy_loss.backward()

    encoder_gradients = named_gradient_audit(
        encoder,
        "encoder",
    )
    decoder_gradients = named_gradient_audit(
        decoder,
        "decoder",
    )

    missing_gradient_names = (
        encoder_gradients["missing_gradient_names"]
        + decoder_gradients["missing_gradient_names"]
    )
    nonfinite_gradient_names = (
        encoder_gradients["nonfinite_gradient_names"]
        + decoder_gradients["nonfinite_gradient_names"]
    )

    assert not missing_gradient_names, (
        f"{student_id}: end-to-end parameters không nhận gradient: "
        f"{missing_gradient_names}"
    )
    assert not nonfinite_gradient_names, (
        f"{student_id}: gradient NaN/Inf: "
        f"{nonfinite_gradient_names}"
    )
    assert encoder_gradients["gradient_l2_norm"] > 0.0
    assert decoder_gradients["gradient_l2_norm"] > 0.0

    accuracy = token_accuracy(
        logits=logits.detach(),
        labels=labels,
        pad_id=PHASE2_SHARED_DECODER_CONFIG[
            "pad_id"
        ],
    )

    # Forward eval lặp lại trên đúng weights và input.
    encoder.eval()
    decoder.eval()
    encoder.zero_grad(set_to_none=True)
    decoder.zero_grad(set_to_none=True)

    with torch.inference_mode():
        eval_memory_first = encoder(images)
        eval_logits_first = decoder(
            decoder_input_ids=decoder_input_ids,
            visual_memory=eval_memory_first,
            decoder_attention_mask=(
                decoder_attention_mask
            ),
            memory_attention_mask=(
                memory_attention_mask
            ),
        )
        eval_memory_second = encoder(images)
        eval_logits_second = decoder(
            decoder_input_ids=decoder_input_ids,
            visual_memory=eval_memory_second,
            decoder_attention_mask=(
                decoder_attention_mask
            ),
            memory_attention_mask=(
                memory_attention_mask
            ),
        )

    memory_repeat_maximum_error = (
        tensors_maximum_error(
            eval_memory_first,
            eval_memory_second,
        )
    )
    logits_repeat_maximum_error = (
        tensors_maximum_error(
            eval_logits_first,
            eval_logits_second,
        )
    )
    assert memory_repeat_maximum_error == 0.0
    assert logits_repeat_maximum_error == 0.0

    peak_gpu_memory_bytes = (
        int(torch.cuda.max_memory_allocated())
        if torch.cuda.is_available()
        else 0
    )
    student_elapsed_seconds = (
        time.perf_counter()
        - student_started_at
    )

    result = {
        "status": "passed",
        "student_id": student_id,
        "student_name": student_spec["student_name"],
        "model_name": student_spec["model_name"],
        "pipeline": [
            "image",
            "pretrained_encoder",
            "visual_projection",
            "shared_transformer_decoder",
            "token_logits",
        ],
        "batch": {
            "split": "validation",
            "batch_size": END_TO_END_SMOKE_BATCH_SIZE,
            "image_ids": smoke_batch["image_ids"].tolist(),
            "annotation_ids": smoke_batch[
                "annotation_ids"
            ].tolist(),
            "test_overlap": 0,
            "image_shape": list(images.shape),
            "decoder_input_shape": list(
                decoder_input_ids.shape
            ),
            "label_shape": list(labels.shape),
        },
        "forward": {
            "raw_encoder_feature_shape": (
                encoder_probe["source_feature_shape"]
            ),
            "visual_memory_shape": (
                expected_visual_memory_shape
            ),
            "memory_attention_mask_shape": list(
                memory_attention_mask.shape
            ),
            "logits_shape": expected_logits_shape,
            "visual_memory_finite": True,
            "logits_finite": True,
        },
        "loss_and_backward": {
            "cross_entropy": float(
                cross_entropy_loss.item()
            ),
            "cross_entropy_finite": True,
            "valid_token_count": accuracy[
                "valid_token_count"
            ],
            "correct_token_count": accuracy[
                "correct_token_count"
            ],
            "teacher_forced_token_accuracy": accuracy[
                "accuracy"
            ],
            "encoder_gradient_l2_norm": (
                encoder_gradients["gradient_l2_norm"]
            ),
            "decoder_gradient_l2_norm": (
                decoder_gradients["gradient_l2_norm"]
            ),
            "missing_gradient_parameter_count": 0,
            "nonfinite_gradient_parameter_count": 0,
            "backward_mode": (
                "train_with_encoder_batchnorm_eval"
            ),
            "frozen_encoder_batchnorm_modules": (
                frozen_batchnorm_modules
            ),
            "optimizer_step_performed": False,
        },
        "determinism": {
            "strict_algorithms": True,
            "flash_sdp_enabled": (
                deterministic_backend[
                    "flash_sdp_enabled"
                ]
            ),
            "memory_efficient_sdp_enabled": (
                deterministic_backend[
                    "memory_efficient_sdp_enabled"
                ]
            ),
            "math_sdp_enabled": (
                deterministic_backend[
                    "math_sdp_enabled"
                ]
            ),
            "encoder_memory_repeat_maximum_error": (
                memory_repeat_maximum_error
            ),
            "end_to_end_logits_repeat_maximum_error": (
                logits_repeat_maximum_error
            ),
        },
        "parameters": {
            "encoder_adapter": encoder_parameters,
            "shared_decoder": decoder_parameters,
            "end_to_end_total": (
                encoder_parameters
                + decoder_parameters
            ),
        },
        "checkpoints": {
            "encoder_path": str(
                encoder_checkpoint_path
            ),
            "encoder_sha256": (
                encoder_checkpoint_sha256[
                    student_id
                ]
            ),
            "decoder_path": str(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "decoder_sha256": (
                decoder_checkpoint_sha256
            ),
            "strict_load": True,
            "encoder_load_mode": (
                encoder_checkpoint_load_mode
            ),
            "decoder_load_mode": (
                decoder_checkpoint_load_mode
            ),
        },
        "resources": {
            "peak_gpu_memory_bytes": (
                peak_gpu_memory_bytes
            ),
            "elapsed_seconds": (
                student_elapsed_seconds
            ),
        },
        "data_policy": {
            "caption_text_read": False,
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
            "training_started": False,
        },
    }

    student_results[student_id] = result

    print(
        "Image shape          :",
        list(images.shape),
    )
    print(
        "Visual memory shape  :",
        expected_visual_memory_shape,
    )
    print(
        "Decoder input shape  :",
        list(decoder_input_ids.shape),
    )
    print(
        "Logits shape         :",
        expected_logits_shape,
    )
    print(
        "Cross-entropy        :",
        f"{cross_entropy_loss.item():.8f}",
    )
    print(
        "Token accuracy       :",
        f"{accuracy['accuracy']:.8f}",
        "(diagnostic only)",
    )
    print(
        "Encoder gradient L2  :",
        f"{encoder_gradients['gradient_l2_norm']:.8f}",
    )
    print(
        "Decoder gradient L2  :",
        f"{decoder_gradients['gradient_l2_norm']:.8f}",
    )
    print("Missing gradients    : 0")
    print("Nonfinite gradients  : 0")
    print(
        "Memory repeat error  :",
        memory_repeat_maximum_error,
    )
    print(
        "Logits repeat error  :",
        logits_repeat_maximum_error,
    )
    print(
        "End-to-end parameters:",
        f"{encoder_parameters + decoder_parameters:,}",
    )
    print(
        "Peak GPU memory      :",
        f"{peak_gpu_memory_bytes / (1024 ** 2):.2f} MiB",
    )
    print(
        "Elapsed              :",
        round(student_elapsed_seconds, 2),
        "giây",
    )

    del result
    del eval_memory_first
    del eval_memory_second
    del eval_logits_first
    del eval_logits_second
    del logits
    del visual_memory
    del memory_attention_mask
    del cross_entropy_loss
    del images
    del decoder_input_ids
    del decoder_attention_mask
    del labels
    del encoder
    del decoder
    del smoke_batch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


assert set(student_results) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    result["status"] == "passed"
    for result in student_results.values()
)
assert all(
    result["forward"]["visual_memory_shape"]
    == [
        END_TO_END_SMOKE_BATCH_SIZE,
        EXPECTED_VISUAL_TOKEN_COUNT,
        EXPECTED_VISUAL_MEMORY_DIMENSION,
    ]
    for result in student_results.values()
)
assert all(
    result["forward"]["logits_shape"][2]
    == EXPECTED_VOCABULARY_SIZE
    for result in student_results.values()
)
assert all(
    result["loss_and_backward"][
        "missing_gradient_parameter_count"
    ] == 0
    for result in student_results.values()
)
assert all(
    result["loss_and_backward"][
        "nonfinite_gradient_parameter_count"
    ] == 0
    for result in student_results.values()
)
assert all(
    result["determinism"][
        "end_to_end_logits_repeat_maximum_error"
    ] == 0.0
    for result in student_results.values()
)

cell8_elapsed_seconds = (
    time.perf_counter()
    - CELL8_STARTED_AT
)

PHASE2_END_TO_END_SMOKE_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_end_to_end_smoke_test_audit.json"
)

combined_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "pipeline": (
        "image -> pretrained encoder -> adapter -> "
        "shared Transformer decoder -> logits"
    ),
    "student_ids": EXPECTED_STUDENT_IDS,
    "student_count": len(student_results),
    "students": student_results,
    "shared_decoder_checkpoint": {
        "path": str(
            SHARED_DECODER_INITIAL_CHECKPOINT_PATH
        ),
        "sha256": decoder_checkpoint_sha256,
        "same_checkpoint_for_all_students": True,
    },
    "deterministic_backend": deterministic_backend,
    "all_checkpoint_loads_strict": True,
    "all_visual_memory_contracts_passed": True,
    "all_logits_shape_contracts_passed": True,
    "all_losses_finite": True,
    "all_backward_gradients_finite": True,
    "all_trainable_parameters_received_gradient": True,
    "all_eval_outputs_deterministic": True,
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "optimizer_step_performed": False,
    "training_started": False,
    "elapsed_seconds": cell8_elapsed_seconds,
}

with open(
    PHASE2_END_TO_END_SMOKE_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        combined_audit,
        audit_file,
        ensure_ascii=False,
        indent=2,
    )

end_to_end_audit_sha256 = sha256_file(
    PHASE2_END_TO_END_SMOKE_AUDIT_PATH
)

print()
print("=" * 72)
print("PHASE 2 CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("End-to-end pipelines      : 3 / 3 PASSED")
print("S1                        : MobileNetV3-Large + shared decoder")
print("S2                        : MobileViT-S + shared decoder")
print("S3                        : TinyViT-5M + shared decoder")
print("Checkpoint strict loads   : 4 / 4 PASSED")
print("Shared decoder reused     : True")
print("Visual memory contract    : [B, 49, 256]")
print("Vocabulary logits         : [B, T, 10080]")
print("Cross-entropy finite      : 3 / 3 PASSED")
print("Backward gradients finite : 3 / 3 PASSED")
print("Unused trainable params   : 0")
print("Strict deterministic mode : True")
print("Flash SDPA disabled       : True")
print("Memory-efficient disabled : True")
print("Math SDPA enabled         : True")
print("Eval repeatability        : 3 / 3 PASSED")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Audit                     :", PHASE2_END_TO_END_SMOKE_AUDIT_PATH)
print("Audit SHA-256             :", end_to_end_audit_sha256)
print(
    "Elapsed                   :",
    round(cell8_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : run one-batch overfit "
    "for S1/S2/S3 before full baseline training"
)


END-TO-END STUDENT SMOKE TEST

Pipeline              : image -> encoder -> adapter -> decoder
Validation batch size : 2
Vocabulary size       : 10080
Visual memory target  : [B, 49, 256]
Decoder checkpoint    : ce1c202034343b8cc2efcbf34dffcdac6671ab221fe5210fbf1f5b00bf7b54d2
Strict deterministic  : True
Flash SDPA            : False
Memory-efficient SDPA: False
Math SDPA             : True
Test data read        : False

S1 — MobileNetV3-Large
------------------------------------------------------------------------
Image shape          : [2, 3, 224, 224]
Visual memory shape  : [2, 49, 256]
Decoder input shape  : [2, 13]
Logits shape         : [2, 13, 10080]
Cross-entropy        : 10.59953213
Token accuracy       : 0.00000000 (diagnostic only)
Encoder gradient L2  : 12.04579894
Decoder gradient L2  : 8.08932248
Missing gradients    : 0
Nonfinite gradients  : 0
Memory repeat error  : 0.0
Logits repeat error  : 0.0
End-to-end parameters: 8,990,352
Peak GPU memory      : 231.28 MiB
Elapsed 

Image shape          : [2, 3, 224, 224]
Visual memory shape  : [2, 49, 256]
Decoder input shape  : [2, 13]
Logits shape         : [2, 13, 10080]
Cross-entropy        : 10.71876431
Token accuracy       : 0.00000000 (diagnostic only)
Encoder gradient L2  : 16.34331969
Decoder gradient L2  : 8.54782696
Missing gradients    : 0
Nonfinite gradients  : 0
Memory repeat error  : 0.0
Logits repeat error  : 0.0
End-to-end parameters: 10,874,112
Peak GPU memory      : 429.11 MiB
Elapsed              : 0.89 giây

S3 — TinyViT-5M
------------------------------------------------------------------------
Image shape          : [2, 3, 224, 224]
Visual memory shape  : [2, 49, 256]
Decoder input shape  : [2, 13]
Logits shape         : [2, 13, 10080]
Cross-entropy        : 10.71560478
Token accuracy       : 0.00000000 (diagnostic only)
Encoder gradient L2  : 23.30803107
Decoder gradient L2  : 8.46749166
Missing gradients    : 0
Nonfinite gradients  : 0
Memory repeat error  : 0.0
Logits repeat error  : 0.0

## Cell 9: One-Batch Overfit Gate

### Mục tiêu

Kiểm tra mỗi Student có thực sự học được trước khi bắt đầu full baseline
training.

Một pipeline forward/backward thành công chưa đảm bảo mô hình có thể tối ưu.
Vì vậy, mỗi Student phải ghi nhớ được một batch nhỏ cố định.

### Dữ liệu

- Chỉ sử dụng Train + Restval.
- Chọn 8 caption thuộc 8 ảnh khác nhau.
- Không sử dụng ảnh trùng trong batch.
- Dùng validation transform để tắt augmentation.
- Không đọc caption text thô.
- Không truy cập Test Set.

### Khởi tạo công bằng

Mỗi Student được nạp lại từ:

- Initial pretrained encoder checkpoint riêng.
- Cùng một Shared Transformer Decoder checkpoint.
- Không kế thừa weights đã cập nhật từ Student trước.

### Cấu hình tối ưu sanity check

- Optimizer: AdamW
- Backbone learning rate: 0.0002
- Adapter learning rate: 0.001
- Decoder learning rate: 0.001
- Weight decay: 0
- Gradient clipping: 1.0
- Tối đa: 300 update
- Early stopping chỉ sau ít nhất 50 update

### Hard gates

- Final loss không quá 25% initial loss.
- Loss giảm tối thiểu 75%.
- Final teacher-forced token accuracy tối thiểu 90%.
- Backbone, adapter và decoder đều phải thay đổi.
- Gradient phải hữu hạn.
- Initial checkpoint trên đĩa không được thay đổi.

### Quy tắc artifact

Weights sau one-batch overfit chỉ là sanity check:

- Không lưu overfit checkpoint.
- Không dùng làm baseline.
- Không dùng để tính metric.
- Full training sau này phải nạp lại initial checkpoint.

In [9]:
import csv
import gc
import hashlib
import json
import time
from datetime import datetime, timezone

import torch
import torch.nn.functional as F
from torch.utils.data import Subset


CELL9_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]
OVERFIT_BATCH_SIZE = 8
MAXIMUM_UPDATE_STEPS = 300
MINIMUM_UPDATE_STEPS = 50
LOG_EVERY_STEPS = 25

BACKBONE_LEARNING_RATE = 2.0e-4
ADAPTER_LEARNING_RATE = 1.0e-3
DECODER_LEARNING_RATE = 1.0e-3
WEIGHT_DECAY = 0.0
GRADIENT_CLIP_NORM = 1.0

EARLY_STOP_TOKEN_ACCURACY = 0.98
EARLY_STOP_LOSS_RATIO = 0.10
PASS_MINIMUM_TOKEN_ACCURACY = 0.90
PASS_MAXIMUM_LOSS_RATIO = 0.25

EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_VISUAL_TOKEN_COUNT = 49
EXPECTED_VISUAL_MEMORY_DIMENSION = 256
EXPECTED_MAXIMUM_DECODER_STEPS = 31

required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "PHASE2_AUDIT_DIR",
    "PHASE2_LOG_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_VALIDATION_TRANSFORMS",
    "PHASE2_TRAIN_CAPTION_INDEX",
    "PHASE2_TRAIN_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_DATASET_CLASS",
    "PHASE2_DATALOADER_FACTORY",
    "PHASE2_SHARED_DECODER_CONFIG",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "student_audits",
    "shared_decoder_checkpoint_sha256",
    "build_adapter_from_probe",
    "build_and_load_encoder",
    "build_and_load_shared_decoder",
    "enable_training_graph_with_frozen_batchnorm",
    "configure_strict_deterministic_attention",
    "named_gradient_audit",
    "token_accuracy",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–8: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 8."
)

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS
assert PHASE2_SHARED_DECODER_CONFIG[
    "vocabulary_size"
] == EXPECTED_VOCABULARY_SIZE
assert PHASE2_SHARED_DECODER_CONFIG[
    "maximum_decoder_steps"
] == EXPECTED_MAXIMUM_DECODER_STEPS
assert not (
    set(PHASE2_TRAIN_IMAGE_PATH_BY_ID)
    & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def select_first_distinct_image_indices(
    caption_index,
    required_count,
):
    selected_indices = []
    selected_image_ids = set()

    for row_index in range(len(caption_index)):
        record = caption_index.record(row_index)
        image_id = int(record["image_id"])

        if image_id in selected_image_ids:
            continue

        assert image_id not in (
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        )
        selected_indices.append(row_index)
        selected_image_ids.add(image_id)

        if len(selected_indices) == required_count:
            break

    assert len(selected_indices) == required_count
    assert len(selected_image_ids) == required_count

    return selected_indices


def build_fixed_overfit_batch(
    student_id,
    selected_indices,
):
    """
    Dùng Training corpus nhưng validation transform để loại augmentation.
    """

    deterministic_train_dataset = (
        PHASE2_DATASET_CLASS(
            caption_index=PHASE2_TRAIN_CAPTION_INDEX,
            image_path_by_id=(
                PHASE2_TRAIN_IMAGE_PATH_BY_ID
            ),
            image_transform=(
                PHASE2_VALIDATION_TRANSFORMS[
                    student_id
                ]
            ),
            held_out_test_ids=(
                PHASE2_HELD_OUT_TEST_IMAGE_IDS
            ),
        )
    )
    fixed_subset = Subset(
        deterministic_train_dataset,
        selected_indices,
    )
    fixed_loader = PHASE2_DATALOADER_FACTORY(
        dataset=fixed_subset,
        batch_size=OVERFIT_BATCH_SIZE,
        shuffle=False,
        seed=PHASE2_SEED,
        num_workers=0,
    )
    batch = next(iter(fixed_loader))

    image_ids = batch["image_ids"].tolist()

    assert len(image_ids) == OVERFIT_BATCH_SIZE
    assert len(set(image_ids)) == OVERFIT_BATCH_SIZE
    assert not (
        set(image_ids)
        & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
    )
    assert set(batch["karpathy_splits"]).issubset(
        {"train", "restval"}
    )

    return batch


def move_batch_to_device(batch):
    return {
        "images": batch["images"].to(
            device,
            non_blocking=True,
        ),
        "decoder_input_ids": batch[
            "decoder_input_ids"
        ].to(
            device,
            non_blocking=True,
        ),
        "decoder_attention_mask": batch[
            "decoder_attention_mask"
        ].to(
            device,
            non_blocking=True,
        ),
        "labels": batch["labels"].to(
            device,
            non_blocking=True,
        ),
    }


def forward_pipeline(encoder, decoder, batch):
    visual_memory = encoder(batch["images"])

    assert list(visual_memory.shape) == [
        OVERFIT_BATCH_SIZE,
        EXPECTED_VISUAL_TOKEN_COUNT,
        EXPECTED_VISUAL_MEMORY_DIMENSION,
    ]
    assert torch.isfinite(visual_memory).all()

    memory_attention_mask = torch.ones(
        visual_memory.shape[:2],
        dtype=torch.bool,
        device=visual_memory.device,
    )
    logits = decoder(
        decoder_input_ids=batch[
            "decoder_input_ids"
        ],
        visual_memory=visual_memory,
        decoder_attention_mask=batch[
            "decoder_attention_mask"
        ],
        memory_attention_mask=(
            memory_attention_mask
        ),
    )

    assert list(logits.shape) == [
        OVERFIT_BATCH_SIZE,
        batch["decoder_input_ids"].shape[1],
        EXPECTED_VOCABULARY_SIZE,
    ]
    assert torch.isfinite(logits).all()

    loss = F.cross_entropy(
        logits.reshape(
            -1,
            EXPECTED_VOCABULARY_SIZE,
        ).float(),
        batch["labels"].reshape(-1),
        ignore_index=PHASE2_SHARED_DECODER_CONFIG[
            "pad_id"
        ],
    )
    assert torch.isfinite(loss)

    accuracy = token_accuracy(
        logits=logits.detach(),
        labels=batch["labels"],
        pad_id=PHASE2_SHARED_DECODER_CONFIG[
            "pad_id"
        ],
    )

    return {
        "visual_memory": visual_memory,
        "logits": logits,
        "loss": loss,
        "accuracy": accuracy,
    }


def evaluate_fixed_batch(encoder, decoder, batch):
    encoder.eval()
    decoder.eval()

    with torch.inference_mode():
        output = forward_pipeline(
            encoder,
            decoder,
            batch,
        )

    return {
        "loss": float(output["loss"].item()),
        "token_accuracy": float(
            output["accuracy"]["accuracy"]
        ),
        "valid_token_count": int(
            output["accuracy"]["valid_token_count"]
        ),
        "correct_token_count": int(
            output["accuracy"]["correct_token_count"]
        ),
    }


def build_optimizer(encoder, decoder):
    backbone_parameters = [
        parameter
        for parameter in encoder.backbone.parameters()
        if parameter.requires_grad
    ]
    adapter_parameters = [
        parameter
        for module in [
            encoder.projection,
            encoder.output_norm,
        ]
        for parameter in module.parameters()
        if parameter.requires_grad
    ]
    adapter_parameters.append(
        encoder.visual_position_embedding
    )
    decoder_parameters = [
        parameter
        for parameter in decoder.parameters()
        if parameter.requires_grad
    ]

    parameter_groups = [
        {
            "name": "backbone",
            "params": backbone_parameters,
            "lr": BACKBONE_LEARNING_RATE,
        },
        {
            "name": "adapter",
            "params": adapter_parameters,
            "lr": ADAPTER_LEARNING_RATE,
        },
        {
            "name": "decoder",
            "params": decoder_parameters,
            "lr": DECODER_LEARNING_RATE,
        },
    ]

    all_parameters = (
        backbone_parameters
        + adapter_parameters
        + decoder_parameters
    )
    parameter_ids = [
        id(parameter)
        for parameter in all_parameters
    ]

    assert len(parameter_ids) == len(
        set(parameter_ids)
    ), "Optimizer parameter groups bị trùng."

    expected_trainable_ids = {
        id(parameter)
        for parameter in list(encoder.parameters())
        + list(decoder.parameters())
        if parameter.requires_grad
    }
    assert set(parameter_ids) == expected_trainable_ids

    optimizer = torch.optim.AdamW(
        parameter_groups,
        betas=(0.9, 0.999),
        eps=1.0e-8,
        weight_decay=WEIGHT_DECAY,
    )

    return optimizer, {
        "backbone_parameter_tensors": len(
            backbone_parameters
        ),
        "adapter_parameter_tensors": len(
            adapter_parameters
        ),
        "decoder_parameter_tensors": len(
            decoder_parameters
        ),
        "total_parameter_tensors": len(
            all_parameters
        ),
    }


def snapshot_update_probes(encoder, decoder):
    return {
        "backbone": next(
            parameter
            for parameter in encoder.backbone.parameters()
            if parameter.requires_grad
        ).detach().clone(),
        "adapter": (
            encoder.projection.weight.detach().clone()
        ),
        "decoder": (
            decoder.token_embedding.weight.detach().clone()
        ),
    }


def update_probe_errors(
    encoder,
    decoder,
    initial_probes,
):
    current_probes = {
        "backbone": next(
            parameter
            for parameter in encoder.backbone.parameters()
            if parameter.requires_grad
        ).detach(),
        "adapter": encoder.projection.weight.detach(),
        "decoder": (
            decoder.token_embedding.weight.detach()
        ),
    }

    return {
        name: float(
            (
                current_probes[name]
                - initial_probes[name]
            ).abs().max().item()
        )
        for name in initial_probes
    }


def write_curve_csv(path, rows):
    fieldnames = [
        "student_id",
        "step",
        "loss",
        "token_accuracy",
        "valid_token_count",
        "correct_token_count",
        "gradient_norm_before_clip",
        "elapsed_seconds",
    ]

    with open(
        path,
        "w",
        encoding="utf-8",
        newline="",
    ) as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(rows)


deterministic_backend = (
    configure_strict_deterministic_attention()
)

selected_caption_indices = (
    select_first_distinct_image_indices(
        caption_index=PHASE2_TRAIN_CAPTION_INDEX,
        required_count=OVERFIT_BATCH_SIZE,
    )
)

PHASE2_OVERFIT_LOG_DIR = (
    PHASE2_LOG_DIR / "one_batch_overfit"
)
PHASE2_OVERFIT_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

decoder_checkpoint_sha256_before = sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
)
assert decoder_checkpoint_sha256_before == (
    shared_decoder_checkpoint_sha256
)

encoder_checkpoint_sha256_before = {
    student_id: sha256_file(
        PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
            student_id
        ]
    )
    for student_id in EXPECTED_STUDENT_IDS
}
for student_id in EXPECTED_STUDENT_IDS:
    assert (
        encoder_checkpoint_sha256_before[
            student_id
        ]
        == student_audits[student_id][
            "initial_checkpoint_sha256"
        ]
    )


print("=" * 72)
print("ONE-BATCH OVERFIT GATE")
print("=" * 72)
print()
print("Corpus split             : train + restval")
print("Distinct images          :", OVERFIT_BATCH_SIZE)
print("Augmentation             : disabled")
print("Maximum update steps     :", MAXIMUM_UPDATE_STEPS)
print("Minimum update steps     :", MINIMUM_UPDATE_STEPS)
print("Backbone learning rate   :", BACKBONE_LEARNING_RATE)
print("Adapter learning rate    :", ADAPTER_LEARNING_RATE)
print("Decoder learning rate    :", DECODER_LEARNING_RATE)
print("Weight decay             :", WEIGHT_DECAY)
print("Gradient clip norm       :", GRADIENT_CLIP_NORM)
print(
    "Pass token accuracy     : >=",
    PASS_MINIMUM_TOKEN_ACCURACY,
)
print(
    "Pass maximum loss ratio : <=",
    PASS_MAXIMUM_LOSS_RATIO,
)
print("Strict deterministic     : True")
print("Test data read           : False")


student_overfit_results = {}

for student_index, student_id in enumerate(
    EXPECTED_STUDENT_IDS,
    start=1,
):
    student_started_at = time.perf_counter()
    student_spec = PHASE2_STUDENT_SPECS[student_id]

    print()
    print(
        f"{student_id} — {student_spec['student_name']}"
    )
    print("-" * 72)

    fixed_batch_cpu = build_fixed_overfit_batch(
        student_id=student_id,
        selected_indices=selected_caption_indices,
    )
    fixed_batch = move_batch_to_device(
        fixed_batch_cpu
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    (
        encoder,
        _,
        encoder_checkpoint_path,
        encoder_checkpoint_load_mode,
    ) = build_and_load_encoder(
        student_id,
        fixed_batch["images"],
    )
    decoder, decoder_checkpoint_load_mode = (
        build_and_load_shared_decoder()
    )

    initial_evaluation = evaluate_fixed_batch(
        encoder,
        decoder,
        fixed_batch,
    )
    initial_loss_is_finite = bool(
        torch.isfinite(
            torch.tensor(initial_evaluation["loss"])
        ).item()
    )
    assert initial_loss_is_finite

    initial_update_probes = snapshot_update_probes(
        encoder,
        decoder,
    )

    frozen_batchnorm_modules = (
        enable_training_graph_with_frozen_batchnorm(
            encoder
        )
    )
    decoder.train()

    optimizer, optimizer_inventory = (
        build_optimizer(
            encoder,
            decoder,
        )
    )
    all_trainable_parameters = [
        parameter
        for group in optimizer.param_groups
        for parameter in group["params"]
    ]

    curve_rows = [
        {
            "student_id": student_id,
            "step": 0,
            "loss": initial_evaluation["loss"],
            "token_accuracy": initial_evaluation[
                "token_accuracy"
            ],
            "valid_token_count": initial_evaluation[
                "valid_token_count"
            ],
            "correct_token_count": (
                initial_evaluation[
                    "correct_token_count"
                ]
            ),
            "gradient_norm_before_clip": "",
            "elapsed_seconds": 0.0,
        }
    ]

    first_step_gradient_audit = None
    stopped_early = False
    completed_steps = 0

    torch.manual_seed(
        PHASE2_SEED + 900 + student_index
    )
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            PHASE2_SEED + 900 + student_index
        )

    for step in range(1, MAXIMUM_UPDATE_STEPS + 1):
        encoder.train()
        for module in encoder.modules():
            if isinstance(
                module,
                torch.nn.modules.batchnorm._BatchNorm,
            ):
                module.eval()
        decoder.train()

        optimizer.zero_grad(set_to_none=True)
        train_output = forward_pipeline(
            encoder,
            decoder,
            fixed_batch,
        )
        train_output["loss"].backward()

        if step == 1:
            first_encoder_gradients = (
                named_gradient_audit(
                    encoder,
                    "encoder",
                )
            )
            first_decoder_gradients = (
                named_gradient_audit(
                    decoder,
                    "decoder",
                )
            )
            first_step_gradient_audit = {
                "encoder": first_encoder_gradients,
                "decoder": first_decoder_gradients,
            }
            assert not (
                first_encoder_gradients[
                    "missing_gradient_names"
                ]
                + first_decoder_gradients[
                    "missing_gradient_names"
                ]
            )
            assert not (
                first_encoder_gradients[
                    "nonfinite_gradient_names"
                ]
                + first_decoder_gradients[
                    "nonfinite_gradient_names"
                ]
            )

        gradient_norm_before_clip = float(
            torch.nn.utils.clip_grad_norm_(
                all_trainable_parameters,
                max_norm=GRADIENT_CLIP_NORM,
                error_if_nonfinite=True,
            ).item()
        )
        optimizer.step()
        completed_steps = step

        step_loss = float(
            train_output["loss"].item()
        )
        step_accuracy = float(
            train_output["accuracy"]["accuracy"]
        )
        elapsed = (
            time.perf_counter()
            - student_started_at
        )
        curve_rows.append(
            {
                "student_id": student_id,
                "step": step,
                "loss": step_loss,
                "token_accuracy": step_accuracy,
                "valid_token_count": train_output[
                    "accuracy"
                ]["valid_token_count"],
                "correct_token_count": train_output[
                    "accuracy"
                ]["correct_token_count"],
                "gradient_norm_before_clip": (
                    gradient_norm_before_clip
                ),
                "elapsed_seconds": elapsed,
            }
        )

        if (
            step == 1
            or step % LOG_EVERY_STEPS == 0
            or step == MAXIMUM_UPDATE_STEPS
        ):
            print(
                f"step={step:03d} | "
                f"loss={step_loss:.6f} | "
                f"token_acc={step_accuracy:.4f} | "
                f"grad_norm={gradient_norm_before_clip:.4f}"
            )

        current_loss_ratio = (
            step_loss
            / initial_evaluation["loss"]
        )
        should_stop_early = (
            step >= MINIMUM_UPDATE_STEPS
            and step_accuracy
            >= EARLY_STOP_TOKEN_ACCURACY
            and current_loss_ratio
            <= EARLY_STOP_LOSS_RATIO
        )

        del train_output

        if should_stop_early:
            stopped_early = True
            break

    final_evaluation = evaluate_fixed_batch(
        encoder,
        decoder,
        fixed_batch,
    )
    final_loss_ratio = (
        final_evaluation["loss"]
        / initial_evaluation["loss"]
    )
    loss_reduction_fraction = (
        1.0 - final_loss_ratio
    )
    update_errors = update_probe_errors(
        encoder,
        decoder,
        initial_update_probes,
    )

    assert final_loss_ratio <= (
        PASS_MAXIMUM_LOSS_RATIO
    ), (
        f"{student_id}: loss ratio={final_loss_ratio:.6f} "
        f"> {PASS_MAXIMUM_LOSS_RATIO}."
    )
    assert final_evaluation[
        "token_accuracy"
    ] >= PASS_MINIMUM_TOKEN_ACCURACY, (
        f"{student_id}: final token accuracy="
        f"{final_evaluation['token_accuracy']:.6f} "
        f"< {PASS_MINIMUM_TOKEN_ACCURACY}."
    )
    assert all(
        error > 0.0
        for error in update_errors.values()
    ), (
        f"{student_id}: có component không cập nhật: "
        f"{update_errors}"
    )

    peak_gpu_memory_bytes = (
        int(torch.cuda.max_memory_allocated())
        if torch.cuda.is_available()
        else 0
    )
    student_elapsed_seconds = (
        time.perf_counter()
        - student_started_at
    )

    curve_path = (
        PHASE2_OVERFIT_LOG_DIR
        / f"{student_id.lower()}_one_batch_overfit_curve.csv"
    )
    write_curve_csv(curve_path, curve_rows)
    curve_sha256 = sha256_file(curve_path)

    student_result = {
        "status": "passed",
        "student_id": student_id,
        "student_name": student_spec["student_name"],
        "model_name": student_spec["model_name"],
        "purpose": "sanity_check_only",
        "allowed_for_full_experiment_metrics": False,
        "batch": {
            "source_split": "train+restval",
            "distinct_images": True,
            "batch_size": OVERFIT_BATCH_SIZE,
            "selected_caption_indices": (
                selected_caption_indices
            ),
            "image_ids": fixed_batch_cpu[
                "image_ids"
            ].tolist(),
            "annotation_ids": fixed_batch_cpu[
                "annotation_ids"
            ].tolist(),
            "karpathy_splits": fixed_batch_cpu[
                "karpathy_splits"
            ],
            "test_overlap": 0,
            "augmentation": "disabled",
            "image_shape": list(
                fixed_batch["images"].shape
            ),
            "decoder_input_shape": list(
                fixed_batch[
                    "decoder_input_ids"
                ].shape
            ),
        },
        "optimization": {
            "optimizer": "AdamW",
            "backbone_learning_rate": (
                BACKBONE_LEARNING_RATE
            ),
            "adapter_learning_rate": (
                ADAPTER_LEARNING_RATE
            ),
            "decoder_learning_rate": (
                DECODER_LEARNING_RATE
            ),
            "weight_decay": WEIGHT_DECAY,
            "gradient_clip_norm": (
                GRADIENT_CLIP_NORM
            ),
            "maximum_steps": MAXIMUM_UPDATE_STEPS,
            "minimum_steps": MINIMUM_UPDATE_STEPS,
            "completed_steps": completed_steps,
            "stopped_early": stopped_early,
            "batchnorm_mode": "eval",
            "other_modules_mode": "train",
            "frozen_batchnorm_module_count": (
                frozen_batchnorm_modules
            ),
            "parameter_inventory": (
                optimizer_inventory
            ),
        },
        "initial_evaluation": initial_evaluation,
        "final_evaluation": final_evaluation,
        "gates": {
            "loss_ratio": final_loss_ratio,
            "maximum_loss_ratio": (
                PASS_MAXIMUM_LOSS_RATIO
            ),
            "loss_reduction_fraction": (
                loss_reduction_fraction
            ),
            "token_accuracy": (
                final_evaluation[
                    "token_accuracy"
                ]
            ),
            "minimum_token_accuracy": (
                PASS_MINIMUM_TOKEN_ACCURACY
            ),
            "loss_gate_passed": True,
            "accuracy_gate_passed": True,
            "all_components_updated": True,
            "component_probe_maximum_updates": (
                update_errors
            ),
            "first_step_gradients": (
                first_step_gradient_audit
            ),
        },
        "initial_checkpoints": {
            "encoder_path": str(
                encoder_checkpoint_path
            ),
            "encoder_sha256": (
                encoder_checkpoint_sha256_before[
                    student_id
                ]
            ),
            "encoder_load_mode": (
                encoder_checkpoint_load_mode
            ),
            "decoder_path": str(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "decoder_sha256": (
                decoder_checkpoint_sha256_before
            ),
            "decoder_load_mode": (
                decoder_checkpoint_load_mode
            ),
            "strict_load": True,
        },
        "curve": {
            "path": str(curve_path),
            "sha256": curve_sha256,
            "rows": len(curve_rows),
        },
        "resources": {
            "peak_gpu_memory_bytes": (
                peak_gpu_memory_bytes
            ),
            "elapsed_seconds": (
                student_elapsed_seconds
            ),
        },
        "data_policy": {
            "caption_text_read": False,
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
            "optimizer_steps_are_sanity_only": True,
            "full_training_started": False,
        },
    }
    student_overfit_results[student_id] = (
        student_result
    )

    print(
        "Initial eval loss     :",
        f"{initial_evaluation['loss']:.8f}",
    )
    print(
        "Final eval loss       :",
        f"{final_evaluation['loss']:.8f}",
    )
    print(
        "Final loss ratio      :",
        f"{final_loss_ratio:.8f}",
    )
    print(
        "Loss reduction        :",
        f"{100.0 * loss_reduction_fraction:.2f}%",
    )
    print(
        "Initial token accuracy:",
        f"{initial_evaluation['token_accuracy']:.6f}",
    )
    print(
        "Final token accuracy  :",
        f"{final_evaluation['token_accuracy']:.6f}",
    )
    print("Completed steps       :", completed_steps)
    print("Stopped early         :", stopped_early)
    print(
        "Component updates     :",
        update_errors,
    )
    print(
        "Peak GPU memory       :",
        f"{peak_gpu_memory_bytes / (1024 ** 2):.2f} MiB",
    )
    print("Curve                  :", curve_path)
    print("Curve SHA-256          :", curve_sha256)
    print(
        "Elapsed                :",
        round(student_elapsed_seconds, 2),
        "giây",
    )

    del optimizer
    del all_trainable_parameters
    del initial_update_probes
    del encoder
    del decoder
    del fixed_batch
    del fixed_batch_cpu

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


decoder_checkpoint_sha256_after = sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
)
encoder_checkpoint_sha256_after = {
    student_id: sha256_file(
        PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
            student_id
        ]
    )
    for student_id in EXPECTED_STUDENT_IDS
}

assert decoder_checkpoint_sha256_after == (
    decoder_checkpoint_sha256_before
)
assert encoder_checkpoint_sha256_after == (
    encoder_checkpoint_sha256_before
)
assert set(student_overfit_results) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    result["status"] == "passed"
    for result in student_overfit_results.values()
)

cell9_elapsed_seconds = (
    time.perf_counter()
    - CELL9_STARTED_AT
)

PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_one_batch_overfit_audit.json"
)

combined_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "purpose": (
        "one-batch learnability gate before full baseline training"
    ),
    "student_ids": EXPECTED_STUDENT_IDS,
    "student_count": len(
        student_overfit_results
    ),
    "students": student_overfit_results,
    "shared_batch_indices": (
        selected_caption_indices
    ),
    "batch_size": OVERFIT_BATCH_SIZE,
    "distinct_images": True,
    "augmentation": "disabled",
    "pass_thresholds": {
        "minimum_token_accuracy": (
            PASS_MINIMUM_TOKEN_ACCURACY
        ),
        "maximum_loss_ratio": (
            PASS_MAXIMUM_LOSS_RATIO
        ),
    },
    "deterministic_backend": (
        deterministic_backend
    ),
    "all_students_loss_gate_passed": True,
    "all_students_accuracy_gate_passed": True,
    "all_components_updated": True,
    "initial_checkpoint_files_unchanged": True,
    "overfit_weights_saved": False,
    "overfit_weights_allowed_for_main_experiment": False,
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "optimizer_steps_are_sanity_only": True,
    "full_training_started": False,
    "elapsed_seconds": cell9_elapsed_seconds,
}

with open(
    PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        combined_audit,
        audit_file,
        ensure_ascii=False,
        indent=2,
    )

overfit_audit_sha256 = sha256_file(
    PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH
)

print()
print("=" * 72)
print("PHASE 2 CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("One-batch overfit gates   : 3 / 3 PASSED")
print("Distinct training images  :", OVERFIT_BATCH_SIZE)
print("Augmentation disabled     : True")
print("Loss reduction gate       : 3 / 3 PASSED")
print("Token accuracy gate       : 3 / 3 PASSED")
print("All components updated    : 3 / 3 PASSED")
print("Gradient finite gate      : 3 / 3 PASSED")
print("Strict deterministic mode : True")
print("Initial checkpoints intact: True")
print("Overfit weights saved     : False")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Optimizer steps performed : sanity check only")
print("Full training started     : False")
print("Audit                     :", PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH)
print("Audit SHA-256             :", overfit_audit_sha256)
print(
    "Elapsed                   :",
    round(cell9_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : lock shared baseline training "
    "and reproducibility contracts"
)


ONE-BATCH OVERFIT GATE

Corpus split             : train + restval
Distinct images          : 8
Augmentation             : disabled
Maximum update steps     : 300
Minimum update steps     : 50
Backbone learning rate   : 0.0002
Adapter learning rate    : 0.001
Decoder learning rate    : 0.001
Weight decay             : 0.0
Gradient clip norm       : 1.0
Pass token accuracy     : >= 0.9
Pass maximum loss ratio : <= 0.25
Strict deterministic     : True
Test data read           : False

S1 — MobileNetV3-Large
------------------------------------------------------------------------
step=001 | loss=10.992104 | token_acc=0.0000 | grad_norm=8.6583
step=025 | loss=0.012090 | token_acc=1.0000 | grad_norm=0.0700
step=050 | loss=0.001570 | token_acc=1.0000 | grad_norm=0.0061
Initial eval loss     : 11.33818150
Final eval loss       : 0.00083527
Final loss ratio      : 0.00007367
Loss reduction        : 99.99%
Initial token accuracy: 0.000000
Final token accuracy  : 1.000000
Completed steps       :

## Cell 10: Lock Baseline Training Contract

### Mục tiêu

Khóa toàn bộ cấu hình trước khi triển khai huấn luyện baseline S1/S2/S3:

- Training data sampling.
- Batch và gradient accumulation.
- Optimizer và scheduler.
- Mixed precision.
- Validation protocol.
- Beam Search.
- Checkpoint selection.
- Early stopping.
- Reproducibility và exact resume.

Cell này chỉ tạo contract và audit, chưa bắt đầu huấn luyện.

### Quy tắc dữ liệu huấn luyện

- Dataset: MS COCO 2014.
- Split: Karpathy Train + Restval.
- 113.287 ảnh huấn luyện.
- Mỗi epoch lấy một caption cho mỗi ảnh.
- Caption được luân phiên deterministic qua các epoch.
- Mỗi ảnh có trọng số như nhau trong một epoch.
- Không sử dụng Test Set.

### Cấu hình huấn luyện

- Physical batch size: 32.
- Gradient accumulation: 2.
- Effective batch size: 64.
- Optimizer: AdamW.
- Backbone LR: 0.0001.
- Adapter LR: 0.0003.
- Decoder LR: 0.0003.
- Weight decay: 0.0001.
- Gradient clipping: 1.0.
- FP16 + GradScaler.
- Warmup 5% rồi cosine decay.
- Tối đa 20 epoch.

### Validation và model selection

- Validation loss: toàn bộ 25.010 captions.
- Generation: toàn bộ 5.000 Validation images.
- Beam size: 3.
- Length penalty: 0.7.
- Metrics: BLEU-1, BLEU-4, METEOR, ROUGE-L, CIDEr-D.
- Primary checkpoint metric: Validation CIDEr-D.
- Early stopping: tối thiểu 5 epoch, patience 4.
- Không dùng Test metrics để chọn model.

### Reproducibility

- Development seed: 42.
- Confirmation seeds: 42, 123, 2026.
- Strict deterministic algorithms.
- Flash và Memory-Efficient SDPA bị tắt.
- Math SDPA được bật.
- Checkpoint phải lưu optimizer, scheduler, scaler, RNG và sampler state.
- Bắt buộc chạy throughput pilot 100 bước trước full training.

In [10]:
import hashlib
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path


CELL10_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]

required_runtime_names = [
    "PHASE2_SEED",
    "PHASE2_ROOT",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_SHARED_DECODER_CONFIG",
    "PHASE2_ENCODER_CONTRACTS",
    "PHASE2_ENCODER_CONTRACT_PATHS",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "SHARED_DECODER_CONTRACT_PATH",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "PHASE2_END_TO_END_SMOKE_AUDIT_PATH",
    "PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH",
    "student_audits",
    "student_overfit_results",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–9: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 9."
)

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS
assert set(PHASE2_ENCODER_CONTRACTS) == set(
    EXPECTED_STUDENT_IDS
)
assert set(student_overfit_results) == set(
    EXPECTED_STUDENT_IDS
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_json(path):
    assert path.is_file(), f"Không tìm thấy JSON: {path}"
    assert path.stat().st_size > 0

    with open(path, "r", encoding="utf-8") as source_file:
        return json.load(source_file)


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as output_file:
        json.dump(
            payload,
            output_file,
            ensure_ascii=False,
            indent=2,
        )


def audit_status_passed(path):
    payload = read_json(path)
    status = str(payload.get("status", "")).lower()
    assert status == "passed", (
        f"Audit chưa PASSED: {path}; status={status!r}"
    )
    return payload


REQUIRED_PRETRAINING_AUDITS = {
    "environment": (
        PHASE2_AUDIT_DIR
        / "phase2_environment_audit.json"
    ),
    "contract_and_parquet": (
        PHASE2_AUDIT_DIR
        / "phase2_contract_and_parquet_audit.json"
    ),
    "preprocessing": (
        PHASE2_AUDIT_DIR
        / "phase2_preprocessing_contract_audit.json"
    ),
    "dataset_and_collator": (
        PHASE2_AUDIT_DIR
        / "phase2_dataset_collator_audit.json"
    ),
    "shared_decoder": (
        PHASE2_AUDIT_DIR
        / "phase2_shared_decoder_audit.json"
    ),
    "encoder_adapters": (
        PHASE2_AUDIT_DIR
        / "phase2_encoder_adapter_contract_audit.json"
    ),
    "end_to_end_smoke": (
        Path(PHASE2_END_TO_END_SMOKE_AUDIT_PATH)
    ),
    "one_batch_overfit": (
        Path(PHASE2_ONE_BATCH_OVERFIT_AUDIT_PATH)
    ),
}

audit_payloads = {
    audit_name: audit_status_passed(audit_path)
    for audit_name, audit_path
    in REQUIRED_PRETRAINING_AUDITS.items()
}
audit_hashes = {
    audit_name: sha256_file(audit_path)
    for audit_name, audit_path
    in REQUIRED_PRETRAINING_AUDITS.items()
}

assert audit_payloads["end_to_end_smoke"][
    "all_checkpoint_loads_strict"
]
assert audit_payloads["end_to_end_smoke"][
    "all_backward_gradients_finite"
]
assert audit_payloads["end_to_end_smoke"][
    "all_trainable_parameters_received_gradient"
]
assert audit_payloads["one_batch_overfit"][
    "all_students_loss_gate_passed"
]
assert audit_payloads["one_batch_overfit"][
    "all_students_accuracy_gate_passed"
]
assert audit_payloads["one_batch_overfit"][
    "initial_checkpoint_files_unchanged"
]
assert audit_payloads["one_batch_overfit"][
    "overfit_weights_saved"
] is False


TRAINING_CONTRACT = {
    "contract_version": "phase2_baseline_training_v2",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked",
    "scope": {
        "phase": "Phase 2 — Student baselines",
        "primary_dataset": "MS COCO 2014 only",
        "split_protocol": "Karpathy",
        "student_ids": EXPECTED_STUDENT_IDS,
        "student_models": {
            student_id: PHASE2_STUDENT_SPECS[
                student_id
            ]["student_name"]
            for student_id in EXPECTED_STUDENT_IDS
        },
        "shared_decoder": (
            "Shared Transformer Decoder"
        ),
        "test_set_status": "sealed",
    },
    "training_data": {
        "membership": ["train", "restval"],
        "unique_images": 113_287,
        "available_captions": 566_747,
        "epoch_sampling_unit": "image",
        "captions_per_image_per_epoch": 1,
        "caption_selection": (
            "deterministic cyclic selection by "
            "(epoch + image-specific offset) modulo caption count"
        ),
        "images_per_epoch": 113_287,
        "samples_per_epoch": 113_287,
        "shuffle_images_each_epoch": True,
        "shuffle_seed_rule": "base_seed + epoch",
        "rationale": (
            "Mỗi ảnh có trọng số bằng nhau trong một epoch; "
            "caption của ảnh được luân phiên qua các epoch, giảm gần "
            "5 lần số microbatch so với duyệt mọi image-caption pair."
        ),
        "augmentation": {
            "enabled": True,
            "geometry": (
                "RandomResizedCrop 224 scale [0.80,1.00]"
            ),
            "horizontal_flip_probability": 0.5,
            "normalization": (
                "locked per pretrained encoder contract"
            ),
            "seeded": True,
        },
    },
    "batching": {
        "physical_batch_size": 32,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 64,
        "drop_last": False,
        "dynamic_right_padding": True,
        "pad_id": 0,
        "label_ignore_index": 0,
        "estimated_microbatches_per_epoch": math.ceil(
            113_287 / 32
        ),
        "estimated_optimizer_steps_per_epoch": math.ceil(
            math.ceil(113_287 / 32) / 2
        ),
        "num_workers": 2,
        "pin_memory": True,
        "persistent_workers": True,
    },
    "optimization": {
        "optimizer": "AdamW",
        "parameter_groups": {
            "backbone": {
                "learning_rate": 1.0e-4,
            },
            "adapter": {
                "learning_rate": 3.0e-4,
            },
            "decoder": {
                "learning_rate": 3.0e-4,
            },
        },
        "betas": [0.9, 0.999],
        "epsilon": 1.0e-8,
        "weight_decay": 1.0e-4,
        "gradient_clip_norm": 1.0,
        "loss": "token Cross-Entropy",
        "label_smoothing": 0.0,
        "scheduler": {
            "name": "cosine decay with linear warmup",
            "warmup_fraction_of_optimizer_steps": 0.05,
            "minimum_learning_rate_factor": 0.05,
            "step_unit": "optimizer_step",
        },
        "mixed_precision": {
            "enabled": True,
            "dtype": "float16",
            "gradient_scaler": True,
            "loss_computed_in_float32": True,
            "initial_scale": 1_024.0,
            "growth_factor": 2.0,
            "backoff_factor": 0.5,
            "growth_interval": 2_000,
            "maximum_overflow_retries_per_run": 10,
            "overflow_policy": (
                "skip nonfinite update, reduce scale, do not advance "
                "optimizer global step or scheduler, then continue"
            ),
        },
    },
    "training_duration": {
        "maximum_epochs": 20,
        "minimum_epochs_before_early_stop": 5,
        "early_stopping": {
            "enabled": True,
            "monitored_metric": "validation_CIDEr_D",
            "mode": "max",
            "patience_metric_evaluations": 4,
            "minimum_delta": 0.001,
        },
        "throughput_pilot": {
            "required_before_full_run": True,
            "optimizer_steps": 100,
            "purpose": (
                "estimate seconds/step, epoch ETA and peak VRAM"
            ),
            "pilot_weights_discarded": True,
            "restart_from_initial_checkpoints_after_pilot": True,
        },
    },
    "model_mode": {
        "encoder_backbone_trainable": True,
        "adapter_trainable": True,
        "decoder_trainable": True,
        "batchnorm_running_statistics": "frozen",
        "batchnorm_affine_parameters": "trainable",
        "visual_memory_shape": ["B", 49, 256],
        "decoder_logits_shape": ["B", "T", 10_080],
        "maximum_decoder_steps": 31,
    },
    "initialization": {
        "model_contracts": {
            "encoders": {
                student_id: {
                    "path": str(
                        PHASE2_ENCODER_CONTRACT_PATHS[
                            student_id
                        ]
                    ),
                    "sha256": sha256_file(
                        PHASE2_ENCODER_CONTRACT_PATHS[
                            student_id
                        ]
                    ),
                }
                for student_id in EXPECTED_STUDENT_IDS
            },
            "shared_decoder": {
                "path": str(
                    SHARED_DECODER_CONTRACT_PATH
                ),
                "sha256": sha256_file(
                    SHARED_DECODER_CONTRACT_PATH
                ),
            },
        },
        "encoder_checkpoints": {
            student_id: {
                "path": str(
                    PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                        student_id
                    ]
                ),
                "sha256": student_audits[
                    student_id
                ]["initial_checkpoint_sha256"],
            }
            for student_id in EXPECTED_STUDENT_IDS
        },
        "shared_decoder_checkpoint": {
            "path": str(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "sha256": sha256_file(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "same_initial_checkpoint_for_all_students": True,
        },
        "strict_state_dict_load": True,
        "one_batch_overfit_weights_reused": False,
    },
}


VALIDATION_AND_SELECTION_POLICY = {
    "policy_version": (
        "phase2_validation_selection_v1"
    ),
    "status": "locked",
    "selection_split": "Karpathy validation",
    "validation_images": 5_000,
    "validation_references": 25_010,
    "teacher_forced_validation": {
        "frequency": "every epoch",
        "corpus": "all 25,010 validation captions",
        "metric": "mean token Cross-Entropy",
        "augmentation": False,
    },
    "generation_validation": {
        "frequency": "every epoch",
        "images": "all 5,000 validation images",
        "one_hypothesis_per_image": True,
        "reference_captions_per_image": "all available",
        "decoding": {
            "algorithm": "beam search",
            "beam_size": 3,
            "length_penalty": 0.7,
            "maximum_decoder_steps": 31,
            "minimum_lexical_tokens_before_eos": 2,
            "forbidden_generation_token_ids": [
                0,
                1,
                3,
            ],
            "eos_id": 2,
        },
        "text_postprocessing": {
            "remove_special_tokens": True,
            "strip_whitespace": True,
            "metric_tokenizer": "COCO PTBTokenizer",
        },
        "metrics": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
        ],
    },
    "checkpoint_selection": {
        "primary_metric": "CIDEr-D",
        "primary_mode": "max",
        "tie_break_order": [
            "BLEU-4 higher",
            "validation Cross-Entropy lower",
            "earlier epoch",
        ],
        "test_metrics_forbidden": True,
        "test_predictions_forbidden": True,
    },
    "checkpoint_retention": {
        "keep_best": True,
        "keep_last": True,
        "keep_every_epoch": False,
        "best_contains": [
            "encoder",
            "adapter",
            "decoder",
            "optimizer",
            "scheduler",
            "gradient_scaler",
            "epoch",
            "global_optimizer_step",
            "best_validation_metrics",
            "random_number_generator_states",
            "contracts_and_checksums",
        ],
    },
}


REPRODUCIBILITY_CONTRACT = {
    "contract_version": (
        "phase2_reproducibility_v1"
    ),
    "status": "locked",
    "development_seed": PHASE2_SEED,
    "confirmation_seeds": [42, 123, 2026],
    "phase2_baseline_rule": (
        "Run all S1/S2/S3 baselines with seed 42. "
        "Use confirmation seeds only after model-selection scope "
        "is frozen; do not inspect Test Set."
    ),
    "determinism": {
        "python_hash_seed": "same as run seed",
        "python_random_seeded": True,
        "numpy_seeded": True,
        "torch_cpu_seeded": True,
        "torch_cuda_seeded": True,
        "cudnn_benchmark": False,
        "cudnn_deterministic": True,
        "deterministic_algorithms": True,
        "deterministic_algorithms_warn_only": False,
        "cublas_workspace_config": ":4096:8",
        "flash_sdp_enabled": False,
        "memory_efficient_sdp_enabled": False,
        "math_sdp_enabled": True,
        "dataloader_generator_seeded": True,
        "worker_seeded": True,
        "epoch_sampler_seeded": True,
    },
    "resume": {
        "exact_resume_required": True,
        "restore": [
            "model state",
            "optimizer state",
            "scheduler state",
            "gradient scaler state",
            "epoch and microbatch position",
            "global optimizer step",
            "Python RNG state",
            "NumPy RNG state",
            "Torch CPU RNG state",
            "Torch CUDA RNG states",
            "sampler epoch and state",
            "best metric and patience counter",
        ],
        "configuration_checksum_must_match": True,
        "dataset_contract_checksum_must_match": True,
    },
    "logging": {
        "training_log_format": "CSV + JSON summary",
        "log_every_optimizer_steps": 100,
        "record": [
            "epoch",
            "microbatch",
            "optimizer step",
            "learning rates by parameter group",
            "train loss",
            "gradient norm",
            "tokens per second",
            "images per second",
            "GPU peak allocated memory",
            "elapsed and ETA",
        ],
        "validation_record": [
            "validation Cross-Entropy",
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
            "generation time",
        ],
    },
    "environment_capture": {
        "required": True,
        "capture_pip_freeze": True,
        "capture_gpu_name": True,
        "capture_cuda_cudnn_versions": True,
        "capture_git_or_notebook_version": True,
        "capture_input_and_contract_sha256": True,
    },
}


def render_validation_policy_markdown(policy):
    decoding = policy["generation_validation"][
        "decoding"
    ]

    return f"""# Phase 2 Validation and Checkpoint-Selection Policy

## Trạng thái

`LOCKED` — mọi thay đổi sau thời điểm này phải tạo contract version mới.

## Validation data

- Karpathy Validation: 5.000 ảnh, 25.010 reference captions.
- Teacher-forced Cross-Entropy: toàn bộ 25.010 captions mỗi epoch.
- Generation metrics: đúng một hypothesis cho mỗi ảnh, đánh giá với toàn bộ
  references của ảnh.
- Không dùng Test Set để chọn epoch, hyperparameter hoặc model.

## Decoding

- Beam size: {decoding["beam_size"]}
- Length penalty: {decoding["length_penalty"]}
- Maximum decoder steps: {decoding["maximum_decoder_steps"]}
- Minimum lexical tokens before EOS:
  {decoding["minimum_lexical_tokens_before_eos"]}
- Forbidden generation IDs: {decoding["forbidden_generation_token_ids"]}
- EOS ID: {decoding["eos_id"]}

## Metrics

- BLEU-1
- BLEU-4
- METEOR
- ROUGE-L
- CIDEr-D

## Checkpoint selection

1. CIDEr-D cao hơn.
2. Nếu hòa: BLEU-4 cao hơn.
3. Nếu tiếp tục hòa: Validation Cross-Entropy thấp hơn.
4. Nếu tiếp tục hòa: chọn epoch sớm hơn.

Chỉ giữ `best` và `last`. Checkpoint phải chứa đầy đủ model, optimizer,
scheduler, scaler, RNG states, sampler state và checksum của contract.
"""


PHASE2_TRAINING_CONTRACT_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_shared_baseline_training_contract.json"
)
PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_validation_and_selection_policy.json"
)
PHASE2_VALIDATION_SELECTION_POLICY_MD_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_validation_and_selection_policy.md"
)
PHASE2_REPRODUCIBILITY_CONTRACT_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_reproducibility_contract.json"
)

write_json(
    PHASE2_TRAINING_CONTRACT_PATH,
    TRAINING_CONTRACT,
)
write_json(
    PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH,
    VALIDATION_AND_SELECTION_POLICY,
)
with open(
    PHASE2_VALIDATION_SELECTION_POLICY_MD_PATH,
    "w",
    encoding="utf-8",
) as policy_file:
    policy_file.write(
        render_validation_policy_markdown(
            VALIDATION_AND_SELECTION_POLICY
        )
    )
write_json(
    PHASE2_REPRODUCIBILITY_CONTRACT_PATH,
    REPRODUCIBILITY_CONTRACT,
)


locked_contract_paths = {
    "training_contract": (
        PHASE2_TRAINING_CONTRACT_PATH
    ),
    "validation_policy_json": (
        PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH
    ),
    "validation_policy_markdown": (
        PHASE2_VALIDATION_SELECTION_POLICY_MD_PATH
    ),
    "reproducibility_contract": (
        PHASE2_REPRODUCIBILITY_CONTRACT_PATH
    ),
}
locked_contract_hashes = {
    name: sha256_file(path)
    for name, path in locked_contract_paths.items()
}

assert TRAINING_CONTRACT["batching"][
    "effective_batch_size"
] == (
    TRAINING_CONTRACT["batching"][
        "physical_batch_size"
    ]
    * TRAINING_CONTRACT["batching"][
        "gradient_accumulation_steps"
    ]
)
assert TRAINING_CONTRACT["training_data"][
    "samples_per_epoch"
] == 113_287
assert VALIDATION_AND_SELECTION_POLICY[
    "checkpoint_selection"
]["primary_metric"] == "CIDEr-D"
assert VALIDATION_AND_SELECTION_POLICY[
    "checkpoint_selection"
]["test_metrics_forbidden"]
assert REPRODUCIBILITY_CONTRACT["determinism"][
    "deterministic_algorithms_warn_only"
] is False


cell10_elapsed_seconds = (
    time.perf_counter()
    - CELL10_STARTED_AT
)

PHASE2_PRETRAINING_GATE_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_pretraining_gate_audit.json"
)

pretraining_gate_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "required_audits": {
        name: {
            "path": str(
                REQUIRED_PRETRAINING_AUDITS[name]
            ),
            "sha256": audit_hashes[name],
            "status": "passed",
        }
        for name in REQUIRED_PRETRAINING_AUDITS
    },
    "required_audit_count": len(
        REQUIRED_PRETRAINING_AUDITS
    ),
    "locked_contracts": {
        name: {
            "path": str(path),
            "sha256": locked_contract_hashes[name],
        }
        for name, path in locked_contract_paths.items()
    },
    "student_ids": EXPECTED_STUDENT_IDS,
    "one_batch_overfit_passed": True,
    "end_to_end_smoke_passed": True,
    "training_contract_locked": True,
    "validation_policy_locked": True,
    "reproducibility_contract_locked": True,
    "caption_text_read": False,
    "image_data_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "training_started": False,
    "elapsed_seconds": cell10_elapsed_seconds,
}
write_json(
    PHASE2_PRETRAINING_GATE_AUDIT_PATH,
    pretraining_gate_audit,
)
pretraining_gate_audit_sha256 = sha256_file(
    PHASE2_PRETRAINING_GATE_AUDIT_PATH
)


PHASE2_BASELINE_TRAINING_READY_PATH = (
    PHASE2_ROOT
    / "PHASE2_BASELINE_TRAINING_READY.json"
)

ready_marker = {
    "status": "ready",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "phase": "Phase 2 — Student baselines",
    "authorized_next_action": (
        "build deterministic sampler/training engine and run "
        "100-step throughput pilot; full training remains gated"
    ),
    "student_ids": EXPECTED_STUDENT_IDS,
    "training_contract": {
        "path": str(
            PHASE2_TRAINING_CONTRACT_PATH
        ),
        "sha256": locked_contract_hashes[
            "training_contract"
        ],
    },
    "validation_policy": {
        "path": str(
            PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH
        ),
        "sha256": locked_contract_hashes[
            "validation_policy_json"
        ],
    },
    "reproducibility_contract": {
        "path": str(
            PHASE2_REPRODUCIBILITY_CONTRACT_PATH
        ),
        "sha256": locked_contract_hashes[
            "reproducibility_contract"
        ],
    },
    "pretraining_gate_audit": {
        "path": str(
            PHASE2_PRETRAINING_GATE_AUDIT_PATH
        ),
        "sha256": (
            pretraining_gate_audit_sha256
        ),
    },
    "test_set_status": "sealed",
    "test_set_used": False,
    "training_started": False,
}
write_json(
    PHASE2_BASELINE_TRAINING_READY_PATH,
    ready_marker,
)
ready_marker_sha256 = sha256_file(
    PHASE2_BASELINE_TRAINING_READY_PATH
)


print("=" * 72)
print("LOCKED BASELINE TRAINING CONTRACT")
print("=" * 72)
print()
print("Dataset                   : MS COCO 2014 only")
print("Split protocol            : Karpathy")
print("Training images / epoch   : 113287")
print("Captions sampled / image  : 1 per epoch, cyclic")
print("Physical / effective batch: 32 / 64")
print("Gradient accumulation     : 2")
print("Maximum epochs            : 20")
print("Optimizer                 : AdamW")
print("LR backbone               : 0.0001")
print("LR adapter / decoder      : 0.0003 / 0.0003")
print("Scheduler                 : warmup 5% + cosine")
print("Mixed precision           : FP16 + GradScaler")
print("GradScaler initial scale  : 1024")
print("FP16 overflow retry limit : 10")
print("Gradient clip             : 1.0")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("Generation                : Beam-3, length penalty 0.7")
print("Primary selection metric  : CIDEr-D")
print("Early stopping            : patience 4, min epoch 5")
print("Development seed          : 42")
print("Confirmation seeds        : 42 / 123 / 2026")
print("Test Set status           : SEALED")

print()
print("=" * 72)
print("PHASE 2 CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Required audits           : 8 / 8 PASSED")
print("One-batch overfit         : 3 / 3 PASSED")
print("Training contract         : LOCKED")
print("Validation policy         : LOCKED")
print("Checkpoint selection      : CIDEr-D on Validation")
print("Reproducibility contract  : LOCKED")
print("Strict deterministic mode : REQUIRED")
print("100-step pilot required   : True")
print("Caption text read         : False")
print("Image data read           : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Training contract         :", PHASE2_TRAINING_CONTRACT_PATH)
print(
    "Training SHA-256          :",
    locked_contract_hashes["training_contract"],
)
print(
    "Validation policy        :",
    PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH,
)
print(
    "Validation SHA-256        :",
    locked_contract_hashes[
        "validation_policy_json"
    ],
)
print(
    "Reproducibility contract :",
    PHASE2_REPRODUCIBILITY_CONTRACT_PATH,
)
print(
    "Reproducibility SHA-256  :",
    locked_contract_hashes[
        "reproducibility_contract"
    ],
)
print("Gate audit                :", PHASE2_PRETRAINING_GATE_AUDIT_PATH)
print("Gate audit SHA-256        :", pretraining_gate_audit_sha256)
print("READY marker              :", PHASE2_BASELINE_TRAINING_READY_PATH)
print("READY SHA-256             :", ready_marker_sha256)
print(
    "Elapsed                   :",
    round(cell10_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : implement deterministic image-balanced "
    "sampler and run 100-step throughput pilot"
)

LOCKED BASELINE TRAINING CONTRACT

Dataset                   : MS COCO 2014 only
Split protocol            : Karpathy
Training images / epoch   : 113287
Captions sampled / image  : 1 per epoch, cyclic
Physical / effective batch: 32 / 64
Gradient accumulation     : 2
Maximum epochs            : 20
Optimizer                 : AdamW
LR backbone               : 0.0001
LR adapter / decoder      : 0.0003 / 0.0003
Scheduler                 : warmup 5% + cosine
Mixed precision           : FP16 + GradScaler
GradScaler initial scale  : 1024
FP16 overflow retry limit : 10
Gradient clip             : 1.0
Validation images         : 5000
Validation references     : 25010
Generation                : Beam-3, length penalty 0.7
Primary selection metric  : CIDEr-D
Early stopping            : patience 4, min epoch 5
Development seed          : 42
Confirmation seeds        : 42 / 123 / 2026
Test Set status           : SEALED

PHASE 2 CELL 10 STATUS
Status                    : PASSED
Required audits      

## Cell 11: Image-Balanced Sampler and Throughput Pilot

### Mục tiêu

1. Xây dựng deterministic image-balanced caption sampler.
2. Kiểm tra caption cycling giữa các epoch.
3. Cài training engine theo contract đã khóa.
4. Chạy pilot 100 optimizer steps cho S1, S2 và S3.
5. Đo throughput, VRAM và thời gian huấn luyện dự kiến.
6. Hủy toàn bộ weights pilot sau khi đo.

### Image-balanced sampler

Mỗi epoch:

- Sử dụng đủ 113.287 ảnh Train + Restval.
- Mỗi ảnh xuất hiện đúng một lần.
- Chọn một caption cho mỗi ảnh.
- Caption được luân phiên theo epoch.
- Shuffle seed bằng `base_seed + epoch`.
- Không sử dụng Python hash ngẫu nhiên.
- Không có Test image.

### Training pilot

Cho mỗi Student:

- Physical batch: 32.
- Gradient accumulation: 2.
- Effective batch: 64.
- 100 optimizer steps.
- 200 microbatches.
- 6.400 ảnh được xử lý.
- FP16 + GradScaler.
- AdamW và warmup/cosine scheduler đã khóa.
- BatchNorm running statistics được giữ cố định.
- Strict deterministic Math SDPA.

### Đo hiệu năng

- Seconds per optimizer step.
- Images per second.
- Tokens per second.
- Peak GPU memory.
- GPU memory utilization.
- ETA một epoch.
- ETA tối đa 20 epoch.

Mười optimizer steps đầu được xem là warm-up và không dùng tính throughput.

### Hard gates

- Sampler lặp lại chính xác với cùng seed/epoch.
- Một ảnh chỉ xuất hiện một lần mỗi epoch.
- Caption phải thay đổi giữa epoch 0 và epoch 1.
- Loss và gradient phải hữu hạn.
- Peak VRAM dưới 90% Tesla T4.
- Initial checkpoint SHA-256 không thay đổi.
- Không lưu hoặc tái sử dụng pilot weights.

In [11]:
import csv
import gc
import hashlib
import json
import math
import random
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Sampler


CELL11_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]
PILOT_EPOCH = 0
PILOT_OPTIMIZER_STEPS = 100
TIMING_WARMUP_OPTIMIZER_STEPS = 10
GRADSCALER_INITIAL_SCALE = 1_024.0
GRADSCALER_GROWTH_FACTOR = 2.0
GRADSCALER_BACKOFF_FACTOR = 0.5
GRADSCALER_GROWTH_INTERVAL = 2_000
MAXIMUM_OVERFLOW_RETRIES = 10

required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "gpu_memory_bytes",
    "PHASE2_ROOT",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_LOG_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_TRAIN_CAPTION_INDEX",
    "PHASE2_TRAIN_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_TRAIN_TRANSFORMS",
    "PHASE2_DATASET_CLASS",
    "PHASE2_COLLATOR",
    "seed_dataloader_worker",
    "TRAINING_CONTRACT",
    "REPRODUCIBILITY_CONTRACT",
    "PHASE2_TRAINING_CONTRACT_PATH",
    "PHASE2_REPRODUCIBILITY_CONTRACT_PATH",
    "PHASE2_BASELINE_TRAINING_READY_PATH",
    "read_json",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "student_audits",
    "shared_decoder_checkpoint_sha256",
    "build_and_load_encoder",
    "build_and_load_shared_decoder",
    "enable_training_graph_with_frozen_batchnorm",
    "configure_strict_deterministic_attention",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–10: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 10."
)

assert list(PHASE2_STUDENT_SPECS) == EXPECTED_STUDENT_IDS
assert TRAINING_CONTRACT["status"] == "locked"
assert read_json(
    PHASE2_BASELINE_TRAINING_READY_PATH
)["status"] == "ready"
assert TRAINING_CONTRACT["training_duration"][
    "throughput_pilot"
]["optimizer_steps"] == PILOT_OPTIMIZER_STEPS
assert TRAINING_CONTRACT["contract_version"] == (
    "phase2_baseline_training_v2"
)
assert TRAINING_CONTRACT["optimization"][
    "mixed_precision"
]["initial_scale"] == GRADSCALER_INITIAL_SCALE
assert TRAINING_CONTRACT["optimization"][
    "mixed_precision"
]["maximum_overflow_retries_per_run"] == (
    MAXIMUM_OVERFLOW_RETRIES
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as output_file:
        json.dump(
            payload,
            output_file,
            ensure_ascii=False,
            indent=2,
        )


class DeterministicImageBalancedCaptionSampler(Sampler):
    """
    Mỗi epoch chọn đúng một caption cho mỗi image.

    Caption row:
      rows[(stable_image_offset + epoch) % caption_count]

    Sau đó shuffle các image bằng generator seed = base_seed + epoch.
    """

    def __init__(
        self,
        caption_index,
        base_seed,
        epoch=0,
    ):
        self.caption_index = caption_index
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

        rows_by_image_id = defaultdict(list)

        for row_index, image_id in enumerate(
            caption_index.image_ids
        ):
            rows_by_image_id[
                int(image_id)
            ].append(int(row_index))

        self.rows_by_image_id = {
            image_id: tuple(row_indices)
            for image_id, row_indices
            in rows_by_image_id.items()
        }
        self.sorted_image_ids = tuple(
            sorted(self.rows_by_image_id)
        )

        assert len(self.sorted_image_ids) == 113_287
        assert not (
            set(self.sorted_image_ids)
            & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
        )

    def __len__(self):
        return len(self.sorted_image_ids)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def stable_caption_offset(
        self,
        image_id,
        caption_count,
    ):
        # Không dùng Python hash() vì hash salt thay đổi giữa process.
        mixed = (
            int(image_id) * 1_000_003
            + self.base_seed * 97_409
        )
        return mixed % caption_count

    def selected_rows_before_shuffle(self, epoch=None):
        selected_epoch = (
            self.epoch
            if epoch is None
            else int(epoch)
        )
        selected_rows = []

        for image_id in self.sorted_image_ids:
            image_rows = self.rows_by_image_id[
                image_id
            ]
            offset = self.stable_caption_offset(
                image_id=image_id,
                caption_count=len(image_rows),
            )
            selected_position = (
                offset + selected_epoch
            ) % len(image_rows)
            selected_rows.append(
                image_rows[selected_position]
            )

        return selected_rows

    def indices_for_epoch(self, epoch=None):
        selected_epoch = (
            self.epoch
            if epoch is None
            else int(epoch)
        )
        selected_rows = (
            self.selected_rows_before_shuffle(
                selected_epoch
            )
        )
        generator = torch.Generator()
        generator.manual_seed(
            self.base_seed + selected_epoch
        )
        permutation = torch.randperm(
            len(selected_rows),
            generator=generator,
        ).tolist()

        return [
            selected_rows[position]
            for position in permutation
        ]

    def __iter__(self):
        return iter(
            self.indices_for_epoch(self.epoch)
        )

    def state_dict(self):
        return {
            "base_seed": self.base_seed,
            "epoch": self.epoch,
            "image_count": len(
                self.sorted_image_ids
            ),
        }

    def load_state_dict(self, state):
        assert int(state["base_seed"]) == (
            self.base_seed
        )
        assert int(state["image_count"]) == len(
            self.sorted_image_ids
        )
        self.epoch = int(state["epoch"])


def audit_sampler(sampler):
    epoch0_first = sampler.indices_for_epoch(0)
    epoch0_second = sampler.indices_for_epoch(0)
    epoch1 = sampler.indices_for_epoch(1)

    assert epoch0_first == epoch0_second
    assert len(epoch0_first) == 113_287
    assert len(set(epoch0_first)) == 113_287
    assert len(epoch1) == 113_287

    epoch0_image_ids = [
        int(
            PHASE2_TRAIN_CAPTION_INDEX.image_ids[
                row_index
            ]
        )
        for row_index in epoch0_first
    ]
    epoch1_image_ids = [
        int(
            PHASE2_TRAIN_CAPTION_INDEX.image_ids[
                row_index
            ]
        )
        for row_index in epoch1
    ]

    assert len(set(epoch0_image_ids)) == 113_287
    assert set(epoch0_image_ids) == set(
        epoch1_image_ids
    )
    assert not (
        set(epoch0_image_ids)
        & set(PHASE2_HELD_OUT_TEST_IMAGE_IDS)
    )

    epoch0_by_image = {
        int(
            PHASE2_TRAIN_CAPTION_INDEX.image_ids[
                row_index
            ]
        ): row_index
        for row_index in epoch0_first
    }
    epoch1_by_image = {
        int(
            PHASE2_TRAIN_CAPTION_INDEX.image_ids[
                row_index
            ]
        ): row_index
        for row_index in epoch1
    }
    changed_caption_count = sum(
        epoch0_by_image[image_id]
        != epoch1_by_image[image_id]
        for image_id in epoch0_by_image
    )
    multi_caption_image_count = sum(
        len(rows) > 1
        for rows in sampler.rows_by_image_id.values()
    )
    assert changed_caption_count == (
        multi_caption_image_count
    )

    caption_count_distribution = dict(
        sorted(
            Counter(
                len(rows)
                for rows in sampler.rows_by_image_id.values()
            ).items()
        )
    )

    return {
        "status": "passed",
        "image_count": len(sampler),
        "rows_per_epoch": len(epoch0_first),
        "unique_images_per_epoch": len(
            set(epoch0_image_ids)
        ),
        "epoch0_repeatable": True,
        "epoch0_epoch1_same_image_membership": True,
        "multi_caption_images": (
            multi_caption_image_count
        ),
        "captions_changed_epoch0_to_epoch1": (
            changed_caption_count
        ),
        "caption_count_per_image_distribution": (
            caption_count_distribution
        ),
        "test_overlap": 0,
        "selection_rule": (
            "(stable_image_offset + epoch) modulo caption_count"
        ),
        "shuffle_seed_rule": "base_seed + epoch",
    }


def build_training_dataloader(
    student_id,
    sampler,
):
    batching = TRAINING_CONTRACT["batching"]
    dataset = PHASE2_DATASET_CLASS(
        caption_index=PHASE2_TRAIN_CAPTION_INDEX,
        image_path_by_id=(
            PHASE2_TRAIN_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_TRAIN_TRANSFORMS[student_id]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )

    worker_generator = torch.Generator()
    worker_generator.manual_seed(
        PHASE2_SEED + sampler.epoch
    )

    loader = DataLoader(
        dataset=dataset,
        batch_size=int(
            batching["physical_batch_size"]
        ),
        sampler=sampler,
        shuffle=False,
        num_workers=int(batching["num_workers"]),
        pin_memory=bool(batching["pin_memory"]),
        drop_last=bool(batching["drop_last"]),
        collate_fn=PHASE2_COLLATOR,
        worker_init_fn=seed_dataloader_worker,
        generator=worker_generator,
        persistent_workers=bool(
            batching["persistent_workers"]
            and int(batching["num_workers"]) > 0
        ),
        prefetch_factor=2,
    )

    assert len(loader) == batching[
        "estimated_microbatches_per_epoch"
    ]

    return loader


def build_baseline_optimizer(encoder, decoder):
    optimization = TRAINING_CONTRACT[
        "optimization"
    ]
    group_contracts = optimization[
        "parameter_groups"
    ]

    backbone_parameters = [
        parameter
        for parameter in encoder.backbone.parameters()
        if parameter.requires_grad
    ]
    adapter_parameters = [
        parameter
        for module in [
            encoder.projection,
            encoder.output_norm,
        ]
        for parameter in module.parameters()
        if parameter.requires_grad
    ]
    adapter_parameters.append(
        encoder.visual_position_embedding
    )
    decoder_parameters = [
        parameter
        for parameter in decoder.parameters()
        if parameter.requires_grad
    ]

    parameter_groups = [
        {
            "name": "backbone",
            "params": backbone_parameters,
            "lr": group_contracts[
                "backbone"
            ]["learning_rate"],
        },
        {
            "name": "adapter",
            "params": adapter_parameters,
            "lr": group_contracts[
                "adapter"
            ]["learning_rate"],
        },
        {
            "name": "decoder",
            "params": decoder_parameters,
            "lr": group_contracts[
                "decoder"
            ]["learning_rate"],
        },
    ]

    all_parameter_ids = [
        id(parameter)
        for group in parameter_groups
        for parameter in group["params"]
    ]
    expected_parameter_ids = {
        id(parameter)
        for parameter in list(encoder.parameters())
        + list(decoder.parameters())
        if parameter.requires_grad
    }

    assert len(all_parameter_ids) == len(
        set(all_parameter_ids)
    )
    assert set(all_parameter_ids) == (
        expected_parameter_ids
    )

    optimizer = torch.optim.AdamW(
        parameter_groups,
        betas=tuple(optimization["betas"]),
        eps=float(optimization["epsilon"]),
        weight_decay=float(
            optimization["weight_decay"]
        ),
    )

    return optimizer


def build_scheduler(optimizer):
    batching = TRAINING_CONTRACT["batching"]
    duration = TRAINING_CONTRACT[
        "training_duration"
    ]
    scheduler_contract = TRAINING_CONTRACT[
        "optimization"
    ]["scheduler"]

    total_optimizer_steps = (
        int(
            batching[
                "estimated_optimizer_steps_per_epoch"
            ]
        )
        * int(duration["maximum_epochs"])
    )
    warmup_steps = max(
        1,
        int(
            round(
                total_optimizer_steps
                * float(
                    scheduler_contract[
                        "warmup_fraction_of_optimizer_steps"
                    ]
                )
            )
        ),
    )
    minimum_factor = float(
        scheduler_contract[
            "minimum_learning_rate_factor"
        ]
    )

    def learning_rate_multiplier(step):
        if step < warmup_steps:
            return max(
                (step + 1) / warmup_steps,
                1.0 / warmup_steps,
            )

        decay_denominator = max(
            1,
            total_optimizer_steps - warmup_steps,
        )
        progress = min(
            1.0,
            (
                step - warmup_steps
            ) / decay_denominator,
        )
        cosine_factor = (
            0.5
            * (1.0 + math.cos(math.pi * progress))
        )
        return (
            minimum_factor
            + (1.0 - minimum_factor)
            * cosine_factor
        )

    scheduler = LambdaLR(
        optimizer,
        lr_lambda=learning_rate_multiplier,
    )

    return scheduler, {
        "total_optimizer_steps": (
            total_optimizer_steps
        ),
        "warmup_optimizer_steps": warmup_steps,
        "minimum_learning_rate_factor": (
            minimum_factor
        ),
    }


def move_training_batch_to_device(batch):
    return {
        "images": batch["images"].to(
            device,
            non_blocking=True,
        ),
        "decoder_input_ids": batch[
            "decoder_input_ids"
        ].to(
            device,
            non_blocking=True,
        ),
        "decoder_attention_mask": batch[
            "decoder_attention_mask"
        ].to(
            device,
            non_blocking=True,
        ),
        "labels": batch["labels"].to(
            device,
            non_blocking=True,
        ),
    }


def pilot_forward_loss(encoder, decoder, batch):
    visual_memory = encoder(batch["images"])
    assert visual_memory.shape[1:] == (
        49,
        256,
    )
    assert torch.isfinite(visual_memory).all()

    memory_attention_mask = torch.ones(
        visual_memory.shape[:2],
        dtype=torch.bool,
        device=visual_memory.device,
    )
    logits = decoder(
        decoder_input_ids=batch[
            "decoder_input_ids"
        ],
        visual_memory=visual_memory,
        decoder_attention_mask=batch[
            "decoder_attention_mask"
        ],
        memory_attention_mask=(
            memory_attention_mask
        ),
    )
    assert logits.shape[-1] == 10_080
    assert torch.isfinite(logits).all()

    loss = F.cross_entropy(
        logits.reshape(-1, 10_080).float(),
        batch["labels"].reshape(-1),
        ignore_index=0,
    )
    assert torch.isfinite(loss)

    valid_tokens = int(
        (batch["labels"] != 0).sum().item()
    )

    return loss, valid_tokens


def write_pilot_csv(path, rows):
    fieldnames = [
        "student_id",
        "optimizer_step",
        "microbatch_count",
        "mean_loss",
        "gradient_norm_before_clip",
        "backbone_lr",
        "adapter_lr",
        "decoder_lr",
        "images_seen",
        "tokens_seen",
        "elapsed_seconds",
        "peak_gpu_memory_mib",
    ]

    with open(
        path,
        "w",
        encoding="utf-8",
        newline="",
    ) as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(rows)


def run_throughput_pilot(
    student_id,
    sampler,
):
    student_started_at = time.perf_counter()
    loader = build_training_dataloader(
        student_id,
        sampler,
    )
    loader_iterator = iter(loader)

    first_batch_cpu = next(loader_iterator)
    first_batch = move_training_batch_to_device(
        first_batch_cpu
    )

    (
        encoder,
        _,
        encoder_checkpoint_path,
        encoder_load_mode,
    ) = build_and_load_encoder(
        student_id,
        first_batch["images"],
    )
    decoder, decoder_load_mode = (
        build_and_load_shared_decoder()
    )

    frozen_batchnorm_count = (
        enable_training_graph_with_frozen_batchnorm(
            encoder
        )
    )
    decoder.train()

    optimizer = build_baseline_optimizer(
        encoder,
        decoder,
    )
    scheduler, scheduler_inventory = (
        build_scheduler(optimizer)
    )

    mixed_precision = TRAINING_CONTRACT[
        "optimization"
    ]["mixed_precision"]
    amp_enabled = bool(
        mixed_precision["enabled"]
        and device.type == "cuda"
    )
    scaler = torch.amp.GradScaler(
        device="cuda",
        init_scale=GRADSCALER_INITIAL_SCALE,
        growth_factor=GRADSCALER_GROWTH_FACTOR,
        backoff_factor=GRADSCALER_BACKOFF_FACTOR,
        growth_interval=GRADSCALER_GROWTH_INTERVAL,
        enabled=amp_enabled,
    )

    accumulation_steps = int(
        TRAINING_CONTRACT["batching"][
            "gradient_accumulation_steps"
        ]
    )
    physical_batch_size = int(
        TRAINING_CONTRACT["batching"][
            "physical_batch_size"
        ]
    )
    expected_microbatches = (
        PILOT_OPTIMIZER_STEPS
        * accumulation_steps
    )
    gradient_clip_norm = float(
        TRAINING_CONTRACT["optimization"][
            "gradient_clip_norm"
        ]
    )

    optimizer.zero_grad(set_to_none=True)

    pilot_rows = []
    optimizer_step = 0
    microbatch_count = 0
    images_seen = 0
    tokens_seen = 0
    accumulated_losses = []
    overflow_retry_count = 0
    overflow_events = []
    timing_measurement_started_at = None
    timing_measurement_start_images = None
    timing_measurement_start_tokens = None

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    torch.manual_seed(
        PHASE2_SEED + 1_100
        + EXPECTED_STUDENT_IDS.index(student_id)
    )
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            PHASE2_SEED + 1_100
            + EXPECTED_STUDENT_IDS.index(student_id)
        )

    while optimizer_step < PILOT_OPTIMIZER_STEPS:
        if microbatch_count == 0:
            batch_cpu = first_batch_cpu
            batch = first_batch
        else:
            batch_cpu = next(loader_iterator)
            batch = move_training_batch_to_device(
                batch_cpu
            )

        microbatch_count += 1
        images_seen += int(
            batch["images"].shape[0]
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            loss, valid_tokens = pilot_forward_loss(
                encoder,
                decoder,
                batch,
            )
            scaled_loss = loss / accumulation_steps

        tokens_seen += valid_tokens
        accumulated_losses.append(
            float(loss.item())
        )
        scaler.scale(scaled_loss).backward()

        del scaled_loss
        del loss

        if (
            microbatch_count % accumulation_steps
            != 0
        ):
            if microbatch_count > 1:
                del batch
                del batch_cpu
            continue

        scaler.unscale_(optimizer)
        all_trainable_parameters = [
            parameter
            for group in optimizer.param_groups
            for parameter in group["params"]
        ]
        gradient_norm = float(
            torch.nn.utils.clip_grad_norm_(
                all_trainable_parameters,
                max_norm=gradient_clip_norm,
                error_if_nonfinite=False,
            ).item()
        )

        gradient_is_finite = math.isfinite(
            gradient_norm
        )

        if not gradient_is_finite:
            if not amp_enabled:
                raise RuntimeError(
                    f"{student_id}: gradient nonfinite khi AMP bị tắt; "
                    "không thể xử lý bằng GradScaler."
                )

            scale_before = float(
                scaler.get_scale()
            )

            # unscale_ đã ghi nhận found_inf. scaler.step sẽ bỏ update,
            # scaler.update sẽ giảm scale; scheduler/global step giữ nguyên.
            scaler.step(optimizer)
            scaler.update()
            scale_after = float(
                scaler.get_scale()
            )
            optimizer.zero_grad(set_to_none=True)

            overflow_retry_count += 1
            overflow_events.append(
                {
                    "after_successful_optimizer_step": (
                        optimizer_step
                    ),
                    "microbatch_count": (
                        microbatch_count
                    ),
                    "scale_before": scale_before,
                    "scale_after": scale_after,
                }
            )
            assert overflow_retry_count <= (
                MAXIMUM_OVERFLOW_RETRIES
            ), (
                f"{student_id}: quá "
                f"{MAXIMUM_OVERFLOW_RETRIES} FP16 overflow retries."
            )
            assert (
                not amp_enabled
                or scale_after < scale_before
            )

            accumulated_losses.clear()

            print(
                "FP16 overflow retry   :",
                overflow_retry_count,
                "| successful_step=",
                optimizer_step,
                "| scale",
                scale_before,
                "->",
                scale_after,
            )

            if microbatch_count > 1:
                del batch
                del batch_cpu
            continue

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        optimizer_step += 1

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        if (
            optimizer_step
            == TIMING_WARMUP_OPTIMIZER_STEPS
        ):
            timing_measurement_started_at = (
                time.perf_counter()
            )
            timing_measurement_start_images = (
                images_seen
            )
            timing_measurement_start_tokens = (
                tokens_seen
            )

        mean_accumulated_loss = float(
            np.mean(accumulated_losses)
        )
        accumulated_losses.clear()
        peak_gpu_memory_bytes = (
            int(torch.cuda.max_memory_allocated())
            if torch.cuda.is_available()
            else 0
        )
        elapsed_seconds = (
            time.perf_counter()
            - student_started_at
        )
        learning_rates = {
            group["name"]: float(group["lr"])
            for group in optimizer.param_groups
        }
        pilot_rows.append(
            {
                "student_id": student_id,
                "optimizer_step": optimizer_step,
                "microbatch_count": (
                    microbatch_count
                ),
                "mean_loss": (
                    mean_accumulated_loss
                ),
                "gradient_norm_before_clip": (
                    gradient_norm
                ),
                "backbone_lr": learning_rates[
                    "backbone"
                ],
                "adapter_lr": learning_rates[
                    "adapter"
                ],
                "decoder_lr": learning_rates[
                    "decoder"
                ],
                "images_seen": images_seen,
                "tokens_seen": tokens_seen,
                "elapsed_seconds": elapsed_seconds,
                "peak_gpu_memory_mib": (
                    peak_gpu_memory_bytes
                    / (1024 ** 2)
                ),
            }
        )

        if (
            optimizer_step == 1
            or optimizer_step % 10 == 0
            or optimizer_step
            == PILOT_OPTIMIZER_STEPS
        ):
            print(
                f"step={optimizer_step:03d} | "
                f"loss={mean_accumulated_loss:.6f} | "
                f"grad={gradient_norm:.4f} | "
                f"lr={learning_rates['decoder']:.8f} | "
                f"peak={peak_gpu_memory_bytes / (1024 ** 2):.1f} MiB"
            )

        if microbatch_count > 1:
            del batch
            del batch_cpu

    assert optimizer_step == PILOT_OPTIMIZER_STEPS
    assert microbatch_count >= expected_microbatches
    assert images_seen >= (
        expected_microbatches
        * physical_batch_size
    )
    assert timing_measurement_started_at is not None

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    timing_seconds = (
        time.perf_counter()
        - timing_measurement_started_at
    )
    timed_optimizer_steps = (
        PILOT_OPTIMIZER_STEPS
        - TIMING_WARMUP_OPTIMIZER_STEPS
    )
    timed_images = (
        images_seen
        - timing_measurement_start_images
    )
    timed_tokens = (
        tokens_seen
        - timing_measurement_start_tokens
    )

    seconds_per_optimizer_step = (
        timing_seconds / timed_optimizer_steps
    )
    images_per_second = (
        timed_images / timing_seconds
    )
    tokens_per_second = (
        timed_tokens / timing_seconds
    )
    optimizer_steps_per_epoch = int(
        TRAINING_CONTRACT["batching"][
            "estimated_optimizer_steps_per_epoch"
        ]
    )
    seconds_per_epoch = (
        seconds_per_optimizer_step
        * optimizer_steps_per_epoch
    )
    maximum_training_seconds = (
        seconds_per_epoch
        * int(
            TRAINING_CONTRACT[
                "training_duration"
            ]["maximum_epochs"]
        )
    )
    peak_gpu_memory_bytes = (
        int(torch.cuda.max_memory_allocated())
        if torch.cuda.is_available()
        else 0
    )
    gpu_memory_utilization = (
        peak_gpu_memory_bytes / gpu_memory_bytes
        if gpu_memory_bytes
        else 0.0
    )

    assert seconds_per_optimizer_step > 0.0
    assert images_per_second > 0.0
    assert tokens_per_second > 0.0
    assert gpu_memory_utilization < 0.90, (
        f"{student_id}: pilot dùng "
        f"{100 * gpu_memory_utilization:.2f}% VRAM, "
        "vượt ngưỡng an toàn 90%."
    )
    assert all(
        math.isfinite(
            float(row["mean_loss"])
        )
        for row in pilot_rows
    )
    assert all(
        math.isfinite(
            float(
                row[
                    "gradient_norm_before_clip"
                ]
            )
        )
        for row in pilot_rows
    )

    pilot_csv_path = (
        PHASE2_PILOT_LOG_DIR
        / f"{student_id.lower()}_throughput_pilot.csv"
    )
    write_pilot_csv(
        pilot_csv_path,
        pilot_rows,
    )
    pilot_csv_sha256 = sha256_file(
        pilot_csv_path
    )

    result = {
        "status": "passed",
        "student_id": student_id,
        "student_name": (
            PHASE2_STUDENT_SPECS[
                student_id
            ]["student_name"]
        ),
        "pilot": {
            "epoch": PILOT_EPOCH,
            "optimizer_steps": optimizer_step,
            "microbatches": microbatch_count,
            "physical_batch_size": (
                physical_batch_size
            ),
            "gradient_accumulation_steps": (
                accumulation_steps
            ),
            "effective_batch_size": (
                physical_batch_size
                * accumulation_steps
            ),
            "images_seen": images_seen,
            "tokens_seen": tokens_seen,
            "minimum_expected_microbatches": (
                expected_microbatches
            ),
            "overflow_retry_count": (
                overflow_retry_count
            ),
            "overflow_events": overflow_events,
            "maximum_overflow_retries": (
                MAXIMUM_OVERFLOW_RETRIES
            ),
            "timing_warmup_optimizer_steps": (
                TIMING_WARMUP_OPTIMIZER_STEPS
            ),
            "timed_optimizer_steps": (
                timed_optimizer_steps
            ),
            "weights_saved": False,
            "weights_discarded": True,
        },
        "mixed_precision": {
            "enabled": amp_enabled,
            "dtype": "float16",
            "gradient_scaler": True,
            "initial_scale": (
                GRADSCALER_INITIAL_SCALE
            ),
            "final_scale": float(
                scaler.get_scale()
            ),
        },
        "scheduler": scheduler_inventory,
        "throughput": {
            "seconds_per_optimizer_step": (
                seconds_per_optimizer_step
            ),
            "images_per_second": (
                images_per_second
            ),
            "tokens_per_second": (
                tokens_per_second
            ),
            "estimated_seconds_per_epoch": (
                seconds_per_epoch
            ),
            "estimated_hours_per_epoch": (
                seconds_per_epoch / 3600.0
            ),
            "estimated_maximum_training_hours": (
                maximum_training_seconds / 3600.0
            ),
        },
        "resources": {
            "peak_gpu_memory_bytes": (
                peak_gpu_memory_bytes
            ),
            "peak_gpu_memory_mib": (
                peak_gpu_memory_bytes
                / (1024 ** 2)
            ),
            "gpu_total_memory_bytes": (
                gpu_memory_bytes
            ),
            "gpu_memory_utilization": (
                gpu_memory_utilization
            ),
            "safe_vram_gate_under_90_percent": True,
        },
        "loss": {
            "first_optimizer_step": float(
                pilot_rows[0]["mean_loss"]
            ),
            "last_optimizer_step": float(
                pilot_rows[-1]["mean_loss"]
            ),
            "all_finite": True,
        },
        "gradient": {
            "all_successful_steps_finite": True,
            "fp16_overflows_handled_by_gradscaler": (
                overflow_retry_count
            ),
            "clip_norm": float(
                TRAINING_CONTRACT[
                    "optimization"
                ]["gradient_clip_norm"]
            ),
        },
        "batchnorm": {
            "running_statistics_frozen": True,
            "frozen_module_count": (
                frozen_batchnorm_count
            ),
        },
        "initial_checkpoints": {
            "encoder_path": str(
                encoder_checkpoint_path
            ),
            "encoder_sha256": (
                student_audits[student_id][
                    "initial_checkpoint_sha256"
                ]
            ),
            "encoder_load_mode": (
                encoder_load_mode
            ),
            "decoder_path": str(
                SHARED_DECODER_INITIAL_CHECKPOINT_PATH
            ),
            "decoder_sha256": (
                shared_decoder_checkpoint_sha256
            ),
            "decoder_load_mode": (
                decoder_load_mode
            ),
        },
        "log": {
            "path": str(pilot_csv_path),
            "sha256": pilot_csv_sha256,
            "rows": len(pilot_rows),
        },
        "data_policy": {
            "source_split": "train+restval",
            "caption_text_read": False,
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
            "full_training_started": False,
        },
    }

    # Pilot state tuyệt đối không được dùng tiếp.
    del scaler
    del scheduler
    del optimizer
    del encoder
    del decoder
    del first_batch
    del first_batch_cpu
    del loader_iterator
    del loader

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


deterministic_backend = (
    configure_strict_deterministic_attention()
)

sampler = DeterministicImageBalancedCaptionSampler(
    caption_index=PHASE2_TRAIN_CAPTION_INDEX,
    base_seed=PHASE2_SEED,
    epoch=PILOT_EPOCH,
)
sampler_audit = audit_sampler(sampler)

PHASE2_SAMPLER_CONTRACT_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_image_balanced_sampler_contract.json"
)
sampler_contract = {
    "contract_version": (
        "phase2_image_balanced_sampler_v1"
    ),
    "status": "locked",
    "class_name": (
        "DeterministicImageBalancedCaptionSampler"
    ),
    "base_seed": PHASE2_SEED,
    "image_count": len(sampler),
    "rows_per_epoch": len(sampler),
    "caption_selection": (
        "(stable_image_offset + epoch) modulo caption_count"
    ),
    "shuffle": True,
    "shuffle_seed": "base_seed + epoch",
    "state_dict": sampler.state_dict(),
    "audit": sampler_audit,
}
write_json(
    PHASE2_SAMPLER_CONTRACT_PATH,
    sampler_contract,
)
sampler_contract_sha256 = sha256_file(
    PHASE2_SAMPLER_CONTRACT_PATH
)

PHASE2_PILOT_LOG_DIR = (
    PHASE2_LOG_DIR / "throughput_pilot"
)
PHASE2_PILOT_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

initial_checkpoint_hashes_before = {
    "decoder": sha256_file(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH
    ),
    **{
        student_id: sha256_file(
            PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                student_id
            ]
        )
        for student_id in EXPECTED_STUDENT_IDS
    },
}
assert initial_checkpoint_hashes_before[
    "decoder"
] == shared_decoder_checkpoint_sha256
for student_id in EXPECTED_STUDENT_IDS:
    assert initial_checkpoint_hashes_before[
        student_id
    ] == student_audits[student_id][
        "initial_checkpoint_sha256"
    ]


print("=" * 72)
print("IMAGE-BALANCED SAMPLER AUDIT")
print("=" * 72)
print()
print("Images per epoch          :", len(sampler))
print("Caption rows per epoch    :", len(sampler))
print("Unique images per epoch   :", sampler_audit["unique_images_per_epoch"])
print("Epoch repeatability       :", sampler_audit["epoch0_repeatable"])
print(
    "Caption changes epoch 0→1:",
    sampler_audit["captions_changed_epoch0_to_epoch1"],
)
print(
    "Captions/image dist.      :",
    sampler_audit[
        "caption_count_per_image_distribution"
    ],
)
print("Test overlap              : 0")

print()
print("=" * 72)
print("100-STEP THROUGHPUT PILOT")
print("=" * 72)
print()
print("Physical batch size       : 32")
print("Gradient accumulation     : 2")
print("Effective batch size      : 64")
print("Optimizer steps / model   :", PILOT_OPTIMIZER_STEPS)
print("Microbatches / model      :", PILOT_OPTIMIZER_STEPS * 2)
print("Images / model            :", PILOT_OPTIMIZER_STEPS * 64)
print("Mixed precision           : FP16 + GradScaler")
print("GradScaler initial scale  :", GRADSCALER_INITIAL_SCALE)
print("Overflow retry limit      :", MAXIMUM_OVERFLOW_RETRIES)
print("Strict deterministic      : True")
print("Pilot weights saved       : False")
print("Test data read            : False")


pilot_results = {}

for student_id in EXPECTED_STUDENT_IDS:
    print()
    print(
        f"{student_id} — "
        f"{PHASE2_STUDENT_SPECS[student_id]['student_name']}"
    )
    print("-" * 72)

    sampler.set_epoch(PILOT_EPOCH)
    pilot_result = run_throughput_pilot(
        student_id=student_id,
        sampler=sampler,
    )
    pilot_results[student_id] = pilot_result

    throughput = pilot_result["throughput"]
    resources = pilot_result["resources"]

    print(
        "Seconds/optimizer step:",
        f"{throughput['seconds_per_optimizer_step']:.4f}",
    )
    print(
        "Images/second         :",
        f"{throughput['images_per_second']:.2f}",
    )
    print(
        "Tokens/second         :",
        f"{throughput['tokens_per_second']:.2f}",
    )
    print(
        "Estimated epoch       :",
        f"{throughput['estimated_hours_per_epoch']:.3f} hours",
    )
    print(
        "Estimated max 20 epoch:",
        f"{throughput['estimated_maximum_training_hours']:.2f} hours",
    )
    print(
        "Peak GPU memory       :",
        f"{resources['peak_gpu_memory_mib']:.2f} MiB",
        f"({100 * resources['gpu_memory_utilization']:.2f}%)",
    )
    print(
        "First / last loss     :",
        f"{pilot_result['loss']['first_optimizer_step']:.6f}",
        "/",
        f"{pilot_result['loss']['last_optimizer_step']:.6f}",
    )
    print(
        "Pilot log             :",
        pilot_result["log"]["path"],
    )


initial_checkpoint_hashes_after = {
    "decoder": sha256_file(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH
    ),
    **{
        student_id: sha256_file(
            PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                student_id
            ]
        )
        for student_id in EXPECTED_STUDENT_IDS
    },
}
assert initial_checkpoint_hashes_after == (
    initial_checkpoint_hashes_before
)
assert set(pilot_results) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    result["status"] == "passed"
    for result in pilot_results.values()
)


cell11_elapsed_seconds = (
    time.perf_counter()
    - CELL11_STARTED_AT
)

PHASE2_THROUGHPUT_PILOT_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_sampler_and_throughput_pilot_audit.json"
)
combined_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sampler": sampler_audit,
    "sampler_contract": {
        "path": str(
            PHASE2_SAMPLER_CONTRACT_PATH
        ),
        "sha256": sampler_contract_sha256,
    },
    "pilot_optimizer_steps_per_student": (
        PILOT_OPTIMIZER_STEPS
    ),
    "student_ids": EXPECTED_STUDENT_IDS,
    "students": pilot_results,
    "all_pilots_passed": True,
    "all_losses_finite": True,
    "all_successful_step_gradients_finite": True,
    "all_fp16_overflows_handled": True,
    "fp16_overflow_retries_total": sum(
        result["pilot"]["overflow_retry_count"]
        for result in pilot_results.values()
    ),
    "all_vram_gates_passed": True,
    "initial_checkpoint_files_unchanged": True,
    "pilot_weights_saved": False,
    "pilot_weights_discarded": True,
    "strict_deterministic_backend": (
        deterministic_backend
    ),
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "full_training_started": False,
    "elapsed_seconds": cell11_elapsed_seconds,
}
write_json(
    PHASE2_THROUGHPUT_PILOT_AUDIT_PATH,
    combined_audit,
)
pilot_audit_sha256 = sha256_file(
    PHASE2_THROUGHPUT_PILOT_AUDIT_PATH
)


PHASE2_TRAINING_ENGINE_PILOT_READY_PATH = (
    PHASE2_ROOT
    / "PHASE2_TRAINING_ENGINE_PILOT_READY.json"
)
pilot_ready_marker = {
    "status": "ready",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_ids": EXPECTED_STUDENT_IDS,
    "sampler_contract": {
        "path": str(
            PHASE2_SAMPLER_CONTRACT_PATH
        ),
        "sha256": sampler_contract_sha256,
    },
    "pilot_audit": {
        "path": str(
            PHASE2_THROUGHPUT_PILOT_AUDIT_PATH
        ),
        "sha256": pilot_audit_sha256,
    },
    "pilot_weights_saved": False,
    "initial_checkpoints_unchanged": True,
    "authorized_next_action": (
        "run exact-resume equivalence audit before full training"
    ),
    "full_training_authorized": False,
    "test_set_status": "sealed",
}
write_json(
    PHASE2_TRAINING_ENGINE_PILOT_READY_PATH,
    pilot_ready_marker,
)
pilot_ready_sha256 = sha256_file(
    PHASE2_TRAINING_ENGINE_PILOT_READY_PATH
)


print()
print("=" * 72)
print("PHASE 2 CELL 11 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Image-balanced sampler    : PASSED")
print("Images per epoch          : 113287")
print("One caption/image/epoch   : True")
print("Epoch caption cycling     : PASSED")
print("Sampler deterministic     : PASSED")
print("Throughput pilots         : 3 / 3 PASSED")
print("Optimizer steps/model     : 100")
print("Physical/effective batch  : 32 / 64")
print("FP16 + GradScaler         : PASSED")
print("Loss finite               : 3 / 3 PASSED")
print("Successful gradients finite: 3 / 3 PASSED")
print(
    "FP16 overflow retries     :",
    sum(
        result["pilot"]["overflow_retry_count"]
        for result in pilot_results.values()
    ),
    "(handled)",
)
print("VRAM safety < 90%         : 3 / 3 PASSED")
print("Initial checkpoints intact: True")
print("Pilot weights saved       : False")
print("Pilot weights discarded   : True")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("Sampler contract          :", PHASE2_SAMPLER_CONTRACT_PATH)
print("Sampler SHA-256           :", sampler_contract_sha256)
print("Pilot audit               :", PHASE2_THROUGHPUT_PILOT_AUDIT_PATH)
print("Pilot audit SHA-256       :", pilot_audit_sha256)
print("READY marker              :", PHASE2_TRAINING_ENGINE_PILOT_READY_PATH)
print("READY SHA-256             :", pilot_ready_sha256)
print(
    "Elapsed                   :",
    round(cell11_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : exact-resume equivalence audit; "
    "full training remains blocked until it passes"
)

IMAGE-BALANCED SAMPLER AUDIT

Images per epoch          : 113287
Caption rows per epoch    : 113287
Unique images per epoch   : 113287
Epoch repeatability       : True
Caption changes epoch 0→1: 113287
Captions/image dist.      : {5: 112979, 6: 304, 7: 4}
Test overlap              : 0

100-STEP THROUGHPUT PILOT

Physical batch size       : 32
Gradient accumulation     : 2
Effective batch size      : 64
Optimizer steps / model   : 100
Microbatches / model      : 200
Images / model            : 6400
Mixed precision           : FP16 + GradScaler
GradScaler initial scale  : 1024.0
Overflow retry limit      : 10
Strict deterministic      : True
Pilot weights saved       : False
Test data read            : False

S1 — MobileNetV3-Large
------------------------------------------------------------------------
step=001 | loss=11.249763 | grad=5.1552 | lr=0.00000034 | peak=991.7 MiB
step=010 | loss=11.230841 | grad=5.0489 | lr=0.00000186 | peak=1097.6 MiB
step=020 | loss=11.101439 | grad=5.0607 

## Cell 12: Exact-Resume Equivalence Audit

### Mục tiêu

Chứng minh checkpoint có thể tiếp tục huấn luyện chính xác, không chỉ nạp được
weights.

Với từng Student, so sánh:

- Nhánh A: chạy liên tục 4 optimizer steps.
- Nhánh B: chạy 2 steps, lưu checkpoint, dựng lại model, nạp checkpoint và
  chạy tiếp 2 steps.

Hai nhánh phải tạo kết quả giống nhau tuyệt đối.

### Stateless augmentation

Random augmentation được seed riêng cho từng sample từ:

- Base seed.
- Epoch.
- Image ID.
- Caption row index.

Vì vậy augmentation không phụ thuộc vào:

- Worker nào xử lý ảnh.
- DataLoader prefetch.
- Thời điểm restart.
- Mid-epoch resume.

### Trạng thái phải lưu

- Encoder và adapter.
- Shared decoder.
- Optimizer.
- Scheduler.
- GradScaler.
- Epoch.
- Global optimizer step.
- Sample cursor.
- Gradient-accumulation position.
- Python RNG.
- NumPy RNG.
- Torch CPU RNG.
- Torch CUDA RNG.
- Sampler state.
- Contract checksums.

### So sánh bitwise

- Batch image/token sequence.
- Loss, gradient norm và learning rate.
- Encoder state.
- Decoder state.
- Optimizer state.
- Scheduler state.
- GradScaler state.
- RNG states.

Checkpoint chỉ được tạo tại optimizer-step boundary. Temporary checkpoint phải
được xóa sau audit.

In [12]:
import copy
import gc
import hashlib
import json
import math
import random
import time
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler


CELL12_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]
RESUME_AUDIT_EPOCH = 0
TOTAL_OPTIMIZER_STEPS = 4
SPLIT_OPTIMIZER_STEPS = 2

required_runtime_names = [
    "PHASE2_SEED",
    "device",
    "PHASE2_ROOT",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_STUDENT_SPECS",
    "PHASE2_TRAIN_CAPTION_INDEX",
    "PHASE2_TRAIN_IMAGE_PATH_BY_ID",
    "PHASE2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE2_TRAIN_TRANSFORMS",
    "PHASE2_DATASET_CLASS",
    "PHASE2_COLLATOR",
    "seed_dataloader_worker",
    "TRAINING_CONTRACT",
    "REPRODUCIBILITY_CONTRACT",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "student_audits",
    "shared_decoder_checkpoint_sha256",
    "DeterministicImageBalancedCaptionSampler",
    "build_baseline_optimizer",
    "build_scheduler",
    "build_and_load_encoder",
    "build_and_load_shared_decoder",
    "enable_training_graph_with_frozen_batchnorm",
    "configure_strict_deterministic_attention",
    "PHASE2_TRAINING_ENGINE_PILOT_READY_PATH",
    "pilot_results",
    "read_json",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–11: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 11."
)

assert read_json(
    PHASE2_TRAINING_ENGINE_PILOT_READY_PATH
)["status"] == "ready"
assert set(pilot_results) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    result["status"] == "passed"
    for result in pilot_results.values()
)
assert TRAINING_CONTRACT["contract_version"] == (
    "phase2_baseline_training_v2"
)
assert REPRODUCIBILITY_CONTRACT["resume"][
    "exact_resume_required"
]


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_tensor(tensor):
    cpu_tensor = (
        tensor.detach()
        .cpu()
        .contiguous()
    )
    digest = hashlib.sha256()
    digest.update(
        str(cpu_tensor.dtype).encode("utf-8")
    )
    digest.update(
        json.dumps(
            list(cpu_tensor.shape)
        ).encode("utf-8")
    )
    digest.update(
        cpu_tensor.numpy().tobytes()
    )
    return digest.hexdigest()


def sha256_training_batch(batch):
    digest = hashlib.sha256()

    for key in [
        "images",
        "decoder_input_ids",
        "decoder_attention_mask",
        "labels",
        "image_ids",
        "annotation_ids",
    ]:
        digest.update(key.encode("utf-8"))
        digest.update(
            sha256_tensor(batch[key]).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as output_file:
        json.dump(
            payload,
            output_file,
            ensure_ascii=False,
            indent=2,
        )


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])

    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all(
            state["torch_cuda"]
        )


def recursive_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: recursive_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            recursive_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            recursive_to_cpu(item)
            for item in value
        )

    if isinstance(value, np.ndarray):
        return value.copy()

    return copy.deepcopy(value)


def recursive_exact_equal(first, second):
    if torch.is_tensor(first):
        return (
            torch.is_tensor(second)
            and first.dtype == second.dtype
            and tuple(first.shape) == tuple(
                second.shape
            )
            and torch.equal(
                first.cpu(),
                second.cpu(),
            )
        )

    if isinstance(first, np.ndarray):
        return (
            isinstance(second, np.ndarray)
            and first.dtype == second.dtype
            and first.shape == second.shape
            and np.array_equal(first, second)
        )

    if isinstance(first, dict):
        return (
            isinstance(second, dict)
            and list(first.keys())
            == list(second.keys())
            and all(
                recursive_exact_equal(
                    first[key],
                    second[key],
                )
                for key in first
            )
        )

    if isinstance(first, (list, tuple)):
        return (
            isinstance(second, type(first))
            and len(first) == len(second)
            and all(
                recursive_exact_equal(
                    first_item,
                    second_item,
                )
                for first_item, second_item
                in zip(first, second)
            )
        )

    return first == second


class StatelessEpochAugmentedDataset(Dataset):
    """
    Random augmentation được seed theo sample thay vì worker RNG.

    Vì vậy batch tại (epoch, row_index) không phụ thuộc prefetch, worker count
    hoặc việc resume giữa epoch.
    """

    def __init__(
        self,
        base_dataset,
        caption_index,
        base_seed,
        epoch,
    ):
        self.base_dataset = base_dataset
        self.caption_index = caption_index
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.base_dataset)

    def sample_seed(self, row_index):
        image_id = int(
            self.caption_index.image_ids[
                row_index
            ]
        )
        mixed = (
            self.base_seed * 1_000_003
            + self.epoch * 97_409
            + image_id * 65_537
            + int(row_index) * 257
        )
        return mixed % (2 ** 63 - 1)

    def __getitem__(self, row_index):
        row_index = int(row_index)
        sample_seed = self.sample_seed(row_index)

        python_state = random.getstate()
        numpy_state = np.random.get_state()

        try:
            random.seed(sample_seed)
            np.random.seed(sample_seed % (2 ** 32))

            with torch.random.fork_rng(
                devices=[],
                enabled=True,
            ):
                torch.manual_seed(sample_seed)
                sample = self.base_dataset[
                    row_index
                ]
        finally:
            random.setstate(python_state)
            np.random.set_state(numpy_state)

        return sample


class ResumeSliceSampler(Sampler):
    def __init__(
        self,
        epoch_indices,
        start_position,
        sample_count,
    ):
        self.epoch_indices = tuple(
            int(index)
            for index in epoch_indices
        )
        self.start_position = int(
            start_position
        )
        self.sample_count = int(sample_count)
        self.stop_position = (
            self.start_position
            + self.sample_count
        )

        assert self.start_position >= 0
        assert self.sample_count > 0
        assert self.stop_position <= len(
            self.epoch_indices
        )

    def __len__(self):
        return self.sample_count

    def __iter__(self):
        return iter(
            self.epoch_indices[
                self.start_position:
                self.stop_position
            ]
        )


def build_resume_audit_loader(
    student_id,
    epoch_indices,
    start_position,
    optimizer_steps,
):
    batching = TRAINING_CONTRACT["batching"]
    physical_batch_size = int(
        batching["physical_batch_size"]
    )
    accumulation_steps = int(
        batching["gradient_accumulation_steps"]
    )
    microbatch_count = (
        int(optimizer_steps)
        * accumulation_steps
    )
    sample_count = (
        microbatch_count
        * physical_batch_size
    )

    base_dataset = PHASE2_DATASET_CLASS(
        caption_index=PHASE2_TRAIN_CAPTION_INDEX,
        image_path_by_id=(
            PHASE2_TRAIN_IMAGE_PATH_BY_ID
        ),
        image_transform=(
            PHASE2_TRAIN_TRANSFORMS[student_id]
        ),
        held_out_test_ids=(
            PHASE2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
    stateless_dataset = (
        StatelessEpochAugmentedDataset(
            base_dataset=base_dataset,
            caption_index=(
                PHASE2_TRAIN_CAPTION_INDEX
            ),
            base_seed=PHASE2_SEED,
            epoch=RESUME_AUDIT_EPOCH,
        )
    )
    slice_sampler = ResumeSliceSampler(
        epoch_indices=epoch_indices,
        start_position=start_position,
        sample_count=sample_count,
    )

    worker_generator = torch.Generator()
    worker_generator.manual_seed(
        PHASE2_SEED
        + RESUME_AUDIT_EPOCH
        + start_position
    )

    loader = DataLoader(
        dataset=stateless_dataset,
        batch_size=physical_batch_size,
        sampler=slice_sampler,
        shuffle=False,
        num_workers=int(batching["num_workers"]),
        pin_memory=bool(batching["pin_memory"]),
        drop_last=False,
        collate_fn=PHASE2_COLLATOR,
        worker_init_fn=seed_dataloader_worker,
        generator=worker_generator,
        persistent_workers=False,
        prefetch_factor=2,
    )

    assert len(loader) == microbatch_count

    return loader, {
        "start_position": start_position,
        "sample_count": sample_count,
        "stop_position": (
            start_position + sample_count
        ),
        "microbatch_count": microbatch_count,
        "optimizer_steps": optimizer_steps,
    }


def move_batch_to_device(batch):
    return {
        "images": batch["images"].to(
            device,
            non_blocking=True,
        ),
        "decoder_input_ids": batch[
            "decoder_input_ids"
        ].to(
            device,
            non_blocking=True,
        ),
        "decoder_attention_mask": batch[
            "decoder_attention_mask"
        ].to(
            device,
            non_blocking=True,
        ),
        "labels": batch["labels"].to(
            device,
            non_blocking=True,
        ),
    }


def forward_loss(encoder, decoder, batch):
    visual_memory = encoder(batch["images"])
    assert visual_memory.shape[1:] == (
        49,
        256,
    )

    memory_attention_mask = torch.ones(
        visual_memory.shape[:2],
        dtype=torch.bool,
        device=visual_memory.device,
    )
    logits = decoder(
        decoder_input_ids=batch[
            "decoder_input_ids"
        ],
        visual_memory=visual_memory,
        decoder_attention_mask=batch[
            "decoder_attention_mask"
        ],
        memory_attention_mask=(
            memory_attention_mask
        ),
    )
    assert logits.shape[-1] == 10_080
    assert torch.isfinite(logits).all()

    loss = F.cross_entropy(
        logits.reshape(-1, 10_080).float(),
        batch["labels"].reshape(-1),
        ignore_index=0,
    )
    assert torch.isfinite(loss)

    return loss


def create_model_training_state(
    student_id,
    probe_images,
):
    (
        encoder,
        _,
        encoder_checkpoint_path,
        encoder_load_mode,
    ) = build_and_load_encoder(
        student_id,
        probe_images,
    )
    decoder, decoder_load_mode = (
        build_and_load_shared_decoder()
    )
    frozen_batchnorm_count = (
        enable_training_graph_with_frozen_batchnorm(
            encoder
        )
    )
    decoder.train()

    optimizer = build_baseline_optimizer(
        encoder,
        decoder,
    )
    scheduler, scheduler_inventory = (
        build_scheduler(optimizer)
    )

    mixed_precision = TRAINING_CONTRACT[
        "optimization"
    ]["mixed_precision"]
    scaler = torch.amp.GradScaler(
        device="cuda",
        init_scale=float(
            mixed_precision["initial_scale"]
        ),
        growth_factor=float(
            mixed_precision["growth_factor"]
        ),
        backoff_factor=float(
            mixed_precision["backoff_factor"]
        ),
        growth_interval=int(
            mixed_precision["growth_interval"]
        ),
        enabled=(device.type == "cuda"),
    )

    return {
        "encoder": encoder,
        "decoder": decoder,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "scaler": scaler,
        "scheduler_inventory": (
            scheduler_inventory
        ),
        "frozen_batchnorm_count": (
            frozen_batchnorm_count
        ),
        "encoder_checkpoint_path": (
            encoder_checkpoint_path
        ),
        "encoder_load_mode": (
            encoder_load_mode
        ),
        "decoder_load_mode": (
            decoder_load_mode
        ),
    }


def train_optimizer_steps(
    training_state,
    loader,
    starting_global_step,
):
    encoder = training_state["encoder"]
    decoder = training_state["decoder"]
    optimizer = training_state["optimizer"]
    scheduler = training_state["scheduler"]
    scaler = training_state["scaler"]

    accumulation_steps = int(
        TRAINING_CONTRACT["batching"][
            "gradient_accumulation_steps"
        ]
    )
    gradient_clip_norm = float(
        TRAINING_CONTRACT["optimization"][
            "gradient_clip_norm"
        ]
    )
    amp_enabled = device.type == "cuda"

    optimizer.zero_grad(set_to_none=True)

    global_step = int(starting_global_step)
    microbatch_count = 0
    sample_count = 0
    batch_sha256 = []
    trajectory = []
    accumulated_losses = []

    for batch_cpu in loader:
        batch_sha256.append(
            sha256_training_batch(batch_cpu)
        )
        batch = move_batch_to_device(batch_cpu)
        microbatch_count += 1
        sample_count += int(
            batch["images"].shape[0]
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            loss = forward_loss(
                encoder,
                decoder,
                batch,
            )
            scaled_loss = loss / accumulation_steps

        accumulated_losses.append(
            float(loss.item())
        )
        scaler.scale(scaled_loss).backward()

        del scaled_loss
        del loss
        del batch
        del batch_cpu

        if (
            microbatch_count % accumulation_steps
            != 0
        ):
            continue

        scaler.unscale_(optimizer)
        trainable_parameters = [
            parameter
            for group in optimizer.param_groups
            for parameter in group["params"]
        ]
        gradient_norm = float(
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=gradient_clip_norm,
                error_if_nonfinite=True,
            ).item()
        )
        assert math.isfinite(gradient_norm)

        scale_before = float(
            scaler.get_scale()
        )
        scaler.step(optimizer)
        scaler.update()
        scale_after = float(
            scaler.get_scale()
        )
        assert scale_after == scale_before, (
            "Exact-resume audit không cho phép overflow; "
            "pilot trước đó phải ổn định."
        )

        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        trajectory.append(
            {
                "global_optimizer_step": (
                    global_step
                ),
                "mean_microbatch_loss": float(
                    np.mean(accumulated_losses)
                ),
                "gradient_norm": gradient_norm,
                "learning_rates": [
                    float(group["lr"])
                    for group in optimizer.param_groups
                ],
                "gradscaler_scale": scale_after,
            }
        )
        accumulated_losses.clear()

    assert microbatch_count % accumulation_steps == 0
    assert not accumulated_losses

    return {
        "ending_global_step": global_step,
        "microbatch_count": microbatch_count,
        "sample_count": sample_count,
        "batch_sha256": batch_sha256,
        "trajectory": trajectory,
    }


def capture_training_state_for_compare(
    training_state,
):
    return {
        "encoder": recursive_to_cpu(
            training_state[
                "encoder"
            ].state_dict()
        ),
        "decoder": recursive_to_cpu(
            training_state[
                "decoder"
            ].state_dict()
        ),
        "optimizer": recursive_to_cpu(
            training_state[
                "optimizer"
            ].state_dict()
        ),
        "scheduler": recursive_to_cpu(
            training_state[
                "scheduler"
            ].state_dict()
        ),
        "scaler": recursive_to_cpu(
            training_state[
                "scaler"
            ].state_dict()
        ),
        "rng": recursive_to_cpu(
            capture_rng_state()
        ),
    }


def build_resume_checkpoint_payload(
    student_id,
    training_state,
    global_optimizer_step,
    sample_cursor,
):
    return {
        "schema_version": "phase2_exact_resume_v1",
        "student_id": student_id,
        "epoch": RESUME_AUDIT_EPOCH,
        "global_optimizer_step": int(
            global_optimizer_step
        ),
        "sample_cursor": int(sample_cursor),
        "gradient_accumulation_position": 0,
        "encoder": recursive_to_cpu(
            training_state[
                "encoder"
            ].state_dict()
        ),
        "decoder": recursive_to_cpu(
            training_state[
                "decoder"
            ].state_dict()
        ),
        "optimizer": recursive_to_cpu(
            training_state[
                "optimizer"
            ].state_dict()
        ),
        "scheduler": recursive_to_cpu(
            training_state[
                "scheduler"
            ].state_dict()
        ),
        "scaler": recursive_to_cpu(
            training_state[
                "scaler"
            ].state_dict()
        ),
        "rng": recursive_to_cpu(
            capture_rng_state()
        ),
        "sampler": {
            "base_seed": PHASE2_SEED,
            "epoch": RESUME_AUDIT_EPOCH,
            "sample_cursor": int(
                sample_cursor
            ),
        },
        "contracts": {
            "training_sha256": sha256_file(
                PHASE2_TRAINING_CONTRACT_PATH
            ),
            "reproducibility_sha256": sha256_file(
                PHASE2_REPRODUCIBILITY_CONTRACT_PATH
            ),
        },
    }


def load_resume_checkpoint_into_state(
    payload,
    training_state,
):
    encoder_result = training_state[
        "encoder"
    ].load_state_dict(
        payload["encoder"],
        strict=True,
    )
    decoder_result = training_state[
        "decoder"
    ].load_state_dict(
        payload["decoder"],
        strict=True,
    )
    assert not encoder_result.missing_keys
    assert not encoder_result.unexpected_keys
    assert not decoder_result.missing_keys
    assert not decoder_result.unexpected_keys

    training_state["optimizer"].load_state_dict(
        payload["optimizer"]
    )
    training_state["scheduler"].load_state_dict(
        payload["scheduler"]
    )
    training_state["scaler"].load_state_dict(
        payload["scaler"]
    )
    restore_rng_state(payload["rng"])


def delete_training_state(training_state):
    for key in [
        "scaler",
        "scheduler",
        "optimizer",
        "encoder",
        "decoder",
    ]:
        if key in training_state:
            del training_state[key]

    del training_state
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_exact_resume_audit_for_student(
    student_id,
    epoch_indices,
):
    student_started_at = time.perf_counter()
    physical_batch_size = int(
        TRAINING_CONTRACT["batching"][
            "physical_batch_size"
        ]
    )
    accumulation_steps = int(
        TRAINING_CONTRACT["batching"][
            "gradient_accumulation_steps"
        ]
    )
    samples_per_optimizer_step = (
        physical_batch_size
        * accumulation_steps
    )
    total_sample_count = (
        TOTAL_OPTIMIZER_STEPS
        * samples_per_optimizer_step
    )
    split_sample_count = (
        SPLIT_OPTIMIZER_STEPS
        * samples_per_optimizer_step
    )

    # Một probe batch chỉ để dựng đúng backbone/adapter shape.
    probe_loader, _ = build_resume_audit_loader(
        student_id=student_id,
        epoch_indices=epoch_indices,
        start_position=0,
        optimizer_steps=1,
    )
    probe_batch_cpu = next(iter(probe_loader))
    probe_images = probe_batch_cpu[
        "images"
    ].to(
        device,
        non_blocking=True,
    )
    del probe_loader

    branch_seed = (
        PHASE2_SEED
        + 12_000
        + EXPECTED_STUDENT_IDS.index(student_id)
    )

    # Branch A — 4 steps liên tục.
    continuous_state = create_model_training_state(
        student_id,
        probe_images,
    )
    seed_all(branch_seed)
    continuous_loader, continuous_slice = (
        build_resume_audit_loader(
            student_id=student_id,
            epoch_indices=epoch_indices,
            start_position=0,
            optimizer_steps=(
                TOTAL_OPTIMIZER_STEPS
            ),
        )
    )
    continuous_run = train_optimizer_steps(
        training_state=continuous_state,
        loader=continuous_loader,
        starting_global_step=0,
    )
    continuous_final = (
        capture_training_state_for_compare(
            continuous_state
        )
    )
    del continuous_loader
    delete_training_state(continuous_state)

    # Branch B1 — 2 steps rồi save.
    split_state = create_model_training_state(
        student_id,
        probe_images,
    )
    seed_all(branch_seed)
    split_loader, split_slice = (
        build_resume_audit_loader(
            student_id=student_id,
            epoch_indices=epoch_indices,
            start_position=0,
            optimizer_steps=(
                SPLIT_OPTIMIZER_STEPS
            ),
        )
    )
    split_first_run = train_optimizer_steps(
        training_state=split_state,
        loader=split_loader,
        starting_global_step=0,
    )
    assert split_first_run[
        "ending_global_step"
    ] == SPLIT_OPTIMIZER_STEPS
    assert split_first_run[
        "sample_count"
    ] == split_sample_count

    resume_payload = (
        build_resume_checkpoint_payload(
            student_id=student_id,
            training_state=split_state,
            global_optimizer_step=(
                SPLIT_OPTIMIZER_STEPS
            ),
            sample_cursor=split_sample_count,
        )
    )
    temporary_checkpoint_path = (
        PHASE2_EXACT_RESUME_TEMP_DIR
        / f"{student_id.lower()}_resume_audit_temp.pt"
    )
    torch.save(
        resume_payload,
        temporary_checkpoint_path,
    )
    temporary_checkpoint_sha256 = sha256_file(
        temporary_checkpoint_path
    )

    del split_loader
    delete_training_state(split_state)
    del resume_payload

    # Branch B2 — rebuild, load và chạy 2 steps còn lại.
    loaded_payload = torch.load(
        temporary_checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    assert loaded_payload[
        "student_id"
    ] == student_id
    assert loaded_payload[
        "global_optimizer_step"
    ] == SPLIT_OPTIMIZER_STEPS
    assert loaded_payload[
        "sample_cursor"
    ] == split_sample_count
    assert loaded_payload[
        "gradient_accumulation_position"
    ] == 0

    resumed_state = create_model_training_state(
        student_id,
        probe_images,
    )
    load_resume_checkpoint_into_state(
        loaded_payload,
        resumed_state,
    )

    remaining_steps = (
        TOTAL_OPTIMIZER_STEPS
        - SPLIT_OPTIMIZER_STEPS
    )
    resumed_loader, resumed_slice = (
        build_resume_audit_loader(
            student_id=student_id,
            epoch_indices=epoch_indices,
            start_position=split_sample_count,
            optimizer_steps=remaining_steps,
        )
    )
    resumed_run = train_optimizer_steps(
        training_state=resumed_state,
        loader=resumed_loader,
        starting_global_step=(
            SPLIT_OPTIMIZER_STEPS
        ),
    )
    resumed_final = (
        capture_training_state_for_compare(
            resumed_state
        )
    )

    del resumed_loader
    delete_training_state(resumed_state)
    del loaded_payload

    temporary_checkpoint_path.unlink()
    assert not temporary_checkpoint_path.exists()

    combined_resume_batch_hashes = (
        split_first_run["batch_sha256"]
        + resumed_run["batch_sha256"]
    )
    combined_resume_trajectory = (
        split_first_run["trajectory"]
        + resumed_run["trajectory"]
    )

    equality = {
        "batch_sequence": (
            continuous_run["batch_sha256"]
            == combined_resume_batch_hashes
        ),
        "trajectory": recursive_exact_equal(
            continuous_run["trajectory"],
            combined_resume_trajectory,
        ),
        "encoder": recursive_exact_equal(
            continuous_final["encoder"],
            resumed_final["encoder"],
        ),
        "decoder": recursive_exact_equal(
            continuous_final["decoder"],
            resumed_final["decoder"],
        ),
        "optimizer": recursive_exact_equal(
            continuous_final["optimizer"],
            resumed_final["optimizer"],
        ),
        "scheduler": recursive_exact_equal(
            continuous_final["scheduler"],
            resumed_final["scheduler"],
        ),
        "scaler": recursive_exact_equal(
            continuous_final["scaler"],
            resumed_final["scaler"],
        ),
        "rng": recursive_exact_equal(
            continuous_final["rng"],
            resumed_final["rng"],
        ),
    }

    assert all(equality.values()), (
        f"{student_id}: exact-resume mismatch: {equality}"
    )
    assert continuous_run[
        "ending_global_step"
    ] == TOTAL_OPTIMIZER_STEPS
    assert resumed_run[
        "ending_global_step"
    ] == TOTAL_OPTIMIZER_STEPS
    assert continuous_run[
        "sample_count"
    ] == total_sample_count
    assert (
        split_first_run["sample_count"]
        + resumed_run["sample_count"]
    ) == total_sample_count

    student_elapsed_seconds = (
        time.perf_counter()
        - student_started_at
    )

    result = {
        "status": "passed",
        "student_id": student_id,
        "student_name": (
            PHASE2_STUDENT_SPECS[
                student_id
            ]["student_name"]
        ),
        "comparison": {
            "continuous_optimizer_steps": (
                TOTAL_OPTIMIZER_STEPS
            ),
            "split_before_save_steps": (
                SPLIT_OPTIMIZER_STEPS
            ),
            "resumed_steps": remaining_steps,
            "total_samples": total_sample_count,
            "checkpoint_at_accumulation_boundary": True,
            "continuous_slice": continuous_slice,
            "split_slice": split_slice,
            "resumed_slice": resumed_slice,
        },
        "exact_equality": equality,
        "all_exact": True,
        "temporary_checkpoint": {
            "sha256_before_delete": (
                temporary_checkpoint_sha256
            ),
            "deleted": True,
        },
        "stateless_augmentation": {
            "enabled": True,
            "seed_components": [
                "base seed",
                "epoch",
                "image_id",
                "caption row index",
            ],
            "independent_of_worker_prefetch": True,
            "batch_sequence_exact": equality[
                "batch_sequence"
            ],
        },
        "trajectory": {
            "continuous": (
                continuous_run["trajectory"]
            ),
            "split_and_resumed": (
                combined_resume_trajectory
            ),
        },
        "initial_checkpoints": {
            "encoder_sha256": (
                student_audits[student_id][
                    "initial_checkpoint_sha256"
                ]
            ),
            "decoder_sha256": (
                shared_decoder_checkpoint_sha256
            ),
        },
        "data_policy": {
            "source_split": "train+restval",
            "caption_text_read": False,
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
            "full_training_started": False,
        },
        "elapsed_seconds": student_elapsed_seconds,
    }

    del continuous_final
    del resumed_final
    del continuous_run
    del split_first_run
    del resumed_run
    del probe_images
    del probe_batch_cpu

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


deterministic_backend = (
    configure_strict_deterministic_attention()
)

image_balanced_sampler = (
    DeterministicImageBalancedCaptionSampler(
        caption_index=(
            PHASE2_TRAIN_CAPTION_INDEX
        ),
        base_seed=PHASE2_SEED,
        epoch=RESUME_AUDIT_EPOCH,
    )
)
epoch_indices = (
    image_balanced_sampler.indices_for_epoch(
        RESUME_AUDIT_EPOCH
    )
)
assert len(epoch_indices) == 113_287

PHASE2_EXACT_RESUME_TEMP_DIR = (
    PHASE2_ROOT / "_temporary_exact_resume_audit"
)
PHASE2_EXACT_RESUME_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Cho phép chạy lại an toàn sau một lần kernel bị ngắt giữa audit. Chỉ xóa
# đúng ba temporary checkpoint có tên do cell này quản lý.
for cleanup_student_id in EXPECTED_STUDENT_IDS:
    stale_temporary_checkpoint = (
        PHASE2_EXACT_RESUME_TEMP_DIR
        / (
            f"{cleanup_student_id.lower()}"
            "_resume_audit_temp.pt"
        )
    )
    if stale_temporary_checkpoint.exists():
        stale_temporary_checkpoint.unlink()

assert not list(
    PHASE2_EXACT_RESUME_TEMP_DIR.iterdir()
), (
    "Temporary exact-resume directory không rỗng. "
    "Hãy kiểm tra artifact từ lần chạy lỗi trước."
)

initial_checkpoint_hashes_before = {
    "decoder": sha256_file(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH
    ),
    **{
        student_id: sha256_file(
            PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                student_id
            ]
        )
        for student_id in EXPECTED_STUDENT_IDS
    },
}
assert initial_checkpoint_hashes_before[
    "decoder"
] == shared_decoder_checkpoint_sha256
for student_id in EXPECTED_STUDENT_IDS:
    assert initial_checkpoint_hashes_before[
        student_id
    ] == student_audits[student_id][
        "initial_checkpoint_sha256"
    ]


print("=" * 72)
print("EXACT-RESUME EQUIVALENCE AUDIT")
print("=" * 72)
print()
print("Continuous branch        : 4 optimizer steps")
print("Resume branch            : 2 + save/load + 2 steps")
print("Physical/effective batch : 32 / 64")
print("Checkpoint boundary      : optimizer-step boundary")
print("Stateless augmentation   : enabled")
print("Strict deterministic     : True")
print("Comparison tolerance     : exact / bitwise")
print("Temporary weights kept   : False")
print("Test data read           : False")


exact_resume_results = {}

for student_id in EXPECTED_STUDENT_IDS:
    print()
    print(
        f"{student_id} — "
        f"{PHASE2_STUDENT_SPECS[student_id]['student_name']}"
    )
    print("-" * 72)

    student_result = (
        run_exact_resume_audit_for_student(
            student_id=student_id,
            epoch_indices=epoch_indices,
        )
    )
    exact_resume_results[student_id] = (
        student_result
    )

    print("Batch sequence exact     : True")
    print("Loss/gradient/LR exact   : True")
    print("Encoder state exact      : True")
    print("Decoder state exact      : True")
    print("Optimizer state exact    : True")
    print("Scheduler state exact    : True")
    print("GradScaler state exact   : True")
    print("RNG states exact         : True")
    print("Temporary checkpoint     : DELETED")
    print(
        "Elapsed                  :",
        round(
            student_result["elapsed_seconds"],
            2,
        ),
        "giây",
    )


assert set(exact_resume_results) == set(
    EXPECTED_STUDENT_IDS
)
assert all(
    result["status"] == "passed"
    and result["all_exact"]
    for result in exact_resume_results.values()
)
assert not list(
    PHASE2_EXACT_RESUME_TEMP_DIR.iterdir()
)
PHASE2_EXACT_RESUME_TEMP_DIR.rmdir()
assert not PHASE2_EXACT_RESUME_TEMP_DIR.exists()

initial_checkpoint_hashes_after = {
    "decoder": sha256_file(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH
    ),
    **{
        student_id: sha256_file(
            PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                student_id
            ]
        )
        for student_id in EXPECTED_STUDENT_IDS
    },
}
assert initial_checkpoint_hashes_after == (
    initial_checkpoint_hashes_before
)


PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH = (
    PHASE2_CONFIG_DIR
    / "phase2_stateless_augmentation_contract.json"
)
stateless_augmentation_contract = {
    "contract_version": (
        "phase2_stateless_augmentation_v1"
    ),
    "status": "locked",
    "class_name": (
        "StatelessEpochAugmentedDataset"
    ),
    "seed_function": (
        "deterministic mix of base_seed, epoch, image_id and row_index"
    ),
    "torch_rng_forked_per_sample": True,
    "python_numpy_rng_restored_per_sample": True,
    "independent_of_worker_count": True,
    "independent_of_worker_prefetch": True,
    "supports_mid_epoch_resume": True,
    "resume_cursor_unit": "sample position in shuffled epoch index list",
    "checkpoint_only_at_optimizer_step_boundary": True,
}
write_json(
    PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH,
    stateless_augmentation_contract,
)
stateless_contract_sha256 = sha256_file(
    PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH
)


cell12_elapsed_seconds = (
    time.perf_counter()
    - CELL12_STARTED_AT
)

PHASE2_EXACT_RESUME_AUDIT_PATH = (
    PHASE2_AUDIT_DIR
    / "phase2_exact_resume_equivalence_audit.json"
)
combined_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_ids": EXPECTED_STUDENT_IDS,
    "students": exact_resume_results,
    "all_students_exact": True,
    "comparison_tolerance": "exact_bitwise",
    "stateless_augmentation_contract": {
        "path": str(
            PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH
        ),
        "sha256": stateless_contract_sha256,
    },
    "strict_deterministic_backend": (
        deterministic_backend
    ),
    "initial_checkpoint_files_unchanged": True,
    "temporary_checkpoints_deleted": True,
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "full_training_started": False,
    "elapsed_seconds": cell12_elapsed_seconds,
}
write_json(
    PHASE2_EXACT_RESUME_AUDIT_PATH,
    combined_audit,
)
exact_resume_audit_sha256 = sha256_file(
    PHASE2_EXACT_RESUME_AUDIT_PATH
)


PHASE2_FULL_BASELINE_TRAINING_READY_PATH = (
    PHASE2_ROOT
    / "PHASE2_FULL_BASELINE_TRAINING_READY.json"
)
full_training_ready_marker = {
    "status": "ready",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_ids": EXPECTED_STUDENT_IDS,
    "training_contract": {
        "path": str(
            PHASE2_TRAINING_CONTRACT_PATH
        ),
        "sha256": sha256_file(
            PHASE2_TRAINING_CONTRACT_PATH
        ),
    },
    "reproducibility_contract": {
        "path": str(
            PHASE2_REPRODUCIBILITY_CONTRACT_PATH
        ),
        "sha256": sha256_file(
            PHASE2_REPRODUCIBILITY_CONTRACT_PATH
        ),
    },
    "stateless_augmentation_contract": {
        "path": str(
            PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH
        ),
        "sha256": stateless_contract_sha256,
    },
    "exact_resume_audit": {
        "path": str(
            PHASE2_EXACT_RESUME_AUDIT_PATH
        ),
        "sha256": exact_resume_audit_sha256,
    },
    "authorized_next_action": (
        "full baseline training for S1/S2/S3 in separate runs, "
        "seed 42, Validation-only checkpoint selection"
    ),
    "full_training_authorized": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    PHASE2_FULL_BASELINE_TRAINING_READY_PATH,
    full_training_ready_marker,
)
full_ready_sha256 = sha256_file(
    PHASE2_FULL_BASELINE_TRAINING_READY_PATH
)


print()
print("=" * 72)
print("PHASE 2 CELL 12 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Exact-resume students     : 3 / 3 PASSED")
print("Batch augmentation exact  : 3 / 3 PASSED")
print("Loss/gradient/LR exact    : 3 / 3 PASSED")
print("Encoder state exact       : 3 / 3 PASSED")
print("Decoder state exact       : 3 / 3 PASSED")
print("Optimizer state exact     : 3 / 3 PASSED")
print("Scheduler state exact     : 3 / 3 PASSED")
print("GradScaler state exact    : 3 / 3 PASSED")
print("RNG states exact          : 3 / 3 PASSED")
print("Stateless augmentation    : LOCKED")
print("Mid-epoch resume          : SUPPORTED")
print("Initial checkpoints intact: True")
print("Temporary checkpoints     : DELETED")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("Exact-resume audit        :", PHASE2_EXACT_RESUME_AUDIT_PATH)
print("Audit SHA-256             :", exact_resume_audit_sha256)
print(
    "Stateless contract        :",
    PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH,
)
print("Stateless SHA-256         :", stateless_contract_sha256)
print("READY marker              :", PHASE2_FULL_BASELINE_TRAINING_READY_PATH)
print("READY SHA-256             :", full_ready_sha256)
print(
    "Elapsed                   :",
    round(cell12_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : run full seed-42 baselines in "
    "three separate Kaggle notebook sessions"
)

EXACT-RESUME EQUIVALENCE AUDIT

Continuous branch        : 4 optimizer steps
Resume branch            : 2 + save/load + 2 steps
Physical/effective batch : 32 / 64
Checkpoint boundary      : optimizer-step boundary
Stateless augmentation   : enabled
Strict deterministic     : True
Comparison tolerance     : exact / bitwise
Temporary weights kept   : False
Test data read           : False

S1 — MobileNetV3-Large
------------------------------------------------------------------------
Batch sequence exact     : True
Loss/gradient/LR exact   : True
Encoder state exact      : True
Decoder state exact      : True
Optimizer state exact    : True
Scheduler state exact    : True
GradScaler state exact   : True
RNG states exact         : True
Temporary checkpoint     : DELETED
Elapsed                  : 8.57 giây

S2 — MobileViT-S
------------------------------------------------------------------------
Batch sequence exact     : True
Loss/gradient/LR exact   : True
Encoder state exact      : Tru

## Cell 13 — Đóng băng Shared Training Bundle

### Mục tiêu

- Xác minh marker cho phép full baseline training.
- Kiểm tra lại checksum của shared decoder và ba encoder checkpoint.
- Đóng băng toàn bộ contract, audit, log và checkpoint dùng chung.
- Sinh manifest với đường dẫn tương đối để có thể tái sử dụng trong các notebook S1/S2/S3.
- Không đọc ảnh, caption hoặc Test Set.
- Không thực hiện training.

### Điều kiện đạt

- Marker `PHASE2_FULL_BASELINE_TRAINING_READY.json` hợp lệ.
- 4/4 initial checkpoint đúng SHA-256.
- 10/10 required shared artifacts tồn tại.
- Toàn bộ manifest rows xác minh thành công.
- Tạo được `PHASE2_SHARED_TRAINING_BUNDLE_READY.json`.

In [13]:
import csv
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path


CELL13_STARTED_AT = time.perf_counter()

EXPECTED_STUDENT_IDS = ["S1", "S2", "S3"]

required_runtime_names = [
    "PHASE2_ROOT",
    "PHASE2_CONFIG_DIR",
    "PHASE2_AUDIT_DIR",
    "PHASE2_CHECKPOINT_DIR",
    "PHASE2_FULL_BASELINE_TRAINING_READY_PATH",
    "PHASE2_TRAINING_CONTRACT_PATH",
    "PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH",
    "PHASE2_REPRODUCIBILITY_CONTRACT_PATH",
    "PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH",
    "PHASE2_EXACT_RESUME_AUDIT_PATH",
    "PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "student_audits",
    "shared_decoder_checkpoint_sha256",
    "read_json",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

assert not missing_runtime_names, (
    "Thiếu biến từ Phase 2 Cell 1–12: "
    f"{missing_runtime_names}. "
    "Nếu Kaggle vừa restart, hãy chạy lại Cell 1 đến Cell 12."
)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as output_file:
        json.dump(
            payload,
            output_file,
            ensure_ascii=False,
            indent=2,
        )


def relative_posix(path, root):
    return path.resolve().relative_to(
        root.resolve()
    ).as_posix()


def artifact_role(relative_path):
    parts = Path(relative_path).parts
    filename = Path(relative_path).name

    if "checkpoints" in parts:
        return "initial_checkpoint"
    if "configs" in parts:
        return "locked_contract"
    if "audits" in parts:
        return "audit_evidence"
    if "logs" in parts:
        return "diagnostic_log"
    if filename.endswith("_READY.json"):
        return "readiness_marker"
    return "supporting_artifact"


phase2_root = Path(PHASE2_ROOT).resolve()
full_ready_marker = read_json(
    PHASE2_FULL_BASELINE_TRAINING_READY_PATH
)

assert full_ready_marker["status"] == "ready"
assert full_ready_marker[
    "full_training_authorized"
] is True
assert full_ready_marker[
    "test_set_status"
] == "sealed"
assert full_ready_marker[
    "test_set_used"
] is False

required_shared_paths = {
    "full_training_ready": Path(
        PHASE2_FULL_BASELINE_TRAINING_READY_PATH
    ),
    "training_contract": Path(
        PHASE2_TRAINING_CONTRACT_PATH
    ),
    "validation_policy": Path(
        PHASE2_VALIDATION_SELECTION_POLICY_JSON_PATH
    ),
    "reproducibility_contract": Path(
        PHASE2_REPRODUCIBILITY_CONTRACT_PATH
    ),
    "stateless_augmentation_contract": Path(
        PHASE2_STATELESS_AUGMENTATION_CONTRACT_PATH
    ),
    "exact_resume_audit": Path(
        PHASE2_EXACT_RESUME_AUDIT_PATH
    ),
    "shared_decoder_checkpoint": Path(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH
    ),
    **{
        f"{student_id.lower()}_encoder_checkpoint": Path(
            PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
                student_id
            ]
        )
        for student_id in EXPECTED_STUDENT_IDS
    },
}

for artifact_name, artifact_path in (
    required_shared_paths.items()
):
    assert artifact_path.is_file(), (
        f"Thiếu required shared artifact "
        f"{artifact_name}: {artifact_path}"
    )
    assert artifact_path.stat().st_size > 0
    assert artifact_path.resolve().is_relative_to(
        phase2_root
    )

assert sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
) == shared_decoder_checkpoint_sha256
for student_id in EXPECTED_STUDENT_IDS:
    assert sha256_file(
        PHASE2_ENCODER_INITIAL_CHECKPOINT_PATHS[
            student_id
        ]
    ) == student_audits[student_id][
        "initial_checkpoint_sha256"
    ]


PHASE2_MANIFEST_DIR = (
    phase2_root / "manifests"
)
PHASE2_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PHASE2_SHARED_BUNDLE_READY_PATH = (
    phase2_root
    / "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)

# Manifest không tự chứa manifest/marker mới để tránh checksum self-reference.
excluded_relative_prefixes = {
    "manifests/",
}
excluded_relative_files = {
    relative_posix(
        PHASE2_SHARED_BUNDLE_READY_PATH,
        phase2_root,
    ),
}

source_artifact_paths = []
for candidate_path in sorted(
    phase2_root.rglob("*")
):
    if not candidate_path.is_file():
        continue

    relative_path = relative_posix(
        candidate_path,
        phase2_root,
    )

    if relative_path in excluded_relative_files:
        continue
    if any(
        relative_path.startswith(prefix)
        for prefix in excluded_relative_prefixes
    ):
        continue

    source_artifact_paths.append(
        candidate_path
    )

assert source_artifact_paths

manifest_rows = []
for artifact_path in source_artifact_paths:
    relative_path = relative_posix(
        artifact_path,
        phase2_root,
    )
    manifest_rows.append(
        {
            "relative_path": relative_path,
            "role": artifact_role(
                relative_path
            ),
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": sha256_file(
                artifact_path
            ),
        }
    )

relative_paths = [
    row["relative_path"]
    for row in manifest_rows
]
assert len(relative_paths) == len(
    set(relative_paths)
)

required_relative_paths = {
    name: relative_posix(path, phase2_root)
    for name, path in required_shared_paths.items()
}
assert set(
    required_relative_paths.values()
).issubset(set(relative_paths))


PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH = (
    PHASE2_MANIFEST_DIR
    / "phase2_shared_training_bundle_manifest.csv"
)
with open(
    PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as manifest_file:
    writer = csv.DictWriter(
        manifest_file,
        fieldnames=[
            "relative_path",
            "role",
            "size_bytes",
            "sha256",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)


PHASE2_SHARED_BUNDLE_SHA256SUMS_PATH = (
    PHASE2_MANIFEST_DIR
    / "phase2_shared_training_bundle_SHA256SUMS.txt"
)
with open(
    PHASE2_SHARED_BUNDLE_SHA256SUMS_PATH,
    "w",
    encoding="utf-8",
    newline="\n",
) as checksum_file:
    for row in manifest_rows:
        checksum_file.write(
            f"{row['sha256']}  "
            f"{row['relative_path']}\n"
        )


PHASE2_SHARED_BUNDLE_MANIFEST_JSON_PATH = (
    PHASE2_MANIFEST_DIR
    / "phase2_shared_training_bundle_manifest.json"
)
manifest_payload = {
    "schema_version": (
        "phase2_shared_training_bundle_manifest_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "bundle_root_name": phase2_root.name,
    "relocatable": True,
    "path_policy": (
        "All artifact paths are POSIX relative to bundle root"
    ),
    "artifact_count": len(manifest_rows),
    "total_size_bytes": sum(
        row["size_bytes"]
        for row in manifest_rows
    ),
    "required_artifacts": (
        required_relative_paths
    ),
    "role_counts": dict(
        sorted(
            {
                role: sum(
                    row["role"] == role
                    for row in manifest_rows
                )
                for role in {
                    row["role"]
                    for row in manifest_rows
                }
            }.items()
        )
    ),
    "artifacts": manifest_rows,
    "data_policy": {
        "primary_dataset": "MS COCO 2014 only",
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "full_training_started": False,
    },
}
write_json(
    PHASE2_SHARED_BUNDLE_MANIFEST_JSON_PATH,
    manifest_payload,
)


manifest_csv_sha256 = sha256_file(
    PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH
)
manifest_json_sha256 = sha256_file(
    PHASE2_SHARED_BUNDLE_MANIFEST_JSON_PATH
)
sha256sums_sha256 = sha256_file(
    PHASE2_SHARED_BUNDLE_SHA256SUMS_PATH
)

# Đọc lại CSV và verify toàn bộ checksum trước khi phát hành marker.
with open(
    PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH,
    "r",
    encoding="utf-8",
    newline="",
) as manifest_file:
    verified_rows = list(
        csv.DictReader(manifest_file)
    )

assert len(verified_rows) == len(
    manifest_rows
)

for row in verified_rows:
    resolved_path = (
        phase2_root / row["relative_path"]
    ).resolve()
    assert resolved_path.is_relative_to(
        phase2_root
    )
    assert resolved_path.is_file()
    assert resolved_path.stat().st_size == int(
        row["size_bytes"]
    )
    assert sha256_file(resolved_path) == (
        row["sha256"]
    )


cell13_elapsed_seconds = (
    time.perf_counter()
    - CELL13_STARTED_AT
)

bundle_ready_marker = {
    "schema_version": (
        "phase2_shared_training_bundle_ready_v1"
    ),
    "status": "ready",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "bundle_root_name": phase2_root.name,
    "relocatable": True,
    "artifact_count": len(manifest_rows),
    "total_size_bytes": manifest_payload[
        "total_size_bytes"
    ],
    "manifests": {
        "csv": {
            "relative_path": relative_posix(
                PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH,
                phase2_root,
            ),
            "sha256": manifest_csv_sha256,
        },
        "json": {
            "relative_path": relative_posix(
                PHASE2_SHARED_BUNDLE_MANIFEST_JSON_PATH,
                phase2_root,
            ),
            "sha256": manifest_json_sha256,
        },
        "sha256sums": {
            "relative_path": relative_posix(
                PHASE2_SHARED_BUNDLE_SHA256SUMS_PATH,
                phase2_root,
            ),
            "sha256": sha256sums_sha256,
        },
    },
    "required_artifacts": required_relative_paths,
    "student_ids": EXPECTED_STUDENT_IDS,
    "training_contract_version": (
        read_json(
            PHASE2_TRAINING_CONTRACT_PATH
        )["contract_version"]
    ),
    "full_training_authorized": True,
    "consumer_rule": (
        "A downstream notebook must discover this marker, resolve paths "
        "relative to its parent directory and verify every manifest row."
    ),
    "recommended_consumers": [
        "phase2_s1_mobilenetv3_baseline_seed42",
        "phase2_s2_mobilevit_s_baseline_seed42",
        "phase2_s3_tinyvit5m_baseline_seed42",
    ],
    "data_policy": {
        "caption_text_read": False,
        "image_data_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "full_training_started": False,
    },
    "elapsed_seconds": cell13_elapsed_seconds,
}
write_json(
    PHASE2_SHARED_BUNDLE_READY_PATH,
    bundle_ready_marker,
)
bundle_ready_sha256 = sha256_file(
    PHASE2_SHARED_BUNDLE_READY_PATH
)


print("=" * 72)
print("PHASE 2 SHARED TRAINING BUNDLE")
print("=" * 72)
print()
print("Bundle root              :", phase2_root)
print("Relocatable              : True")
print("Artifact count           :", len(manifest_rows))
print(
    "Total size              :",
    f"{manifest_payload['total_size_bytes'] / (1024 ** 2):.2f} MiB",
)
print(
    "Role counts             :",
    manifest_payload["role_counts"],
)
print("Manifest verification    : ALL PASSED")
print("Initial checkpoints      : 4 / 4 VERIFIED")
print("Full training authorized : True")
print("Test Set status          : SEALED")

print()
print("=" * 72)
print("PHASE 2 CELL 13 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Shared bundle             : READY")
print("Relocatable paths         : True")
print("Manifest rows verified    :", len(verified_rows))
print("Checksum rows passed      :", len(verified_rows))
print("Required artifacts        : 10 / 10 PASSED")
print("Student initial encoders  : 3 / 3 VERIFIED")
print("Shared decoder checkpoint : VERIFIED")
print("Caption text read         : False")
print("Image data read           : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("Manifest CSV              :", PHASE2_SHARED_BUNDLE_MANIFEST_CSV_PATH)
print("Manifest CSV SHA-256      :", manifest_csv_sha256)
print("Manifest JSON             :", PHASE2_SHARED_BUNDLE_MANIFEST_JSON_PATH)
print("Manifest JSON SHA-256     :", manifest_json_sha256)
print("SHA256SUMS                :", PHASE2_SHARED_BUNDLE_SHA256SUMS_PATH)
print("SHA256SUMS SHA-256        :", sha256sums_sha256)
print("READY marker              :", PHASE2_SHARED_BUNDLE_READY_PATH)
print("READY SHA-256             :", bundle_ready_sha256)
print(
    "Elapsed                   :",
    round(cell13_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : Save Version, then attach this "
    "notebook output to three separate seed-42 training notebooks"
)

PHASE 2 SHARED TRAINING BUNDLE

Bundle root              : /kaggle/working//phase2_student_baselines
Relocatable              : True
Artifact count           : 38
Total size              : 74.18 MiB
Role counts             : {'audit_evidence': 11, 'diagnostic_log': 6, 'initial_checkpoint': 4, 'locked_contract': 14, 'readiness_marker': 3}
Manifest verification    : ALL PASSED
Initial checkpoints      : 4 / 4 VERIFIED
Full training authorized : True
Test Set status          : SEALED

PHASE 2 CELL 13 STATUS
Status                    : PASSED
Shared bundle             : READY
Relocatable paths         : True
Manifest rows verified    : 38
Checksum rows passed      : 38
Required artifacts        : 10 / 10 PASSED
Student initial encoders  : 3 / 3 VERIFIED
Shared decoder checkpoint : VERIFIED
Caption text read         : False
Image data read           : False
Test Parquet opened       : False
Test images read          : False
Test Set status           : SEALED
Test Set used             : Fals